# 3\.1\.1 Construct Unsupervised Modeling Matrices

This notebook is the bridge between feature engineering and unsupervised modeling\. Up through 1\.5\.6, the focus was on creating, evaluating, and selecting mobility features that describe transportation conditions across NYC Taxi Zones\. The goal here is different\. We are no longer deciding which features to keep\. Instead, we are converting those selected features into modeling\-ready matrices that can be reused consistently across PCA, UMAP, clustering, density estimation, and anomaly\-detection workflows\.

A key design principle of this notebook is that transportation\-system coverage is not treated as a data\-quality problem\. Some Taxi Zones have subway stations while others do not\. Some areas have stronger bus coverage than others\. These are real characteristics of the transportation network and should not cause otherwise valid observations to be discarded\. Modality\-specific matrices should maximize legitimate coverage while preserving the canonical Taxi Zone × Date × Temporal Bucket analytical grain\.

By the end of this notebook, we should have a set of standardized raw and scaled modeling matrices for Bus, Subway, Taxi, FHVHV, and Multimodal analyses, along with the metadata and QA artifacts needed to support downstream dimensionality reduction and anomaly\-detection workflows\.

## Notebook Setup

We start with shared imports and display settings so the rest of the notebook uses one consistent matrix\-preparation environment\.

In [1]:
# ---------------------------------------------------------------------
# Imports and notebook display settings
# ---------------------------------------------------------------------

from pathlib import Path
import json
import time
import warnings

import duckdb
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

print("Notebook setup imports loaded.")

Notebook setup imports loaded.


These paths and controls define the modeling\-matrix handoff\. The input panel comes from 1\.5\.5, while feature membership and metadata come from 1\.5\.6\.

In [2]:
# ---------------------------------------------------------------------
# Project paths, output controls, and matrix-preparation settings
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

INPUT_PANEL_DIR = PIPELINE_DATA_DIR / "1.5.5.final_tables"
INPUT_FEATURE_SET_DIR = PIPELINE_DATA_DIR / "1.5.6.final_tables"
OUTPUT_DIR = PIPELINE_DATA_DIR / "3.1.1.final_tables"
INTERMEDIATE_OUTPUT_DIR = PIPELINE_DATA_DIR / "3.1.1.intermediate_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PANEL_PATH = INPUT_PANEL_DIR / "variability_anomaly_feature_panel.parquet"
INPUT_FEATURE_CATALOG_PATH = INPUT_FEATURE_SET_DIR / "modeling_feature_catalog.csv"
INPUT_FEATURE_SET_DEFINITIONS_PATH = INPUT_FEATURE_SET_DIR / "modeling_feature_set_definitions.csv"

REBUILD_MODELING_MATRICES = False
WRITE_MODELING_OUTPUTS = True

MODEL_FEATURE_SET_NAMES = [
    "bus",
    "subway",
    "taxi",
    "fhvhv",
    "multimodal",
]

GRAIN_COLUMNS = [
    "taxi_zone_id",
    "date",
    "temporal_bucket",
]

REQUIRED_METADATA_COLUMNS = [
    "taxi_zone_id",
    "zone",
    "borough",
    "date",
    "temporal_bucket",
    "pre_post_cp",
]

OPTIONAL_METADATA_COLUMNS = [
    "canonical_location_id",
    "year",
    "month",
    "day_of_week",
    "hour",
    "is_weekend",
    "service_zone",
    "taxi_zone_geom_id",
    "centroid_lat",
    "centroid_lon",
    "centroid_x",
    "centroid_y",
    "in_congestion_pricing_zone",
    "congestion_pricing_zone",
    "congestion_pricing_region",
    "cp_region",
    "cp_zone_type",
    "policy_region",
    "spatial_region",
    "cbd_flag",
    "is_cbd",
    "cbd_spillover_category",
]

VALID_CP_PERIOD_VALUES = {"pre_cp", "post_cp"}

SCALER_NAME = "standard_scaler"
SCALER_WITH_MEAN = True
SCALER_WITH_STD = True

print(f"Input panel: {INPUT_PANEL_PATH}")
print(f"Feature catalog: {INPUT_FEATURE_CATALOG_PATH}")
print(f"Feature-set definitions: {INPUT_FEATURE_SET_DEFINITIONS_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Matrix feature sets: {', '.join(MODEL_FEATURE_SET_NAMES)}")

Input panel: pipeline_data/1.5.5.final_tables/variability_anomaly_feature_panel.parquet
Feature catalog: pipeline_data/1.5.6.final_tables/modeling_feature_catalog.csv
Feature-set definitions: pipeline_data/1.5.6.final_tables/modeling_feature_set_definitions.csv
Output directory: pipeline_data/3.1.1.final_tables
Matrix feature sets: bus, subway, taxi, fhvhv, multimodal


A few small helpers keep the QA tables compact and make failures easier to read when the notebook is run end to end\.

In [3]:
# ---------------------------------------------------------------------
# Shared validation helpers
# ---------------------------------------------------------------------

def status_label(condition):
    return "pass" if condition else "fail"


def require_columns(df, required_columns, df_name):
    missing_columns = [column for column in required_columns if column not in df.columns]
    assert not missing_columns, (
        f"{df_name} is missing required columns: {missing_columns}"
    )
    return missing_columns


def compact_path_status(paths_by_name):
    records = []
    for asset_name, asset_path in paths_by_name.items():
        exists = asset_path.exists()
        records.append(
            {
                "asset": asset_name,
                "path": str(asset_path),
                "exists": exists,
                "status": status_label(exists),
            }
        )
    return pd.DataFrame(records)


def normalize_name_series(series):
    return series.astype(str).str.strip().str.lower()

print("Validation helpers ready.")

Validation helpers ready.


## 3\.1\.1\.1 Load Modeling Assets and Validate Inputs

Before constructing any modeling matrices, we first load the final feature panel from 1\.5\.5 and the curated feature\-selection outputs from 1\.5\.6\. These assets define both the available mobility features and the modality\-specific feature\-set definitions that will be used throughout the rest of Chapter 3\.

This section should verify that the expected files exist, confirm feature counts and feature\-set definitions, validate the canonical Taxi Zone × Date × Temporal Bucket grain, and ensure that feature metadata and selection decisions were loaded successfully\. The objective is simply to confirm that the modeling inputs are complete and internally consistent before any matrix construction begins\.

First, verify that the 1\.5\.5 panel and 1\.5\.6 modeling\-definition files are present\. This check should fail early if the handoff assets are missing or stale\.

In [4]:
# ---------------------------------------------------------------------
# Validate required handoff asset paths
# ---------------------------------------------------------------------

required_asset_paths = {
    "1.5.5 final feature panel": INPUT_PANEL_PATH,
    "1.5.6 selected feature catalog": INPUT_FEATURE_CATALOG_PATH,
    "1.5.6 feature-set definitions": INPUT_FEATURE_SET_DEFINITIONS_PATH,
}

asset_path_status_df = compact_path_status(required_asset_paths)
display(asset_path_status_df)

missing_assets = asset_path_status_df.loc[
    ~asset_path_status_df["exists"],
    ["asset", "path"],
]

assert missing_assets.empty, (
    "Missing required modeling handoff assets:\n"
    + missing_assets.to_string(index=False)
)

print("Required handoff assets found.")

,asset,path,exists,status
0,1.5.5 final feature panel,pipeline_data/1.5.5.final_tables/variability_a...,True,pass
1,1.5.6 selected feature catalog,pipeline_data/1.5.6.final_tables/modeling_feat...,True,pass
2,1.5.6 feature-set definitions,pipeline_data/1.5.6.final_tables/modeling_feat...,True,pass


Required handoff assets found.


Findings\. All required handoff assets are available in the expected pipeline locations\. That gives this notebook a clean starting point: the final 1\.5\.5 feature panel supplies the source data, while the 1\.5\.6 catalog and feature\-set definitions supply the modeling\-column decisions\.

Load the selected feature catalog and feature\-set definitions from 1\.5\.6\. These tables are the source of truth for which columns may enter each matrix\.

In [5]:
# ---------------------------------------------------------------------
# Load and validate 1.5.6 feature catalog and feature-set definitions
# ---------------------------------------------------------------------

feature_catalog_df = pd.read_csv(INPUT_FEATURE_CATALOG_PATH)
feature_set_definitions_df = pd.read_csv(INPUT_FEATURE_SET_DEFINITIONS_PATH)


def find_column(df, candidate_columns, df_name, role_name):
    for column in candidate_columns:
        if column in df.columns:
            return column
    raise AssertionError(
        f"Could not find a {role_name} column in {df_name}. "
        f"Tried: {candidate_columns}. Available columns: {list(df.columns)}"
    )

CATALOG_FEATURE_COLUMN = find_column(
    feature_catalog_df,
    ["feature_name", "feature", "feature_column", "column_name"],
    "feature_catalog_df",
    "feature-name",
)

FEATURE_SET_FEATURE_COLUMN = find_column(
    feature_set_definitions_df,
    ["feature_name", "feature", "feature_column", "column_name"],
    "feature_set_definitions_df",
    "feature-name",
)

FEATURE_SET_NAME_COLUMN = find_column(
    feature_set_definitions_df,
    ["feature_set", "modeling_feature_set", "feature_set_name", "modeling_view", "matrix_name"],
    "feature_set_definitions_df",
    "feature-set",
)

feature_catalog_df[CATALOG_FEATURE_COLUMN] = feature_catalog_df[CATALOG_FEATURE_COLUMN].astype(str).str.strip()
feature_set_definitions_df[FEATURE_SET_FEATURE_COLUMN] = feature_set_definitions_df[FEATURE_SET_FEATURE_COLUMN].astype(str).str.strip()
feature_set_definitions_df[FEATURE_SET_NAME_COLUMN] = normalize_name_series(
    feature_set_definitions_df[FEATURE_SET_NAME_COLUMN]
)

catalog_features = set(feature_catalog_df[CATALOG_FEATURE_COLUMN])
feature_set_features = set(feature_set_definitions_df[FEATURE_SET_FEATURE_COLUMN])
feature_set_names = set(feature_set_definitions_df[FEATURE_SET_NAME_COLUMN])

unexpected_feature_sets = sorted(feature_set_names - set(MODEL_FEATURE_SET_NAMES))
missing_feature_sets = sorted(set(MODEL_FEATURE_SET_NAMES) - feature_set_names)
features_missing_from_catalog = sorted(feature_set_features - catalog_features)
traffic_features_in_default_sets = sorted(
    feature for feature in feature_set_features
    if "traffic" in feature.lower()
)

feature_set_load_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "catalog_row_count",
            "value": len(feature_catalog_df),
            "status": "reference",
        },
        {
            "qa_check": "catalog_unique_features",
            "value": feature_catalog_df[CATALOG_FEATURE_COLUMN].nunique(),
            "status": "pass" if feature_catalog_df[CATALOG_FEATURE_COLUMN].nunique() == 142 else "review",
        },
        {
            "qa_check": "feature_set_definition_rows",
            "value": len(feature_set_definitions_df),
            "status": "reference",
        },
        {
            "qa_check": "feature_sets_found",
            "value": ", ".join(sorted(feature_set_names)),
            "status": status_label(not unexpected_feature_sets and not missing_feature_sets),
        },
        {
            "qa_check": "features_missing_from_catalog",
            "value": len(features_missing_from_catalog),
            "status": status_label(len(features_missing_from_catalog) == 0),
        },
        {
            "qa_check": "traffic_features_in_default_sets",
            "value": len(traffic_features_in_default_sets),
            "status": status_label(len(traffic_features_in_default_sets) == 0),
        },
    ]
)

feature_set_count_df = (
    feature_set_definitions_df
    .groupby(FEATURE_SET_NAME_COLUMN, dropna=False)[FEATURE_SET_FEATURE_COLUMN]
    .nunique()
    .reset_index(name="feature_count")
    .sort_values(FEATURE_SET_NAME_COLUMN)
    .reset_index(drop=True)
)

display(feature_set_load_summary_df)
display(feature_set_count_df)

assert not unexpected_feature_sets, f"Unexpected feature sets found: {unexpected_feature_sets}"
assert not missing_feature_sets, f"Expected feature sets not found: {missing_feature_sets}"
assert not features_missing_from_catalog, (
    "Feature-set definitions reference features not present in the catalog: "
    f"{features_missing_from_catalog[:20]}"
)
assert not traffic_features_in_default_sets, (
    "Traffic features are present in default unsupervised feature sets: "
    f"{traffic_features_in_default_sets[:20]}"
)

print("1.5.6 feature catalog and feature-set definitions loaded.")

,qa_check,value,status
0,catalog_row_count,142,reference
1,catalog_unique_features,142,pass
2,feature_set_definition_rows,318,reference
3,feature_sets_found,"bus, fhvhv, multimodal, subway, taxi",pass
4,features_missing_from_catalog,0,pass
5,traffic_features_in_default_sets,0,pass


,feature_set_name,feature_count
0,bus,49
1,fhvhv,51
2,multimodal,142
3,subway,27
4,taxi,49


1.5.6 feature catalog and feature-set definitions loaded.


Findings\. The 1\.5\.6 handoff assets loaded cleanly: the selected feature catalog contains the expected 142\-feature modeling base, and the feature\-set definition table contains the expected Bus, Subway, Taxi, FHVHV, and Multimodal modeling views\. The feature\-set definitions are also consistent with the catalog, and no Traffic features appear in the default unsupervised feature sets\.

Use DuckDB to inspect the 1\.5\.5 panel before materializing matrices\. This confirms the panel grain, metadata availability, selected\-feature availability, and congestion\-pricing labels without loading every engineered column into memory\.

In [6]:
# ---------------------------------------------------------------------
# Validate 1.5.5 panel schema, grain, metadata, and selected features
# ---------------------------------------------------------------------

panel_path_sql = str(INPUT_PANEL_PATH).replace("\\", "/").replace("'", "''")

with duckdb.connect() as con:
    panel_schema_df = con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{panel_path_sql}')"
    ).fetchdf()

    panel_summary = con.execute(
        f"""
        SELECT
            COUNT(*) AS row_count,
            COUNT(DISTINCT taxi_zone_id) AS unique_taxi_zones,
            COUNT(DISTINCT temporal_bucket) AS unique_temporal_buckets,
            MIN(CAST(date AS DATE)) AS min_date,
            MAX(CAST(date AS DATE)) AS max_date
        FROM read_parquet('{panel_path_sql}')
        """
    ).fetchone()

    duplicate_grain_rows = con.execute(
        f"""
        SELECT COALESCE(SUM(row_count - 1), 0) AS duplicate_grain_rows
        FROM (
            SELECT
                taxi_zone_id,
                CAST(date AS DATE) AS date,
                temporal_bucket,
                COUNT(*) AS row_count
            FROM read_parquet('{panel_path_sql}')
            GROUP BY 1, 2, 3
            HAVING COUNT(*) > 1
        )
        """
    ).fetchone()[0]

    pre_post_cp_counts_df = con.execute(
        f"""
        SELECT
            pre_post_cp,
            COUNT(*) AS row_count
        FROM read_parquet('{panel_path_sql}')
        GROUP BY 1
        ORDER BY 1
        """
    ).fetchdf()

panel_columns = panel_schema_df["column_name"].tolist()
panel_column_set = set(panel_columns)

missing_required_metadata_columns = [
    column for column in REQUIRED_METADATA_COLUMNS
    if column not in panel_column_set
]

available_metadata_columns = [
    column for column in REQUIRED_METADATA_COLUMNS + OPTIONAL_METADATA_COLUMNS
    if column in panel_column_set
]

selected_feature_columns = sorted(feature_set_features)
selected_features_missing_from_panel = [
    feature for feature in selected_feature_columns
    if feature not in panel_column_set
]

unexpected_pre_post_cp_values = sorted(
    set(pre_post_cp_counts_df["pre_post_cp"].dropna()) - VALID_CP_PERIOD_VALUES
)

panel_validation_df = pd.DataFrame(
    [
        {
            "qa_check": "row_count",
            "value": int(panel_summary[0]),
            "status": "reference",
        },
        {
            "qa_check": "column_count",
            "value": len(panel_columns),
            "status": "reference",
        },
        {
            "qa_check": "date_range",
            "value": f"{panel_summary[3]} to {panel_summary[4]}",
            "status": "reference",
        },
        {
            "qa_check": "unique_taxi_zones",
            "value": int(panel_summary[1]),
            "status": "reference",
        },
        {
            "qa_check": "unique_temporal_buckets",
            "value": int(panel_summary[2]),
            "status": "reference",
        },
        {
            "qa_check": "duplicate_grain_rows",
            "value": int(duplicate_grain_rows),
            "status": status_label(duplicate_grain_rows == 0),
        },
        {
            "qa_check": "missing_required_metadata_columns",
            "value": len(missing_required_metadata_columns),
            "status": status_label(len(missing_required_metadata_columns) == 0),
        },
        {
            "qa_check": "available_metadata_columns",
            "value": len(available_metadata_columns),
            "status": "reference",
        },
        {
            "qa_check": "selected_features_missing_from_panel",
            "value": len(selected_features_missing_from_panel),
            "status": status_label(len(selected_features_missing_from_panel) == 0),
        },
        {
            "qa_check": "unexpected_pre_post_cp_values",
            "value": len(unexpected_pre_post_cp_values),
            "status": status_label(len(unexpected_pre_post_cp_values) == 0),
        },
    ]
)

pre_post_cp_counts_df["row_pct"] = (
    pre_post_cp_counts_df["row_count"]
    / pre_post_cp_counts_df["row_count"].sum()
).round(5)

metadata_columns_df = pd.DataFrame(
    {"metadata_column": available_metadata_columns}
)

display(panel_validation_df)
display(pre_post_cp_counts_df)
display(metadata_columns_df)

assert duplicate_grain_rows == 0, (
    "Duplicate rows detected at the Taxi Zone x Date x Temporal Bucket grain."
)
assert not missing_required_metadata_columns, (
    "Missing required metadata columns: "
    f"{missing_required_metadata_columns}"
)
assert not selected_features_missing_from_panel, (
    "Selected 1.5.6 features are missing from the 1.5.5 panel: "
    f"{selected_features_missing_from_panel[:20]}"
)
assert not unexpected_pre_post_cp_values, (
    "Unexpected pre_post_cp values found: "
    f"{unexpected_pre_post_cp_values}. Expected only pre_cp/post_cp."
)

print("1.5.5 panel schema, grain, metadata, and selected-feature availability validated.")

,qa_check,value,status
0,row_count,1559590,reference
1,column_count,342,reference
2,date_range,2023-01-01 to 2026-03-31,reference
3,unique_taxi_zones,263,reference
4,unique_temporal_buckets,10,reference
5,duplicate_grain_rows,0,pass
6,missing_required_metadata_columns,0,pass
7,available_metadata_columns,10,reference
8,selected_features_missing_from_panel,0,pass
9,unexpected_pre_post_cp_values,0,pass


,pre_post_cp,row_count,row_pct
0,post_cp,593065,0.38027
1,pre_cp,966525,0.61973


,metadata_column
0,taxi_zone_id
1,zone
2,borough
3,date
4,temporal_bucket
5,pre_post_cp
6,canonical_location_id
7,year
8,month
9,day_of_week


1.5.5 panel schema, grain, metadata, and selected-feature availability validated.


Findings\. The 1\.5\.5 panel is present and internally consistent for matrix preparation\. It preserves the Taxi Zone × Date × Temporal Bucket grain with no duplicate grain rows, spans the expected January 2023 through March 2026 analysis window, and includes both canonical congestion\-pricing period labels\. The selected 1\.5\.6 feature columns are available in the panel, so 3\.1\.1 can move into coverage profiling without reopening feature engineering or feature selection\.

## 3\.1\.1\.2 Evaluate Modeling Coverage per Modality

Before evaluating the Multimodal matrix, we first diagnose missingness inside each modality\-specific modeling view\. Bus, Subway, Taxi, and FHVHV each have different coverage rules, different undefined\-value situations, and different feature descendants\. Treating them separately helps distinguish structural absence, true zero activity, insufficient history, undefined speed or ratio values, and possible data\-quality loss\.

This section focuses only on modality\-specific modeling sets\. The goal is to identify the main reasons for missingness within Bus, Subway, Taxi, and FHVHV before deciding row inclusion, zero\-fill eligibility, or multimodal thresholds\. Intermodal and Multimodal coverage are evaluated separately in the next section after the modality\-specific missingness patterns are clear\.

### Classify Missingness per Modality

Start by classifying selected features within the four single\-modality modeling views\. The classification is intentionally practical: it separates count\-like features, speed or average features, temporal\-history descendants, decomposition features, deviation features, spatial/context features, and ratio or interaction\-style features because each class has a different missing\-value interpretation\.

In [7]:
# ---------------------------------------------------------------------
# Classify selected features for single-modality coverage diagnosis
# ---------------------------------------------------------------------

UNIMODAL_FEATURE_SET_NAMES = [
    "bus",
    "subway",
    "taxi",
    "fhvhv",
]


def first_existing_column(df, candidate_columns):
    for column in candidate_columns:
        if column in df.columns:
            return column
    return None


FEATURE_FAMILY_COLUMN = first_existing_column(
    feature_catalog_df,
    ["feature_family", "family", "feature_group", "feature_category"],
)
FEATURE_SOURCE_COLUMN = first_existing_column(
    feature_catalog_df,
    ["source_notebook", "source", "notebook", "origin_notebook"],
)
FEATURE_MODALITY_COLUMN = first_existing_column(
    feature_catalog_df,
    ["modality", "primary_modality", "feature_modality", "transportation_mode"],
)
SOURCE_METRIC_COLUMN = first_existing_column(
    feature_catalog_df,
    ["source_metric", "base_metric", "metric", "core_metric"],
)

metadata_columns_for_merge = [CATALOG_FEATURE_COLUMN]
metadata_columns_for_merge.extend(
    column for column in [
        FEATURE_FAMILY_COLUMN,
        FEATURE_SOURCE_COLUMN,
        FEATURE_MODALITY_COLUMN,
        SOURCE_METRIC_COLUMN,
    ]
    if column is not None and column not in metadata_columns_for_merge
)

feature_metadata_df = (
    feature_catalog_df[metadata_columns_for_merge]
    .drop_duplicates(CATALOG_FEATURE_COLUMN)
    .rename(columns={CATALOG_FEATURE_COLUMN: "feature_name"})
)

unimodal_feature_catalog_df = (
    feature_set_definitions_df.loc[
        feature_set_definitions_df[FEATURE_SET_NAME_COLUMN].isin(UNIMODAL_FEATURE_SET_NAMES),
        [FEATURE_SET_NAME_COLUMN, FEATURE_SET_FEATURE_COLUMN],
    ]
    .drop_duplicates()
    .rename(
        columns={
            FEATURE_SET_NAME_COLUMN: "feature_set",
            FEATURE_SET_FEATURE_COLUMN: "feature_name",
        }
    )
    .merge(feature_metadata_df, on="feature_name", how="left")
)


def classify_feature_reason(feature_name, feature_family=None):
    name = str(feature_name).lower()
    family = "" if pd.isna(feature_family) else str(feature_family).lower()

    if "spatial" in family or "context" in family:
        return "spatial_or_context"
    if (
        "interaction" in family
        or "ratio" in name
        or "share" in name
        or "interaction" in name
        or "cross_modal" in name
        or "divergence" in name
        or "spillover" in name
    ):
        return "interaction_or_ratio"
    if "decomposition" in family or any(token in name for token in ["residual", "seasonal", "fourier"]):
        return "decomposition"
    if "deviation" in family or "anomaly" in family or any(token in name for token in ["deviation", "zscore", "percentile"]):
        return "deviation_anomaly"
    if "temporal" in family or any(token in name for token in ["lag", "rolling", "ema", "pct_change", "change"]):
        return "temporal_history"
    if any(token in name for token in ["speed", "avg", "average", "duration", "distance", "mean"]):
        return "speed_average_or_rate"
    if any(token in name for token in ["trip_count", "ridership", "count", "volume", "demand"]):
        return "demand_count_or_volume"
    return "other_selected_feature"


if FEATURE_FAMILY_COLUMN is None:
    unimodal_feature_catalog_df["feature_family_for_coverage"] = "unknown"
else:
    unimodal_feature_catalog_df["feature_family_for_coverage"] = (
        unimodal_feature_catalog_df[FEATURE_FAMILY_COLUMN]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
    )

unimodal_feature_catalog_df["coverage_reason_class"] = unimodal_feature_catalog_df.apply(
    lambda row: classify_feature_reason(
        row["feature_name"],
        row.get(FEATURE_FAMILY_COLUMN) if FEATURE_FAMILY_COLUMN is not None else None,
    ),
    axis=1,
)

unimodal_feature_reason_inventory_df = (
    unimodal_feature_catalog_df
    .groupby(["feature_set", "coverage_reason_class"], dropna=False)
    .agg(feature_count=("feature_name", "nunique"))
    .reset_index()
    .sort_values(["feature_set", "coverage_reason_class"])
    .reset_index(drop=True)
)

display(unimodal_feature_reason_inventory_df)
display(unimodal_feature_catalog_df.head(30))

print("Single-modality selected features classified for coverage diagnosis.")

,feature_set,coverage_reason_class,feature_count
0,bus,decomposition,10
1,bus,demand_count_or_volume,1
2,bus,deviation_anomaly,2
3,bus,interaction_or_ratio,9
4,bus,spatial_or_context,9
5,bus,speed_average_or_rate,2
6,bus,temporal_history,16
7,fhvhv,decomposition,10
8,fhvhv,demand_count_or_volume,2
9,fhvhv,deviation_anomaly,2


,feature_set,feature_name,feature_family,source_notebook,modality,source_metric,feature_family_for_coverage,coverage_reason_class
0,bus,avg_bus_speed_deviation_abs_zscore,anomaly_severity,1.5.5,bus,avg_bus_speed,anomaly_severity,deviation_anomaly
1,bus,bus_trip_count_deviation_abs_zscore,anomaly_severity,1.5.5,bus,bus_trip_count,anomaly_severity,deviation_anomaly
2,bus,avg_bus_speed_fourier20_residual_reconstructed,fourier20_residual_reconstruction,1.5.4,bus,avg_bus_speed,fourier20_residual_reconstruction,decomposition
3,bus,bus_trip_count_fourier20_residual_reconstructed,fourier20_residual_reconstruction,1.5.4,bus,bus_trip_count,fourier20_residual_reconstruction,decomposition
4,bus,avg_bus_speed_local_vs_connected_zscore,multimodal_interaction,1.5.3,bus,avg_bus_speed,multimodal_interaction,interaction_or_ratio
5,bus,bus_trip_count_local_vs_connected_zscore,multimodal_interaction,1.5.3,bus,bus_trip_count,multimodal_interaction,interaction_or_ratio
6,bus,avg_bus_speed,raw_metric,pre_1.5_engineering,bus,avg_bus_speed,raw_metric,speed_average_or_rate
7,bus,avg_bus_travel_time,raw_metric,pre_1.5_engineering,bus,avg_bus_travel_time,raw_metric,speed_average_or_rate
8,bus,bus_trip_count,raw_metric,pre_1.5_engineering,bus,bus_trip_count,raw_metric,demand_count_or_volume
9,bus,borough_mean_avg_bus_speed,spatial_context,1.5.2,bus,avg_bus_speed,spatial_context,spatial_or_context


Single-modality selected features classified for coverage diagnosis.


Findings\. The single\-modality feature sets are not raw\-only matrices\. Each view combines a small core signal layer with a much larger set of engineered descendants\. Bus, FHVHV, and Taxi all contain substantial interaction/ratio feature groups, while Subway is smaller but still includes interaction/ratio features\. This matters because interaction/ratio features depend on cross\-system context and should not automatically define whether a Bus, Taxi, FHVHV, or Subway observation is valid\.

### Filter Out Missingness from Missing Base Metrics and Inspect Remainder

Before treating engineered missingness as a row\-retention problem, first check whether the underlying modality signal exists\. The key question is: among rows with usable base/core signal, how often do selected engineered descendants still become null? If that population is small, complete\-case filtering may be acceptable\. If it is large, engineered features are destroying otherwise valid mobility observations\.

In [8]:
# ---------------------------------------------------------------------
# Measure engineered-descendant missingness among rows with usable base signal
# ---------------------------------------------------------------------

CORE_SIGNAL_REASON_CLASSES = {
    "demand_count_or_volume",
    "speed_average_or_rate",
}

ENGINEERED_DESCENDANT_REASON_CLASSES = {
    "interaction_or_ratio",
    "temporal_history",
    "decomposition",
    "deviation_anomaly",
}


def duckdb_identifier(column_name):
    return '"' + column_name.replace('"', '""') + '"'


def non_null_count_expression(feature_columns):
    if not feature_columns:
        return "0"
    return " + ".join(
        f"CASE WHEN {duckdb_identifier(feature)} IS NOT NULL THEN 1 ELSE 0 END"
        for feature in feature_columns
    )


base_signal_engineered_gap_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_reason_df = unimodal_feature_catalog_df.loc[
            unimodal_feature_catalog_df["feature_set"] == feature_set_name,
            ["feature_name", "coverage_reason_class"],
        ].drop_duplicates()

        core_features = sorted(
            feature_reason_df.loc[
                feature_reason_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        engineered_features = sorted(
            feature_reason_df.loc[
                feature_reason_df["coverage_reason_class"].isin(ENGINEERED_DESCENDANT_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )

        core_feature_count = len(core_features)
        engineered_feature_count = len(engineered_features)
        core_non_null_expression = non_null_count_expression(core_features)
        engineered_non_null_expression = non_null_count_expression(engineered_features)

        gap_query = f"""
        WITH row_coverage AS (
            SELECT
                ({core_non_null_expression}) AS core_non_null_count,
                ({engineered_non_null_expression}) AS engineered_non_null_count
            FROM read_parquet('{panel_path_sql}')
        )
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN core_non_null_count > 0 THEN 1 ELSE 0 END) AS rows_with_any_core_signal,
            SUM(CASE WHEN core_non_null_count = {core_feature_count} THEN 1 ELSE 0 END) AS rows_with_complete_core_signal,
            SUM(CASE WHEN core_non_null_count > 0 AND engineered_non_null_count < {engineered_feature_count} THEN 1 ELSE 0 END) AS any_core_but_incomplete_engineered_rows,
            SUM(CASE WHEN core_non_null_count = {core_feature_count} AND engineered_non_null_count < {engineered_feature_count} THEN 1 ELSE 0 END) AS complete_core_but_incomplete_engineered_rows,
            SUM(CASE WHEN core_non_null_count = {core_feature_count} AND engineered_non_null_count = {engineered_feature_count} THEN 1 ELSE 0 END) AS complete_core_and_complete_engineered_rows,
            AVG(core_non_null_count) AS avg_core_non_null_count,
            AVG(engineered_non_null_count) AS avg_engineered_non_null_count
        FROM row_coverage
        """
        result = con.execute(gap_query).fetchone()

        total_rows = int(result[0])
        rows_with_any_core_signal = int(result[1])
        rows_with_complete_core_signal = int(result[2])
        any_core_but_incomplete_engineered_rows = int(result[3])
        complete_core_but_incomplete_engineered_rows = int(result[4])
        complete_core_and_complete_engineered_rows = int(result[5])

        base_signal_engineered_gap_records.append(
            {
                "feature_set": feature_set_name,
                "core_feature_count": core_feature_count,
                "engineered_descendant_feature_count": engineered_feature_count,
                "total_rows": total_rows,
                "rows_with_any_core_signal": rows_with_any_core_signal,
                "rows_with_any_core_signal_pct": round(rows_with_any_core_signal / total_rows, 5),
                "rows_with_complete_core_signal": rows_with_complete_core_signal,
                "rows_with_complete_core_signal_pct": round(rows_with_complete_core_signal / total_rows, 5),
                "any_core_but_incomplete_engineered_rows": any_core_but_incomplete_engineered_rows,
                "any_core_but_incomplete_engineered_pct_of_any_core": round(
                    any_core_but_incomplete_engineered_rows / rows_with_any_core_signal,
                    5,
                ) if rows_with_any_core_signal else np.nan,
                "complete_core_but_incomplete_engineered_rows": complete_core_but_incomplete_engineered_rows,
                "complete_core_but_incomplete_engineered_pct_of_complete_core": round(
                    complete_core_but_incomplete_engineered_rows / rows_with_complete_core_signal,
                    5,
                ) if rows_with_complete_core_signal else np.nan,
                "complete_core_and_complete_engineered_rows": complete_core_and_complete_engineered_rows,
                "complete_core_and_complete_engineered_pct_of_complete_core": round(
                    complete_core_and_complete_engineered_rows / rows_with_complete_core_signal,
                    5,
                ) if rows_with_complete_core_signal else np.nan,
                "avg_core_non_null_pct": round(result[6] / core_feature_count, 5) if core_feature_count else np.nan,
                "avg_engineered_non_null_pct": round(result[7] / engineered_feature_count, 5) if engineered_feature_count else np.nan,
            }
        )

base_signal_engineered_gap_df = pd.DataFrame(base_signal_engineered_gap_records)
base_signal_engineered_gap_df = base_signal_engineered_gap_df.sort_values("feature_set").reset_index(drop=True)

base_signal_engineered_gap_display_df = base_signal_engineered_gap_df[
    [
        "feature_set",
        "rows_with_complete_core_signal",
        "complete_core_but_incomplete_engineered_rows",
        "complete_core_but_incomplete_engineered_pct_of_complete_core",
        "complete_core_and_complete_engineered_pct_of_complete_core",
        "avg_core_non_null_pct",
        "avg_engineered_non_null_pct",
    ]
].copy()

base_signal_engineered_gap_display_df = (
    base_signal_engineered_gap_display_df
    .sort_values("complete_core_but_incomplete_engineered_pct_of_complete_core", ascending=False)
    .rename(
        columns={
            "rows_with_complete_core_signal": "complete_core_rows",
            "complete_core_but_incomplete_engineered_rows": "engineered_gap_rows",
            "complete_core_but_incomplete_engineered_pct_of_complete_core": "engineered_gap_pct",
            "complete_core_and_complete_engineered_pct_of_complete_core": "complete_engineered_pct",
            "avg_core_non_null_pct": "avg_core_non_null_pct",
            "avg_engineered_non_null_pct": "avg_engineered_non_null_pct",
        }
    )
    .reset_index(drop=True)
)

display(base_signal_engineered_gap_display_df)

print("Base/core signal versus engineered-descendant missingness quantified.")

,feature_set,complete_core_rows,engineered_gap_rows,engineered_gap_pct,complete_engineered_pct,avg_core_non_null_pct,avg_engineered_non_null_pct
0,fhvhv,1559590,778915,0.49944,0.50056,1.00000,0.96394
1,bus,1476478,695804,0.47126,0.52874,0.94671,0.92965
2,taxi,1125143,418452,0.37191,0.62809,0.79108,0.87098
3,subway,925080,78854,0.08524,0.91476,0.59316,0.58979


Base/core signal versus engineered-descendant missingness quantified.


Findings\. Complete core signal is available for large observation universes across all four modalities, but engineered descendants are much less consistently complete\. Among complete\-core rows, FHVHV has engineered gaps in 49\.9% of rows, Bus in 47\.1%, Taxi in 37\.2%, and Subway in 8\.5%\. This shows that the main risk is not missing base mobility signal; it is losing valid observations because some engineered descendants are undefined\. Complete\-case filtering across the full selected feature set would therefore be too destructive for FHVHV, Bus, and Taxi, and only relatively mild for Subway\.

If complete\-core rows still lose engineered descendants, identify which engineered family is responsible\. This separates ordinary history\-window loss from decomposition availability, baseline/deviation availability, and ratio or interaction features\.

In [9]:
# ---------------------------------------------------------------------
# Attribute engineered-descendant loss among complete-core rows
# ---------------------------------------------------------------------

engineered_gap_reason_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_reason_df = unimodal_feature_catalog_df.loc[
            unimodal_feature_catalog_df["feature_set"] == feature_set_name,
            ["feature_name", "coverage_reason_class"],
        ].drop_duplicates()

        core_features = sorted(
            feature_reason_df.loc[
                feature_reason_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        core_feature_count = len(core_features)
        core_non_null_expression = non_null_count_expression(core_features)

        for reason_class in sorted(ENGINEERED_DESCENDANT_REASON_CLASSES):
            reason_features = sorted(
                feature_reason_df.loc[
                    feature_reason_df["coverage_reason_class"] == reason_class,
                    "feature_name",
                ].tolist()
            )
            if not reason_features:
                continue

            reason_feature_count = len(reason_features)
            reason_non_null_expression = non_null_count_expression(reason_features)

            reason_gap_query = f"""
            WITH row_coverage AS (
                SELECT
                    ({core_non_null_expression}) AS core_non_null_count,
                    ({reason_non_null_expression}) AS reason_non_null_count
                FROM read_parquet('{panel_path_sql}')
            )
            SELECT
                SUM(CASE WHEN core_non_null_count = {core_feature_count} THEN 1 ELSE 0 END) AS complete_core_rows,
                SUM(CASE WHEN core_non_null_count = {core_feature_count} AND reason_non_null_count < {reason_feature_count} THEN 1 ELSE 0 END) AS complete_core_but_incomplete_reason_rows,
                SUM(CASE WHEN core_non_null_count = {core_feature_count} AND reason_non_null_count = {reason_feature_count} THEN 1 ELSE 0 END) AS complete_core_and_complete_reason_rows,
                AVG(CASE WHEN core_non_null_count = {core_feature_count} THEN reason_non_null_count ELSE NULL END) AS avg_reason_non_null_count_among_complete_core
            FROM row_coverage
            """
            result = con.execute(reason_gap_query).fetchone()
            complete_core_rows = int(result[0])
            incomplete_reason_rows = int(result[1])
            complete_reason_rows = int(result[2])

            engineered_gap_reason_records.append(
                {
                    "feature_set": feature_set_name,
                    "engineered_reason_class": reason_class,
                    "engineered_feature_count": reason_feature_count,
                    "complete_core_rows": complete_core_rows,
                    "complete_core_but_incomplete_reason_rows": incomplete_reason_rows,
                    "incomplete_reason_pct_of_complete_core": round(
                        incomplete_reason_rows / complete_core_rows,
                        5,
                    ) if complete_core_rows else np.nan,
                    "complete_core_and_complete_reason_rows": complete_reason_rows,
                    "complete_reason_pct_of_complete_core": round(
                        complete_reason_rows / complete_core_rows,
                        5,
                    ) if complete_core_rows else np.nan,
                    "avg_reason_non_null_pct_among_complete_core": round(
                        result[3] / reason_feature_count,
                        5,
                    ) if reason_feature_count and result[3] is not None else np.nan,
                }
            )

engineered_gap_reason_df = pd.DataFrame(engineered_gap_reason_records)
engineered_gap_reason_df = engineered_gap_reason_df.sort_values(
    ["feature_set", "incomplete_reason_pct_of_complete_core"],
    ascending=[True, False],
).reset_index(drop=True)

engineered_gap_reason_pivot_df = (
    engineered_gap_reason_df
    .pivot_table(
        index="engineered_reason_class",
        columns="feature_set",
        values="incomplete_reason_pct_of_complete_core",
        aggfunc="first",
    )
    .reset_index()
    .fillna(0)
)

reason_order = (
    engineered_gap_reason_df
    .groupby("engineered_reason_class")["incomplete_reason_pct_of_complete_core"]
    .max()
    .sort_values(ascending=False)
    .index
)

engineered_gap_reason_pivot_df["engineered_reason_class"] = pd.Categorical(
    engineered_gap_reason_pivot_df["engineered_reason_class"],
    categories=reason_order,
    ordered=True,
)

engineered_gap_reason_pivot_df = (
    engineered_gap_reason_pivot_df
    .sort_values("engineered_reason_class")
    .reset_index(drop=True)
)

display(engineered_gap_reason_pivot_df)

print("Engineered-descendant loss attributed by reason class among complete-core rows.")

feature_set,engineered_reason_class,bus,fhvhv,subway,taxi
0,interaction_or_ratio,0.47052,0.49874,0.07071,0.33082
1,temporal_history,0.00228,0.02482,0.01473,0.10241
2,decomposition,0.00044,0.00185,0.00000,0.01047
3,deviation_anomaly,0.00002,0.00399,0.00731,0.00097


Engineered-descendant loss attributed by reason class among complete-core rows.


Findings\. The engineered\-descendant gap is dominated by interaction/ratio features\. Among complete\-core rows, interaction/ratio features are incomplete for 49\.9% of FHVHV rows, 47\.1% of Bus rows, 33\.1% of Taxi rows, and 7\.1% of Subway rows\. Other feature classes are much smaller by comparison: decomposition and deviation/anomaly gaps are near zero for most modalities, and temporal\-history gaps are small for Bus and Subway\. Taxi is the exception outside interactions, with temporal\-history gaps affecting 10\.2% of complete\-core rows, which warrants separate inspection\.

> ✅ Decision\. Interaction/ratio features should be excluded from the primary single\-modality matrices and evaluated in the intermodal/multimodal layer instead\. FHVHV percent\-change and percent\-delta temporal features should also move to an extended layer because their missingness is driven by zero\-activity denominator behavior\. Primary Bus, Subway, Taxi, and FHVHV row eligibility should be anchored on usable core modality signal, then checked against the remaining primary features\. This preserves valid single\-modality mobility states while avoiding broad mean imputation or artificial cross\-system values\.

### Remove Intermodal Features and Inspect Remaining Missingness

After moving interaction/ratio features out of the primary single\-modality matrices, apply two additional primary\-matrix rules\. FHVHV percent\-change and percent\-delta temporal features move to an extended layer because their missingness is driven by zero\-activity denominator behavior\. Special zones 105, 264, and 265 are excluded from primary matrices because their spatial context is incomplete or not comparable to normal Taxi Zones\.

In [10]:
# ---------------------------------------------------------------------
# Define primary single-modality feature and row eligibility exclusions
# ---------------------------------------------------------------------

PRIMARY_EXCLUDED_REASON_CLASSES = {
    "interaction_or_ratio",
}

PRIMARY_EXCLUDED_FEATURES_BY_SET = {
    "fhvhv": {
        "fhvhv_avg_trip_speed_pct_change_1_same_bucket",
        "fhvhv_trip_count_pct_change_1_same_bucket",
        "fhvhv_avg_trip_speed_cp_pct_delta_same_bucket",
        "fhvhv_trip_count_cp_pct_delta_same_bucket",
    },
}

PRIMARY_EXCLUDED_TAXI_ZONE_IDS = {
    105,  # Governor's Island / Ellis Island / Liberty Island
    264,  # Unknown
    265,  # Unknown
}

primary_excluded_zone_id_sql = ", ".join(
    str(zone_id) for zone_id in sorted(PRIMARY_EXCLUDED_TAXI_ZONE_IDS)
)
primary_zone_exclusion_sql = f"taxi_zone_id NOT IN ({primary_excluded_zone_id_sql})"


def is_primary_excluded_feature(row):
    feature_set_name = row["feature_set"]
    feature_name = row["feature_name"]
    return feature_name in PRIMARY_EXCLUDED_FEATURES_BY_SET.get(feature_set_name, set())


unimodal_feature_catalog_with_primary_status_df = unimodal_feature_catalog_df.copy()
unimodal_feature_catalog_with_primary_status_df["exclude_from_primary_single_modality"] = (
    unimodal_feature_catalog_with_primary_status_df["coverage_reason_class"].isin(PRIMARY_EXCLUDED_REASON_CLASSES)
    | unimodal_feature_catalog_with_primary_status_df.apply(is_primary_excluded_feature, axis=1)
)

unimodal_feature_catalog_with_primary_status_df["primary_single_modality_status"] = np.where(
    unimodal_feature_catalog_with_primary_status_df["exclude_from_primary_single_modality"],
    "move_to_intermodal_or_extended_layer",
    "retain_for_primary_review",
)

primary_unimodal_feature_catalog_df = (
    unimodal_feature_catalog_with_primary_status_df.loc[
        ~unimodal_feature_catalog_with_primary_status_df["exclude_from_primary_single_modality"]
    ]
    .copy()
    .reset_index(drop=True)
)

primary_feature_decision_summary_df = (
    unimodal_feature_catalog_with_primary_status_df
    .groupby(["feature_set", "coverage_reason_class", "primary_single_modality_status"], dropna=False)
    .agg(feature_count=("feature_name", "nunique"))
    .reset_index()
    .sort_values(["feature_set", "primary_single_modality_status", "coverage_reason_class"])
    .reset_index(drop=True)
)

primary_feature_count_summary_df = (
    primary_feature_decision_summary_df
    .groupby(["feature_set", "primary_single_modality_status"], dropna=False)
    .agg(feature_count=("feature_count", "sum"))
    .reset_index()
    .sort_values(["feature_set", "primary_single_modality_status"])
    .reset_index(drop=True)
)

primary_specific_feature_exclusion_df = (
    unimodal_feature_catalog_with_primary_status_df.loc[
        unimodal_feature_catalog_with_primary_status_df.apply(is_primary_excluded_feature, axis=1),
        ["feature_set", "feature_name", "coverage_reason_class", "source_metric"],
    ]
    .sort_values(["feature_set", "feature_name"])
    .reset_index(drop=True)
)

primary_zone_exclusion_df = pd.DataFrame(
    {
        "taxi_zone_id": sorted(PRIMARY_EXCLUDED_TAXI_ZONE_IDS),
        "primary_row_status": "exclude_from_primary_matrices",
        "reason": [
            "islands_have_non-comparable_spatial_context",
            "unknown_zone_has_no_normal_spatial_context",
            "unknown_zone_has_no_normal_spatial_context",
        ],
    }
)

primary_feature_decision_pivot_df = (
    primary_feature_decision_summary_df
    .pivot_table(
        index="feature_set",
        columns="primary_single_modality_status",
        values="feature_count",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

primary_feature_decision_pivot_df["total_selected"] = (
    primary_feature_decision_pivot_df
    .drop(columns=["feature_set"])
    .sum(axis=1)
)

primary_feature_decision_pivot_display_df = primary_feature_decision_pivot_df.rename(
    columns={
        "move_to_intermodal_or_extended_layer": "moved_features",
        "retain_for_primary_review": "primary_features",
    }
)

display(primary_feature_decision_pivot_display_df)

print("Primary single-modality feature candidates and row exclusions defined.")

primary_single_modality_status,feature_set,moved_features,primary_features,total_selected
0,bus,9,40,49
1,fhvhv,14,37,51
2,subway,6,21,27
3,taxi,11,38,49


Primary single-modality feature candidates and row exclusions defined.


Findings\. Separating interaction and extended\-layer features leaves substantial primary feature sets for each single\-modality matrix: 40 Bus features, 37 FHVHV features, 21 Subway features, and 38 Taxi features remain for primary review\. This is a targeted separation, not a broad feature reduction\. The primary matrices stay focused on within\-modality signal, while cross\-system features are preserved for intermodal and Multimodal handling\.

Recalculate row coverage for the revised primary feature candidates after feature exclusions and special\-zone row exclusions\. The denominator is rows outside Taxi Zones 105, 264, and 265 with complete core signal for that modality\.

In [11]:
# ---------------------------------------------------------------------
# Recalculate primary row coverage after feature and zone exclusions
# ---------------------------------------------------------------------

def primary_non_null_count_expression(feature_columns):
    if not feature_columns:
        return "0"
    return " + ".join(
        f"CASE WHEN {duckdb_identifier(feature)} IS NOT NULL THEN 1 ELSE 0 END"
        for feature in feature_columns
    )


primary_coverage_after_exclusion_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_reason_df = primary_unimodal_feature_catalog_df.loc[
            primary_unimodal_feature_catalog_df["feature_set"] == feature_set_name,
            ["feature_name", "coverage_reason_class"],
        ].drop_duplicates()

        core_features = sorted(
            feature_reason_df.loc[
                feature_reason_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        primary_features = sorted(feature_reason_df["feature_name"].tolist())

        core_feature_count = len(core_features)
        primary_feature_count = len(primary_features)
        core_non_null_expression = primary_non_null_count_expression(core_features)
        primary_non_null_expression = primary_non_null_count_expression(primary_features)

        coverage_query = f"""
        WITH eligible_rows AS (
            SELECT *
            FROM read_parquet('{panel_path_sql}')
            WHERE {primary_zone_exclusion_sql}
        ), row_coverage AS (
            SELECT
                ({core_non_null_expression}) AS core_non_null_count,
                ({primary_non_null_expression}) AS primary_non_null_count
            FROM eligible_rows
        )
        SELECT
            COUNT(*) AS eligible_rows,
            SUM(CASE WHEN core_non_null_count = {core_feature_count} THEN 1 ELSE 0 END) AS complete_core_rows,
            SUM(CASE WHEN core_non_null_count = {core_feature_count} AND primary_non_null_count = {primary_feature_count} THEN 1 ELSE 0 END) AS complete_core_and_complete_primary_rows,
            SUM(CASE WHEN core_non_null_count = {core_feature_count} AND primary_non_null_count < {primary_feature_count} THEN 1 ELSE 0 END) AS complete_core_but_incomplete_primary_rows,
            AVG(CASE WHEN core_non_null_count = {core_feature_count} THEN primary_non_null_count ELSE NULL END) AS avg_primary_non_null_count_among_complete_core
        FROM row_coverage
        """
        result = con.execute(coverage_query).fetchone()

        eligible_rows = int(result[0])
        complete_core_rows = int(result[1])
        complete_primary_rows = int(result[2])
        incomplete_primary_rows = int(result[3])

        primary_coverage_after_exclusion_records.append(
            {
                "feature_set": feature_set_name,
                "core_feature_count": core_feature_count,
                "primary_feature_count_after_exclusion": primary_feature_count,
                "eligible_rows_after_zone_exclusion": eligible_rows,
                "complete_core_rows": complete_core_rows,
                "complete_core_row_pct_of_eligible_rows": round(complete_core_rows / eligible_rows, 5),
                "complete_core_and_complete_primary_rows": complete_primary_rows,
                "complete_primary_pct_of_complete_core": round(
                    complete_primary_rows / complete_core_rows,
                    5,
                ) if complete_core_rows else np.nan,
                "complete_core_but_incomplete_primary_rows": incomplete_primary_rows,
                "incomplete_primary_pct_of_complete_core": round(
                    incomplete_primary_rows / complete_core_rows,
                    5,
                ) if complete_core_rows else np.nan,
                "avg_primary_non_null_pct_among_complete_core": round(
                    result[4] / primary_feature_count,
                    5,
                ) if primary_feature_count and result[4] is not None else np.nan,
            }
        )

primary_coverage_after_exclusion_df = pd.DataFrame(primary_coverage_after_exclusion_records)

primary_coverage_after_exclusion_display_df = primary_coverage_after_exclusion_df[
    [
        "feature_set",
        "eligible_rows_after_zone_exclusion",
        "complete_core_rows",
        "primary_feature_count_after_exclusion",
        "complete_core_and_complete_primary_rows",
        "complete_primary_pct_of_complete_core",
        "complete_core_but_incomplete_primary_rows",
        "incomplete_primary_pct_of_complete_core",
    ]
].copy()

primary_coverage_after_exclusion_display_df = primary_coverage_after_exclusion_display_df.sort_values(
    "incomplete_primary_pct_of_complete_core",
    ascending=False,
)

primary_coverage_after_exclusion_display_df = primary_coverage_after_exclusion_display_df.rename(
    columns={
        "eligible_rows_after_zone_exclusion": "eligible_rows",
        "primary_feature_count_after_exclusion": "primary_features",
        "complete_core_and_complete_primary_rows": "complete_primary_rows",
        "complete_primary_pct_of_complete_core": "complete_primary_pct",
        "complete_core_but_incomplete_primary_rows": "incomplete_primary_rows",
        "incomplete_primary_pct_of_complete_core": "incomplete_primary_pct",
    }
)

display(primary_coverage_after_exclusion_display_df)

print("Primary row coverage recalculated after feature and zone exclusions.")

,feature_set,eligible_rows,complete_core_rows,primary_features,complete_primary_rows,complete_primary_pct,incomplete_primary_rows,incomplete_primary_pct
2,taxi,1541800,1113555,38,995875,0.89432,117680,0.10568
1,subway,1541800,925080,21,911455,0.98527,13625,0.01473
3,fhvhv,1541800,1541800,37,1537629,0.99729,4171,0.00271
0,bus,1541800,1476478,40,1472947,0.99761,3531,0.00239


Primary row coverage recalculated after feature and zone exclusions.


Findings\. After interaction/ratio exclusion and special\-zone handling, Bus, FHVHV, and Subway have very small remaining primary\-completeness gaps among complete\-core rows\. Bus loses 0\.24%, FHVHV loses 0\.27%, and Subway loses 1\.47%\. Taxi remains the only unresolved primary\-modality concern: 10\.57% of complete\-core Taxi rows still have at least one missing primary feature\. This points to a targeted Taxi investigation rather than a general missingness problem across all modalities\.

List the remaining null\-bearing primary features after the first cleanup pass\. This is a diagnostic checkpoint, not the final row rule: it shows which feature families still create nulls after interaction/ratio features, unstable FHVHV percent\-change features, and special zones are moved out of the primary eligibility problem\.

In [12]:
# ---------------------------------------------------------------------
# Identify remaining missing primary features among eligible complete-core rows
# ---------------------------------------------------------------------

remaining_primary_missingness_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_reason_df = primary_unimodal_feature_catalog_df.loc[
            primary_unimodal_feature_catalog_df["feature_set"] == feature_set_name,
            ["feature_name", "coverage_reason_class", "feature_family_for_coverage"],
        ].drop_duplicates()

        core_features = sorted(
            feature_reason_df.loc[
                feature_reason_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        primary_features = sorted(feature_reason_df["feature_name"].tolist())
        core_feature_count = len(core_features)
        core_non_null_expression = primary_non_null_count_expression(core_features)

        null_expressions = [
            f"SUM(CASE WHEN {duckdb_identifier(feature)} IS NULL THEN 1 ELSE 0 END) AS {duckdb_identifier(feature)}"
            for feature in primary_features
        ]

        feature_missingness_query = f"""
        WITH complete_core_rows AS (
            SELECT *
            FROM read_parquet('{panel_path_sql}')
            WHERE {primary_zone_exclusion_sql}
              AND ({core_non_null_expression}) = {core_feature_count}
        )
        SELECT
            COUNT(*) AS complete_core_rows,
            {', '.join(null_expressions)}
        FROM complete_core_rows
        """
        feature_missingness_result = con.execute(feature_missingness_query).fetchdf().iloc[0]
        complete_core_rows = int(feature_missingness_result["complete_core_rows"])

        for feature in primary_features:
            missing_rows = int(feature_missingness_result[feature])
            if missing_rows == 0:
                continue

            feature_metadata = feature_reason_df.loc[
                feature_reason_df["feature_name"] == feature
            ].iloc[0]

            remaining_primary_missingness_records.append(
                {
                    "feature_set": feature_set_name,
                    "feature_name": feature,
                    "coverage_reason_class": feature_metadata["coverage_reason_class"],
                    "feature_family_for_coverage": feature_metadata["feature_family_for_coverage"],
                    "complete_core_rows": complete_core_rows,
                    "missing_rows_among_complete_core": missing_rows,
                    "missing_pct_among_complete_core": round(
                        missing_rows / complete_core_rows,
                        5,
                    ) if complete_core_rows else np.nan,
                }
            )

remaining_primary_missing_features_df = pd.DataFrame(remaining_primary_missingness_records)

if remaining_primary_missing_features_df.empty:
    remaining_primary_missing_features_df = pd.DataFrame(
        columns=[
            "feature_set",
            "feature_name",
            "coverage_reason_class",
            "feature_family_for_coverage",
            "complete_core_rows",
            "missing_rows_among_complete_core",
            "missing_pct_among_complete_core",
        ]
    )

remaining_primary_missing_features_df = remaining_primary_missing_features_df.sort_values(
    ["feature_set", "missing_pct_among_complete_core", "feature_name"],
    ascending=[True, False, True],
).reset_index(drop=True)

remaining_primary_missingness_reason_summary_df = (
    remaining_primary_missing_features_df
    .groupby(["feature_set", "coverage_reason_class"], as_index=False)
    .agg(
        feature_count_with_missingness=("feature_name", "nunique"),
        max_missing_pct_among_complete_core=("missing_pct_among_complete_core", "max"),
        median_missing_pct_among_complete_core=("missing_pct_among_complete_core", "median"),
        total_missing_feature_rows=("missing_rows_among_complete_core", "sum"),
    )
)

remaining_primary_missingness_reason_pivot_df = (
    remaining_primary_missingness_reason_summary_df
    .pivot_table(
        index="coverage_reason_class",
        columns="feature_set",
        values="max_missing_pct_among_complete_core",
        aggfunc="first",
    )
    .reset_index()
    .fillna(0)
)

remaining_primary_top5_by_modality_df = (
    remaining_primary_missing_features_df
    .sort_values(["feature_set", "missing_pct_among_complete_core"], ascending=[True, False])
    .groupby("feature_set", as_index=False)
    .head(5)
    [
        [
            "feature_set",
            "feature_name",
            "coverage_reason_class",
            "missing_rows_among_complete_core",
            "missing_pct_among_complete_core",
        ]
    ]
)

remaining_primary_top5_by_modality_display_df = remaining_primary_top5_by_modality_df.rename(
    columns={
        "feature_name": "feature",
        "coverage_reason_class": "reason",
        "missing_rows_among_complete_core": "missing_rows",
        "missing_pct_among_complete_core": "missing_pct",
    }
)

display(remaining_primary_missingness_reason_pivot_df)
display(remaining_primary_top5_by_modality_display_df)

print("Remaining missing primary features identified among eligible complete-core rows.")

feature_set,coverage_reason_class,bus,fhvhv,subway,taxi
0,decomposition,0.00044,0.00022,0.00000,0.01056
1,deviation_anomaly,0.00002,0.00102,0.00731,0.00097
2,temporal_history,0.00198,0.00169,0.01473,0.10317


,feature_set,feature,reason,missing_rows,missing_pct
0,bus,avg_bus_speed_abs_change_1_same_bucket,temporal_history,2926,0.00198
1,bus,avg_bus_speed_lag_1_same_bucket,temporal_history,2926,0.00198
2,bus,avg_bus_speed_pct_change_1_same_bucket,temporal_history,2926,0.00198
3,bus,bus_trip_count_abs_change_1_same_bucket,temporal_history,2926,0.00198
4,bus,bus_trip_count_lag_1_same_bucket,temporal_history,2926,0.00198
26,fhvhv,fhvhv_avg_trip_speed_abs_change_1_same_bucket,temporal_history,2600,0.00169
27,fhvhv,fhvhv_avg_trip_speed_lag_1_same_bucket,temporal_history,2600,0.00169
28,fhvhv,fhvhv_avg_trip_speed_rolling_std_7_same_bucket,temporal_history,2600,0.00169
29,fhvhv,fhvhv_trip_count_abs_change_1_same_bucket,temporal_history,2600,0.00169
30,fhvhv,fhvhv_trip_count_lag_1_same_bucket,temporal_history,2600,0.00169


Remaining missing primary features identified among eligible complete-core rows.


Findings\. After removing interaction features from the primary modality review, the remaining gaps are concentrated in temporal\-history features\. Bus and FHVHV are nearly complete, with their largest remaining gaps below 0\.2%\. Subway has a modest temporal\-history gap of about 1\.5% and one anomaly\-severity gap below 1%\. Taxi is the clear exception: two temporal\-history features are missing for about 10% of complete\-core Taxi rows, and several Taxi speed\-decomposition features are missing for about 1%\. This confirms that the next decision should focus on Taxi sparse\-service behavior rather than broad row dropping across all modalities\.

### Explore Unresolved Missingness

Inspect the remaining temporal\-history gaps before deciding whether to drop rows\. These gaps may be expected time\-series edges, such as the first available observation for a Taxi Zone and temporal bucket, but the Taxi gap is large enough that it should be verified directly\.

In [13]:
# ---------------------------------------------------------------------
# Summarize temporal-history gaps among complete-core rows
# ---------------------------------------------------------------------

temporal_gap_summary_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        temporal_features = sorted(
            remaining_primary_missing_features_df.loc[
                (remaining_primary_missing_features_df["feature_set"] == feature_set_name)
                & (remaining_primary_missing_features_df["coverage_reason_class"] == "temporal_history"),
                "feature_name",
            ].tolist()
        )
        if not temporal_features:
            continue

        feature_reason_df = primary_unimodal_feature_catalog_df.loc[
            primary_unimodal_feature_catalog_df["feature_set"] == feature_set_name,
            ["feature_name", "coverage_reason_class"],
        ].drop_duplicates()
        core_features = sorted(
            feature_reason_df.loc[
                feature_reason_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        core_feature_count = len(core_features)
        core_non_null_expression = primary_non_null_count_expression(core_features)
        temporal_missing_expression = " OR ".join(
            f"{duckdb_identifier(feature)} IS NULL"
            for feature in temporal_features
        )

        temporal_gap_query = f"""
        WITH complete_core_rows AS (
            SELECT
                taxi_zone_id,
                CAST(date AS DATE) AS date,
                temporal_bucket,
                ROW_NUMBER() OVER (
                    PARTITION BY taxi_zone_id, temporal_bucket
                    ORDER BY CAST(date AS DATE)
                ) AS sequence_position,
                CASE WHEN {temporal_missing_expression} THEN 1 ELSE 0 END AS has_temporal_gap
            FROM read_parquet('{panel_path_sql}')
            WHERE ({core_non_null_expression}) = {core_feature_count}
        )
        SELECT
            '{feature_set_name}' AS feature_set,
            COUNT(*) AS complete_core_rows,
            SUM(has_temporal_gap) AS rows_with_any_temporal_gap,
            SUM(CASE WHEN has_temporal_gap = 1 AND sequence_position = 1 THEN 1 ELSE 0 END) AS temporal_gap_rows_at_first_sequence_position,
            SUM(CASE WHEN has_temporal_gap = 1 AND sequence_position <= 7 THEN 1 ELSE 0 END) AS temporal_gap_rows_within_first_7_positions,
            COUNT(DISTINCT CASE WHEN has_temporal_gap = 1 THEN taxi_zone_id ELSE NULL END) AS zones_with_temporal_gap
        FROM complete_core_rows
        """
        temporal_gap_summary_records.append(con.execute(temporal_gap_query).fetchdf())

temporal_gap_summary_df = pd.concat(temporal_gap_summary_records, ignore_index=True)
temporal_gap_summary_df["temporal_gap_pct_of_complete_core"] = (
    temporal_gap_summary_df["rows_with_any_temporal_gap"]
    / temporal_gap_summary_df["complete_core_rows"].replace(0, np.nan)
).round(5)
temporal_gap_summary_df["first_position_share_of_temporal_gaps"] = (
    temporal_gap_summary_df["temporal_gap_rows_at_first_sequence_position"]
    / temporal_gap_summary_df["rows_with_any_temporal_gap"].replace(0, np.nan)
).round(5)
temporal_gap_summary_df["first_7_positions_share_of_temporal_gaps"] = (
    temporal_gap_summary_df["temporal_gap_rows_within_first_7_positions"]
    / temporal_gap_summary_df["rows_with_any_temporal_gap"].replace(0, np.nan)
).round(5)

temporal_gap_summary_display_df = (
    temporal_gap_summary_df
    [
        [
            "feature_set",
            "complete_core_rows",
            "rows_with_any_temporal_gap",
            "temporal_gap_pct_of_complete_core",
            "temporal_gap_rows_at_first_sequence_position",
            "first_position_share_of_temporal_gaps",
            "temporal_gap_rows_within_first_7_positions",
            "first_7_positions_share_of_temporal_gaps",
            "zones_with_temporal_gap",
        ]
    ]
    .sort_values("temporal_gap_pct_of_complete_core", ascending=False)
    .rename(
        columns={
            "complete_core_rows": "core_rows",
            "rows_with_any_temporal_gap": "temporal_gap_rows",
            "temporal_gap_pct_of_complete_core": "temporal_gap_pct",
            "temporal_gap_rows_at_first_sequence_position": "first_position_rows",
            "first_position_share_of_temporal_gaps": "first_position_share",
            "temporal_gap_rows_within_first_7_positions": "first_7_position_rows",
            "first_7_positions_share_of_temporal_gaps": "first_7_position_share",
            "zones_with_temporal_gap": "gap_zones",
        }
    )
    .reset_index(drop=True)
)

display(temporal_gap_summary_display_df)

print("Temporal-history gaps summarized among complete-core rows.")

,feature_set,core_rows,temporal_gap_rows,temporal_gap_pct,first_position_rows,first_position_share,first_7_position_rows,first_7_position_share,gap_zones
0,taxi,1125143,115225.0,0.10241,2616.0,0.02270,8710.0,0.07559,263
1,subway,925080,13625.0,0.01473,1560.0,0.11450,1576.0,0.11567,156
2,bus,1476478,3366.0,0.00228,2498.0,0.74213,2576.0,0.76530,252
3,fhvhv,1559590,2630.0,0.00169,2630.0,1.00000,2630.0,1.00000,263


Temporal-history gaps summarized among complete-core rows.


Findings\. Taxi is the only modality with a large temporal\-history gap: about 10% of complete\-core Taxi rows have at least one missing temporal descendant, and only a small share of those gaps occur at the first observation\. That points to intermittent Taxi service rather than a simple first\-day lag issue\. Subway has a modest gap of about 1\.5%, while Bus and FHVHV are very small\. Bus gaps are mostly first\-observation effects, and FHVHV gaps are entirely first\-observation effects in this view\. This supports simple complete\-row filtering for Bus, Subway, and FHVHV temporal leftovers, while Taxi needs a sparse\-service rule\.

For Taxi and FHVHV, inspect temporal\-history gaps against the underlying base metrics\. A demand/count gap may represent true zero activity, while a speed gap usually means the speed measure is not applicable because no trips were observed\. Percent\-change features need special care because they can be undefined when the previous value is zero or missing\.

In [14]:
# ---------------------------------------------------------------------
# Classify Taxi/FHVHV temporal gaps by base-metric availability
# ---------------------------------------------------------------------

TEMPORAL_BASE_SIGNAL_FEATURE_SETS = ["taxi", "fhvhv"]

temporal_base_signal_features_df = (
    remaining_primary_missing_features_df.loc[
        (remaining_primary_missing_features_df["feature_set"].isin(TEMPORAL_BASE_SIGNAL_FEATURE_SETS))
        & (remaining_primary_missing_features_df["coverage_reason_class"] == "temporal_history"),
        ["feature_set", "feature_name"],
    ]
    .merge(
        primary_unimodal_feature_catalog_df[
            ["feature_set", "feature_name", SOURCE_METRIC_COLUMN]
        ].drop_duplicates(),
        on=["feature_set", "feature_name"],
        how="left",
    )
    .dropna(subset=[SOURCE_METRIC_COLUMN])
    .rename(columns={SOURCE_METRIC_COLUMN: "source_metric"})
)

temporal_base_signal_records = []

with duckdb.connect() as con:
    for _, feature_row in temporal_base_signal_features_df.iterrows():
        feature_set_name = feature_row["feature_set"]
        feature_name = feature_row["feature_name"]
        source_metric = feature_row["source_metric"]

        feature_policy_df = primary_unimodal_feature_catalog_df.loc[
            primary_unimodal_feature_catalog_df["feature_set"] == feature_set_name
        ].copy()
        core_features = sorted(
            feature_policy_df.loc[
                feature_policy_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        core_feature_count = len(core_features)
        core_non_null_expression = primary_non_null_count_expression(core_features)

        base_signal_query = f"""
        WITH complete_core_rows AS (
            SELECT
                taxi_zone_id,
                CAST(date AS DATE) AS date,
                temporal_bucket,
                {duckdb_identifier(feature_name)} AS engineered_value,
                {duckdb_identifier(source_metric)} AS current_base_value,
                LAG({duckdb_identifier(source_metric)}) OVER (
                    PARTITION BY taxi_zone_id, temporal_bucket
                    ORDER BY CAST(date AS DATE)
                ) AS previous_base_value
            FROM read_parquet('{panel_path_sql}')
            WHERE ({core_non_null_expression}) = {core_feature_count}
              AND {primary_zone_exclusion_sql}
        )
        SELECT
            '{feature_set_name}' AS feature_set,
            '{feature_name}' AS feature_name,
            '{source_metric}' AS source_metric,
            COUNT(*) AS complete_core_rows,
            SUM(CASE WHEN engineered_value IS NULL THEN 1 ELSE 0 END) AS engineered_missing_rows,
            SUM(CASE WHEN engineered_value IS NULL AND current_base_value IS NULL THEN 1 ELSE 0 END) AS current_base_null_rows,
            SUM(CASE WHEN engineered_value IS NULL AND current_base_value = 0 THEN 1 ELSE 0 END) AS current_base_zero_rows,
            SUM(CASE WHEN engineered_value IS NULL AND current_base_value > 0 THEN 1 ELSE 0 END) AS current_base_positive_rows,
            SUM(CASE WHEN engineered_value IS NULL AND previous_base_value IS NULL THEN 1 ELSE 0 END) AS previous_base_null_rows,
            SUM(CASE WHEN engineered_value IS NULL AND previous_base_value = 0 THEN 1 ELSE 0 END) AS previous_base_zero_rows,
            SUM(CASE WHEN engineered_value IS NULL AND previous_base_value > 0 THEN 1 ELSE 0 END) AS previous_base_positive_rows
        FROM complete_core_rows
        """
        temporal_base_signal_records.append(con.execute(base_signal_query).fetchdf())

temporal_base_signal_df = pd.concat(temporal_base_signal_records, ignore_index=True)
temporal_base_signal_df["engineered_missing_pct_of_complete_core"] = (
    temporal_base_signal_df["engineered_missing_rows"]
    / temporal_base_signal_df["complete_core_rows"].replace(0, np.nan)
).round(5)

for column in [
    "current_base_null_rows",
    "current_base_zero_rows",
    "current_base_positive_rows",
    "previous_base_null_rows",
    "previous_base_zero_rows",
    "previous_base_positive_rows",
]:
    temporal_base_signal_df[column.replace("rows", "share_of_missing")] = (
        temporal_base_signal_df[column]
        / temporal_base_signal_df["engineered_missing_rows"].replace(0, np.nan)
    ).round(5)

base_signal_summary_columns = [
    "feature_set",
    "feature_name",
    "source_metric",
    "complete_core_rows",
    "engineered_missing_rows",
    "engineered_missing_pct_of_complete_core",
    "current_base_zero_share_of_missing",
    "previous_base_zero_share_of_missing",
    "current_base_positive_share_of_missing",
    "previous_base_positive_share_of_missing",
]

display(
    temporal_base_signal_df[base_signal_summary_columns]
    .sort_values(["feature_set", "engineered_missing_pct_of_complete_core", "feature_name"], ascending=[True, False, True])
    .reset_index(drop=True)
)

print("Taxi/FHVHV temporal gaps classified by base-metric availability.")

,feature_set,feature_name,source_metric,complete_core_rows,engineered_missing_rows,engineered_missing_pct_of_complete_core,current_base_zero_share_of_missing,previous_base_zero_share_of_missing,current_base_positive_share_of_missing,previous_base_positive_share_of_missing
0,fhvhv,fhvhv_avg_trip_speed_abs_change_1_same_bucket,fhvhv_avg_trip_speed,1541800,2600.0,0.00169,0.01692,0.00000,0.98308,0.00000
1,fhvhv,fhvhv_avg_trip_speed_lag_1_same_bucket,fhvhv_avg_trip_speed,1541800,2600.0,0.00169,0.01692,0.00000,0.98308,0.00000
2,fhvhv,fhvhv_avg_trip_speed_rolling_std_7_same_bucket,fhvhv_avg_trip_speed,1541800,2600.0,0.00169,0.01692,0.00000,0.98308,0.00000
3,fhvhv,fhvhv_trip_count_abs_change_1_same_bucket,fhvhv_trip_count,1541800,2600.0,0.00169,0.01692,0.00000,0.98308,0.00000
4,fhvhv,fhvhv_trip_count_lag_1_same_bucket,fhvhv_trip_count,1541800,2600.0,0.00169,0.01692,0.00000,0.98308,0.00000
5,fhvhv,fhvhv_trip_count_rolling_std_7_same_bucket,fhvhv_trip_count,1541800,2600.0,0.00169,0.01692,0.00000,0.98308,0.00000
6,taxi,taxi_avg_trip_speed_lag_1_same_bucket,taxi_avg_trip_speed,1113555,114889.0,0.10317,0.07833,0.07709,0.92167,0.90037
7,taxi,taxi_trip_count_pct_change_1_same_bucket,taxi_trip_count,1113555,114889.0,0.10317,0.00000,0.00000,1.00000,0.97747
8,taxi,taxi_avg_trip_speed_rolling_std_7_same_bucket,taxi_avg_trip_speed,1113555,15636.0,0.01404,0.10879,0.09638,0.89121,0.73804
9,taxi,taxi_trip_count_abs_change_1_same_bucket,taxi_trip_count,1113555,1491.0,0.00134,0.00000,0.00000,1.00000,0.00000


Taxi/FHVHV temporal gaps classified by base-metric availability.


Findings\. The Taxi/FHVHV temporal check separates two cases\. FHVHV's largest remaining gaps are concentrated in percent\-change and percent\-delta descendants, where zero observed activity can make the denominator undefined\. Those features should not be treated as evidence of bad rows\. Taxi's remaining gaps are more geographic: they reflect low\-continuity yellow\-cab service in sparse\-service zones, so those rows should be preserved with an explicit sparse\-service policy\.

Inspect the two dominant Taxi temporal gaps more closely\. The base\-metric diagnostic showed that these missing engineered values usually occur even when current and previous base values are positive, so the likely issue is not zero activity\. This check tests whether the engineered gaps are caused by irregular date history: the previous available same\-zone/same\-bucket observation may not be the exact expected previous calendar observation\.

In [15]:
# ---------------------------------------------------------------------
# Summarize Taxi temporal gaps by previous observation distance
# ---------------------------------------------------------------------

TAXI_TEMPORAL_GAP_FEATURES = [
    "taxi_avg_trip_speed_lag_1_same_bucket",
    "taxi_trip_count_pct_change_1_same_bucket",
]

TAXI_FEATURE_SET_NAME = "taxi"

taxi_temporal_gap_feature_metadata_df = (
    primary_unimodal_feature_catalog_df.loc[
        (primary_unimodal_feature_catalog_df["feature_set"] == TAXI_FEATURE_SET_NAME)
        & (primary_unimodal_feature_catalog_df["feature_name"].isin(TAXI_TEMPORAL_GAP_FEATURES)),
        ["feature_set", "feature_name", SOURCE_METRIC_COLUMN],
    ]
    .drop_duplicates()
    .rename(columns={SOURCE_METRIC_COLUMN: "source_metric"})
)

feature_reason_df = primary_unimodal_feature_catalog_df.loc[
    primary_unimodal_feature_catalog_df["feature_set"] == TAXI_FEATURE_SET_NAME,
    ["feature_name", "coverage_reason_class"],
].drop_duplicates()

taxi_core_features = sorted(
    feature_reason_df.loc[
        feature_reason_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
        "feature_name",
    ].tolist()
)
taxi_core_feature_count = len(taxi_core_features)
taxi_core_non_null_expression = primary_non_null_count_expression(taxi_core_features)

taxi_temporal_date_gap_records = []

with duckdb.connect() as con:
    for _, feature_row in taxi_temporal_gap_feature_metadata_df.iterrows():
        feature_name = feature_row["feature_name"]
        source_metric = feature_row["source_metric"]

        date_gap_query = f"""
        WITH same_bucket_history AS (
            SELECT
                taxi_zone_id,
                CAST(date AS DATE) AS date,
                temporal_bucket,
                {duckdb_identifier(feature_name)} AS engineered_value,
                LAG(CAST(date AS DATE)) OVER (
                    PARTITION BY taxi_zone_id, temporal_bucket
                    ORDER BY CAST(date AS DATE)
                ) AS previous_observation_date
            FROM read_parquet('{panel_path_sql}')
            WHERE ({taxi_core_non_null_expression}) = {taxi_core_feature_count}
              AND {primary_zone_exclusion_sql}
        ), classified AS (
            SELECT
                *,
                DATE_DIFF('day', previous_observation_date, date) AS days_since_previous_observation,
                CASE
                    WHEN previous_observation_date IS NULL THEN 'no_previous_observation'
                    WHEN DATE_DIFF('day', previous_observation_date, date) = 1 THEN 'previous_day'
                    WHEN DATE_DIFF('day', previous_observation_date, date) BETWEEN 2 AND 7 THEN 'previous_2_to_7_days'
                    WHEN DATE_DIFF('day', previous_observation_date, date) > 7 THEN 'previous_gt_7_days'
                    ELSE 'other_date_gap'
                END AS previous_observation_gap_class,
                CASE WHEN engineered_value IS NULL THEN 1 ELSE 0 END AS engineered_missing
            FROM same_bucket_history
        )
        SELECT
            '{feature_name}' AS feature_name,
            '{source_metric}' AS source_metric,
            previous_observation_gap_class,
            COUNT(*) AS complete_core_rows,
            SUM(engineered_missing) AS engineered_missing_rows,
            AVG(days_since_previous_observation) AS avg_days_since_previous_observation,
            COUNT(DISTINCT CASE WHEN engineered_missing = 1 THEN taxi_zone_id ELSE NULL END) AS zones_with_missing_engineered
        FROM classified
        GROUP BY 1, 2, 3
        """
        taxi_temporal_date_gap_records.append(con.execute(date_gap_query).fetchdf())

taxi_temporal_date_gap_df = pd.concat(taxi_temporal_date_gap_records, ignore_index=True)
taxi_temporal_date_gap_df["engineered_missing_pct_within_gap_class"] = (
    taxi_temporal_date_gap_df["engineered_missing_rows"]
    / taxi_temporal_date_gap_df["complete_core_rows"].replace(0, np.nan)
).round(5)

display(
    taxi_temporal_date_gap_df.sort_values(
        ["feature_name", "previous_observation_gap_class"]
    ).reset_index(drop=True)
)

print("Taxi temporal gaps summarized by previous observation distance.")

,feature_name,source_metric,previous_observation_gap_class,complete_core_rows,engineered_missing_rows,avg_days_since_previous_observation,zones_with_missing_engineered,engineered_missing_pct_within_gap_class
0,taxi_avg_trip_speed_lag_1_same_bucket,taxi_avg_trip_speed,no_previous_observation,2589,2589.0,NaN,260,1.00000
1,taxi_avg_trip_speed_lag_1_same_bucket,taxi_avg_trip_speed,previous_2_to_7_days,368899,84975.0,4.444225,213,0.23035
2,taxi_avg_trip_speed_lag_1_same_bucket,taxi_avg_trip_speed,previous_day,714742,0.0,1.000000,0,0.00000
3,taxi_avg_trip_speed_lag_1_same_bucket,taxi_avg_trip_speed,previous_gt_7_days,27325,27325.0,22.949021,186,1.00000
4,taxi_trip_count_pct_change_1_same_bucket,taxi_trip_count,no_previous_observation,2589,2589.0,NaN,260,1.00000
5,taxi_trip_count_pct_change_1_same_bucket,taxi_trip_count,previous_2_to_7_days,368899,84975.0,4.444225,213,0.23035
6,taxi_trip_count_pct_change_1_same_bucket,taxi_trip_count,previous_day,714742,0.0,1.000000,0,0.00000
7,taxi_trip_count_pct_change_1_same_bucket,taxi_trip_count,previous_gt_7_days,27325,27325.0,22.949021,186,1.00000


Taxi temporal gaps summarized by previous observation distance.


Findings\. Taxi's dominant temporal gaps are geographic service\-sparsity gaps, not random data loss\. Missing lag and percent\-change values are concentrated in outer and edge zones where yellow\-cab activity is intermittent: eastern Queens, southern Brooklyn, Staten Island, EWR, and similar low\-continuity areas\. This argues against dropping those rows from the primary Taxi matrix\.

### Resolve Primary Missingness Rules

The coverage diagnostics show that the remaining missingness is not one single problem\. Subway coverage is structural and should be evaluated only inside its station\-serving universe\. Bus missingness is small after interaction features are moved out\. FHVHV becomes clean once zero\-denominator percent\-change features and special zones are removed from the primary view\. Taxi is different: the largest gaps occur in sparse yellow\-cab geographies where same\-zone, same\-bucket history is intermittent, especially in outer Queens, southern Brooklyn, Staten Island, EWR, and edge\-zone areas\.

> ✅ Decision\. Primary single\-modality matrices should preserve legitimate sparse\-service observations instead of dropping them\. For Subway and Bus, the row universe remains coverage\-based because system presence is not universal\. For Taxi and FHVHV, the row universe starts from the eligible zone\-date\-bucket panel because those services are represented across the full dataset\. Demand absence is encoded as zero; unobserved speed or travel\-time values are neutral\-imputed with indicator columns during matrix construction so sparse service does not become fake zero\-speed congestion\.

Define the primary missing\-value policy at the feature level\. This policy does not rebuild features; it records how 3\.1\.1 should handle remaining nulls when constructing primary single\-modality matrices\.

In [16]:
# ---------------------------------------------------------------------
# Define primary missing-value policy for single-modality matrices
# ---------------------------------------------------------------------

FULL_UNIVERSE_SERVICE_FEATURE_SETS = {"taxi", "fhvhv"}
COVERAGE_LIMITED_FEATURE_SETS = {"bus", "subway"}

PRIMARY_ROW_UNIVERSE_RULE_BY_SET = {
    "bus": "complete_core_coverage_universe",
    "subway": "complete_core_coverage_universe",
    "taxi": "eligible_zone_universe_with_sparse_service_imputation",
    "fhvhv": "eligible_zone_universe_with_sparse_service_imputation",
}


def infer_primary_signal_type(feature_name, source_metric):
    feature_text = f"{feature_name} {source_metric}".lower()

    if any(token in feature_text for token in ["trip_count", "ridership", "pickup_count", "dropoff_count", "count", "volume"]):
        return "demand_or_count"
    if any(token in feature_text for token in ["speed", "travel_time", "trip_distance", "distance_traveled", "duration", "rate"]):
        return "continuous_trip_measure"
    if any(token in feature_text for token in ["distance_to_", "neighbor", "connected_zone", "cbd", "gateway"]):
        return "spatial_or_context"
    return "other"


def assign_primary_null_action(feature_set, signal_type):
    if feature_set in FULL_UNIVERSE_SERVICE_FEATURE_SETS and signal_type == "demand_or_count":
        return "zero_fill_sparse_service_absence"
    if feature_set in FULL_UNIVERSE_SERVICE_FEATURE_SETS and signal_type == "continuous_trip_measure":
        return "neutral_impute_with_missing_indicator"
    return "complete_case_required"


primary_missing_value_policy_df = primary_unimodal_feature_catalog_df.copy()
primary_missing_value_policy_df["primary_row_universe_rule"] = primary_missing_value_policy_df["feature_set"].map(
    PRIMARY_ROW_UNIVERSE_RULE_BY_SET
)
primary_missing_value_policy_df["primary_signal_type"] = primary_missing_value_policy_df.apply(
    lambda row: infer_primary_signal_type(
        row["feature_name"],
        row.get(SOURCE_METRIC_COLUMN, ""),
    ),
    axis=1,
)
primary_missing_value_policy_df["primary_null_action"] = primary_missing_value_policy_df.apply(
    lambda row: assign_primary_null_action(
        row["feature_set"],
        row["primary_signal_type"],
    ),
    axis=1,
)
primary_missing_value_policy_df["adds_missing_indicator"] = (
    primary_missing_value_policy_df["primary_null_action"] == "neutral_impute_with_missing_indicator"
)

primary_missing_policy_feature_df = primary_missing_value_policy_df[
    [
        "feature_set",
        "feature_name",
        "coverage_reason_class",
        SOURCE_METRIC_COLUMN,
        "primary_signal_type",
        "primary_null_action",
        "adds_missing_indicator",
    ]
].copy()

primary_missing_policy_summary_df = (
    primary_missing_value_policy_df
    .groupby(
        [
            "feature_set",
            "primary_row_universe_rule",
            "primary_signal_type",
            "primary_null_action",
            "adds_missing_indicator",
        ],
        dropna=False,
    )["feature_name"]
    .nunique()
    .reset_index(name="feature_count")
    .sort_values(["feature_set", "primary_null_action", "primary_signal_type"])
    .reset_index(drop=True)
)

display(primary_missing_policy_summary_df)

print("Primary missing-value policy defined for single-modality matrices.")

,feature_set,primary_row_universe_rule,primary_signal_type,primary_null_action,adds_missing_indicator,feature_count
0,bus,complete_core_coverage_universe,continuous_trip_measure,complete_case_required,False,19
1,bus,complete_core_coverage_universe,demand_or_count,complete_case_required,False,16
2,bus,complete_core_coverage_universe,spatial_or_context,complete_case_required,False,5
3,fhvhv,eligible_zone_universe_with_sparse_service_imp...,spatial_or_context,complete_case_required,False,5
4,fhvhv,eligible_zone_universe_with_sparse_service_imp...,continuous_trip_measure,neutral_impute_with_missing_indicator,True,16
5,fhvhv,eligible_zone_universe_with_sparse_service_imp...,demand_or_count,zero_fill_sparse_service_absence,False,16
6,subway,complete_core_coverage_universe,demand_or_count,complete_case_required,False,15
7,subway,complete_core_coverage_universe,other,complete_case_required,False,1
8,subway,complete_core_coverage_universe,spatial_or_context,complete_case_required,False,5
9,taxi,eligible_zone_universe_with_sparse_service_imp...,spatial_or_context,complete_case_required,False,5


Primary missing-value policy defined for single-modality matrices.


Findings\. The policy table turns the discovery work into reusable matrix\-construction rules\. Bus and Subway remain coverage\-limited workflows\. Taxi and FHVHV are treated as full\-universe sparse\-service workflows: demand/count nulls encode no observed activity, while speed, travel\-time, and distance\-like trip measures are imputed neutrally and flagged with missingness indicators\.

Recalculate primary coverage using the resolved row\-universe and missing\-value policy\. This check separates fillable sparse\-service nulls from non\-fillable nulls that would still block matrix construction\.

In [17]:
# ---------------------------------------------------------------------
# Recalculate effective primary coverage after missing-value policy
# ---------------------------------------------------------------------


def sql_and(expressions):
    expressions = [expression for expression in expressions if expression]
    return " AND ".join(expressions) if expressions else "TRUE"


def missing_any_expression_or_false(features):
    if not features:
        return "FALSE"
    return " OR ".join(
        f"{duckdb_identifier(feature)} IS NULL"
        for feature in features
    )


primary_policy_coverage_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_policy_df = primary_missing_value_policy_df.loc[
            primary_missing_value_policy_df["feature_set"] == feature_set_name
        ].copy()

        primary_features = sorted(feature_policy_df["feature_name"].tolist())
        fillable_features = sorted(
            feature_policy_df.loc[
                feature_policy_df["primary_null_action"].isin(
                    [
                        "zero_fill_sparse_service_absence",
                        "neutral_impute_with_missing_indicator",
                    ]
                ),
                "feature_name",
            ].tolist()
        )
        required_complete_features = sorted(
            feature_policy_df.loc[
                ~feature_policy_df["primary_null_action"].isin(
                    [
                        "zero_fill_sparse_service_absence",
                        "neutral_impute_with_missing_indicator",
                    ]
                ),
                "feature_name",
            ].tolist()
        )

        core_features = sorted(
            feature_policy_df.loc[
                feature_policy_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        core_feature_count = len(core_features)

        row_filters = [primary_zone_exclusion_sql]
        if PRIMARY_ROW_UNIVERSE_RULE_BY_SET[feature_set_name] == "complete_core_coverage_universe":
            row_filters.append(f"({primary_non_null_count_expression(core_features)}) = {core_feature_count}")

        row_filter_sql = sql_and(row_filters)
        fillable_missing_expression = missing_any_expression_or_false(fillable_features)
        required_missing_expression = missing_any_expression_or_false(required_complete_features)

        coverage_query = f"""
        SELECT
            '{feature_set_name}' AS feature_set,
            '{PRIMARY_ROW_UNIVERSE_RULE_BY_SET[feature_set_name]}' AS row_universe_rule,
            COUNT(*) AS row_universe_rows,
            {len(primary_features)} AS primary_feature_count,
            {len(fillable_features)} AS fillable_feature_count,
            {len(required_complete_features)} AS required_complete_feature_count,
            SUM(CASE WHEN {fillable_missing_expression} THEN 1 ELSE 0 END) AS rows_needing_fill_or_imputation,
            SUM(CASE WHEN {required_missing_expression} THEN 1 ELSE 0 END) AS rows_with_nonfillable_nulls,
            COUNT(*) - SUM(CASE WHEN {required_missing_expression} THEN 1 ELSE 0 END) AS effective_primary_rows
        FROM read_parquet('{panel_path_sql}')
        WHERE {row_filter_sql}
        """
        primary_policy_coverage_records.append(con.execute(coverage_query).fetchdf())

primary_policy_coverage_df = pd.concat(primary_policy_coverage_records, ignore_index=True)
primary_policy_coverage_df["rows_needing_fill_or_imputation_pct"] = (
    primary_policy_coverage_df["rows_needing_fill_or_imputation"]
    / primary_policy_coverage_df["row_universe_rows"].replace(0, np.nan)
).round(5)
primary_policy_coverage_df["nonfillable_null_pct"] = (
    primary_policy_coverage_df["rows_with_nonfillable_nulls"]
    / primary_policy_coverage_df["row_universe_rows"].replace(0, np.nan)
).round(5)
primary_policy_coverage_df["effective_primary_row_pct"] = (
    primary_policy_coverage_df["effective_primary_rows"]
    / primary_policy_coverage_df["row_universe_rows"].replace(0, np.nan)
).round(5)

primary_policy_coverage_df = primary_policy_coverage_df.sort_values("feature_set").reset_index(drop=True)

display(primary_policy_coverage_df)

print("Effective primary coverage recalculated after missing-value policy.")

,feature_set,row_universe_rule,row_universe_rows,primary_feature_count,fillable_feature_count,required_complete_feature_count,rows_needing_fill_or_imputation,rows_with_nonfillable_nulls,effective_primary_rows,rows_needing_fill_or_imputation_pct,nonfillable_null_pct,effective_primary_row_pct
0,bus,complete_core_coverage_universe,1476478,40,0,40,0.0,3531.0,1472947.0,0.00000,0.00239,0.99761
1,fhvhv,eligible_zone_universe_with_sparse_service_imp...,1541800,37,32,5,4171.0,0.0,1541800.0,0.00271,0.00000,1.00000
2,subway,complete_core_coverage_universe,925080,21,0,21,0.0,13625.0,911455.0,0.00000,0.01473,0.98527
3,taxi,eligible_zone_universe_with_sparse_service_imp...,1541800,38,33,5,545925.0,0.0,1541800.0,0.35408,0.00000,1.00000


Effective primary coverage recalculated after missing-value policy.


Findings\. The resolved coverage check shows why the policy is needed\. Bus and Subway can use complete\-case filtering within their coverage universes because their remaining losses are small and coverage\-specific\. Taxi and FHVHV preserve their eligible sparse\-service rows by treating demand absence as zero and neutral\-imputing unobserved continuous trip measures with indicator columns\.

If any non\-fillable nulls remain, inspect only those leftovers\. This block should be unnecessary once the policy fully resolves the four primary modality matrices, but it provides a compact audit trail if a feature still needs a targeted rule\.

In [18]:
# ---------------------------------------------------------------------
# Inspect remaining non-fillable primary nulls after policy
# ---------------------------------------------------------------------

remaining_nonfillable_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_policy_df = primary_missing_value_policy_df.loc[
            primary_missing_value_policy_df["feature_set"] == feature_set_name
        ].copy()
        required_complete_features = sorted(
            feature_policy_df.loc[
                ~feature_policy_df["primary_null_action"].isin(
                    [
                        "zero_fill_sparse_service_absence",
                        "neutral_impute_with_missing_indicator",
                    ]
                ),
                "feature_name",
            ].tolist()
        )
        if not required_complete_features:
            continue

        core_features = sorted(
            feature_policy_df.loc[
                feature_policy_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        core_feature_count = len(core_features)

        row_filters = [primary_zone_exclusion_sql]
        if PRIMARY_ROW_UNIVERSE_RULE_BY_SET[feature_set_name] == "complete_core_coverage_universe":
            row_filters.append(f"({primary_non_null_count_expression(core_features)}) = {core_feature_count}")
        row_filter_sql = sql_and(row_filters)

        null_expressions = [
            f"SUM(CASE WHEN {duckdb_identifier(feature)} IS NULL THEN 1 ELSE 0 END) AS {duckdb_identifier(feature)}"
            for feature in required_complete_features
        ]

        feature_query = f"""
        SELECT
            COUNT(*) AS row_universe_rows,
            {', '.join(null_expressions)}
        FROM read_parquet('{panel_path_sql}')
        WHERE {row_filter_sql}
        """
        feature_result = con.execute(feature_query).fetchdf().iloc[0]
        row_universe_rows = int(feature_result["row_universe_rows"])

        for feature in required_complete_features:
            missing_rows = int(feature_result[feature])
            if missing_rows == 0:
                continue
            feature_metadata = feature_policy_df.loc[
                feature_policy_df["feature_name"] == feature
            ].iloc[0]
            remaining_nonfillable_records.append(
                {
                    "feature_set": feature_set_name,
                    "feature_name": feature,
                    "coverage_reason_class": feature_metadata["coverage_reason_class"],
                    "primary_signal_type": feature_metadata["primary_signal_type"],
                    "primary_null_action": feature_metadata["primary_null_action"],
                    "row_universe_rows": row_universe_rows,
                    "missing_rows": missing_rows,
                    "missing_pct": round(missing_rows / row_universe_rows, 5) if row_universe_rows else np.nan,
                }
            )

remaining_nonfillable_primary_nulls_df = pd.DataFrame(remaining_nonfillable_records)
if not remaining_nonfillable_primary_nulls_df.empty:
    remaining_nonfillable_primary_nulls_df = remaining_nonfillable_primary_nulls_df.sort_values(
        ["feature_set", "missing_pct", "feature_name"],
        ascending=[True, False, True],
    ).reset_index(drop=True)
    display(remaining_nonfillable_primary_nulls_df)
else:
    display(pd.DataFrame({"status": ["no remaining non-fillable primary nulls"]}))

print("Remaining non-fillable primary nulls inspected.")

,feature_set,feature_name,coverage_reason_class,primary_signal_type,primary_null_action,row_universe_rows,missing_rows,missing_pct
0,bus,avg_bus_speed_abs_change_1_same_bucket,temporal_history,continuous_trip_measure,complete_case_required,1476478,2926,0.00198
1,bus,avg_bus_speed_lag_1_same_bucket,temporal_history,continuous_trip_measure,complete_case_required,1476478,2926,0.00198
2,bus,avg_bus_speed_pct_change_1_same_bucket,temporal_history,continuous_trip_measure,complete_case_required,1476478,2926,0.00198
3,bus,bus_trip_count_abs_change_1_same_bucket,temporal_history,demand_or_count,complete_case_required,1476478,2926,0.00198
4,bus,bus_trip_count_lag_1_same_bucket,temporal_history,demand_or_count,complete_case_required,1476478,2926,0.00198
5,bus,bus_trip_count_pct_change_1_same_bucket,temporal_history,demand_or_count,complete_case_required,1476478,2926,0.00198
6,bus,avg_bus_speed_rolling_std_7_same_bucket,temporal_history,continuous_trip_measure,complete_case_required,1476478,2514,0.00170
7,bus,bus_trip_count_rolling_std_7_same_bucket,temporal_history,demand_or_count,complete_case_required,1476478,2514,0.00170
8,bus,avg_bus_speed_fourier20_residual_reconstructed,decomposition,continuous_trip_measure,complete_case_required,1476478,650,0.00044
9,bus,avg_bus_speed_residual,decomposition,continuous_trip_measure,complete_case_required,1476478,650,0.00044


Remaining non-fillable primary nulls inspected.


Findings\. The remaining\-null audit confirms the resolution logic: non\-fillable nulls are no longer an unresolved Taxi/FHVHV problem\. Any required\-complete nulls that remain belong to coverage\-limited Bus/Subway edge cases and will be handled by the documented row universe rather than by imputation\.

### Summarize the Coverage Resolution Journey

This final coverage summary turns the modality\-by\-modality investigation into a compact handoff table\. The earlier blocks diagnosed why missingness appears: small coverage\-limited gaps for Bus and Subway, sparse\-service continuity issues for Taxi, and unstable percent\-style descendants for FHVHV\. This subsection summarizes those findings as calculated QA outputs rather than hardcoded notes\.

The goal is to show, for each primary modality, how many rows were at risk before resolution, which policy resolves the issue, and what null\-clean state is expected before matrix construction\. This is the bridge from exploration to implementation: the next section applies these decisions mechanically and verifies that the prepared primary inputs contain no remaining feature nulls\.

In [19]:
# ---------------------------------------------------------------------
# Build the primary modality resolution overview
# ---------------------------------------------------------------------

if "primary_coverage_after_exclusion_df" in globals():
    primary_resolution_source_df = primary_coverage_after_exclusion_df.copy()
elif "primary_coverage_after_interaction_exclusion_df" in globals():
    primary_resolution_source_df = primary_coverage_after_interaction_exclusion_df.copy()
else:
    raise NameError(
        "Run the coverage recalculation block before this summary. "
        "Expected primary_coverage_after_exclusion_df or "
        "primary_coverage_after_interaction_exclusion_df to exist."
    )

required_resolution_source_columns = [
    "feature_set",
    "eligible_rows_after_zone_exclusion",
    "complete_core_rows",
    "primary_feature_count_after_exclusion",
    "complete_core_but_incomplete_primary_rows",
    "incomplete_primary_pct_of_complete_core",
]
missing_resolution_source_columns = [
    column for column in required_resolution_source_columns
    if column not in primary_resolution_source_df.columns
]
assert not missing_resolution_source_columns, (
    "Coverage summary is missing required columns: "
    f"{missing_resolution_source_columns}"
)

modality_resolution_label_by_set = {
    "bus": "coverage-aware complete-row filtering",
    "subway": "coverage-aware complete-row filtering",
    "taxi": "sparse-service fill/imputation policy",
    "fhvhv": "sparse-service fill/imputation policy plus pct-feature exclusion",
}

primary_resolution_overview_df = primary_resolution_source_df[
    required_resolution_source_columns
].copy()
primary_resolution_overview_df = primary_resolution_overview_df.rename(
    columns={
        "eligible_rows_after_zone_exclusion": "eligible_rows",
        "primary_feature_count_after_exclusion": "primary_feature_count",
        "complete_core_but_incomplete_primary_rows": "rows_with_primary_nulls_before_resolution",
        "incomplete_primary_pct_of_complete_core": "rows_with_primary_nulls_before_resolution_pct",
    }
)
primary_resolution_overview_df["rows_with_primary_nulls_before_resolution_pct"] = (
    primary_resolution_overview_df["rows_with_primary_nulls_before_resolution_pct"] * 100
).round(3)
primary_resolution_overview_df["primary_resolution"] = primary_resolution_overview_df["feature_set"].map(
    modality_resolution_label_by_set
)
primary_resolution_overview_df["expected_remaining_null_pct_after_resolution"] = 0.0
primary_resolution_overview_df["resolution_status"] = "resolved by documented policy"
primary_resolution_overview_df = primary_resolution_overview_df.sort_values(
    "feature_set"
).reset_index(drop=True)

display(primary_resolution_overview_df)

print("Primary modality resolution overview built from coverage diagnostics.")

,feature_set,eligible_rows,complete_core_rows,primary_feature_count,rows_with_primary_nulls_before_resolution,rows_with_primary_nulls_before_resolution_pct,primary_resolution,expected_remaining_null_pct_after_resolution,resolution_status
0,bus,1541800,1476478,40,3531,0.239,coverage-aware complete-row filtering,0.0,resolved by documented policy
1,fhvhv,1541800,1541800,37,4171,0.271,sparse-service fill/imputation policy plus pct...,0.0,resolved by documented policy
2,subway,1541800,925080,21,13625,1.473,coverage-aware complete-row filtering,0.0,resolved by documented policy
3,taxi,1541800,1113555,38,117680,10.568,sparse-service fill/imputation policy,0.0,resolved by documented policy


Primary modality resolution overview built from coverage diagnostics.


Findings\. The primary modality review now has a clear resolution path for all four systems\. The key columns are \`complete\_core\_rows\`, which defines the usable signal universe for each mode, and \`rows\_with\_primary\_nulls\_before\_resolution\_pct\`, which shows how destructive complete\-case filtering would be after the major intermodal features were removed\. Bus has only 3,531 complete\-core rows with remaining primary nulls, or 0\.239%, so coverage\-aware complete\-row filtering is a low\-loss choice\. Subway has a smaller applicable universe: 925,080 complete\-core rows out of 1,541,800 eligible rows, reflecting the fact that Subway is structurally unavailable in many Taxi Zones\. Within that subway\-serving universe, 13,625 rows, or 1\.473%, still have primary nulls and can be handled with complete\-row filtering\. Taxi is the major exception: 117,680 complete\-core rows, or 10\.568%, have primary nulls before resolution, so dropping rows would remove meaningful sparse\-service geography\. FHVHV has only 4,171 affected rows, or 0\.271%, after unstable percentage descendants are moved out of the primary feature set\. The documented policy resolves each modality with an expected 0% remaining null rate before matrix construction\.

### Section Summary

> The modality coverage review shows that primary matrix preparation needs system\-specific rules\. Bus and Subway can be handled with coverage\-aware complete\-row filtering because their remaining nulls are small and tied to their applicable service universes\. Taxi and FHVHV should preserve sparse\-service observations: missing demand represents no observed activity, while missing continuous trip measures should be neutral\-imputed and flagged\. These decisions establish the primary modality policy used in the next section\.

## 3\.1\.1\.3 Finalize Resolved Primary Modality Inputs

The coverage review above leaves us with a practical set of modality\-specific rules\. Bus and Subway need coverage\-aware row universes\. Taxi and FHVHV need sparse\-service handling so true absence of observed service does not become row loss\. This section turns those decisions into reusable preparation assets for the rest of the notebook\.

The output of this section is not a modeling matrix yet\. It is the cleaned handoff layer: resolved row filters, feature handling rules, neutral imputation values, missingness indicators, and QA checks showing that the four primary modality inputs can be made null\-clean before intermodal and multimodal coverage are evaluated\.

Compute neutral imputation values for continuous Taxi and FHVHV trip measures\. These values are used only when service is absent or unobserved; demand/count features use zero\-fill instead\.

In [20]:
# ---------------------------------------------------------------------
# Compute neutral imputation values for continuous sparse-service features
# ---------------------------------------------------------------------

continuous_imputation_features_df = primary_missing_value_policy_df.loc[
    primary_missing_value_policy_df["primary_null_action"] == "neutral_impute_with_missing_indicator",
    ["feature_set", "feature_name", "primary_signal_type", "primary_null_action"],
].drop_duplicates()

continuous_imputation_records = []

with duckdb.connect() as con:
    for _, feature_row in continuous_imputation_features_df.iterrows():
        feature_set_name = feature_row["feature_set"]
        feature_name = feature_row["feature_name"]

        imputation_query = f"""
        SELECT
            '{feature_set_name}' AS feature_set,
            '{feature_name}' AS feature_name,
            COUNT(*) AS eligible_rows,
            SUM(CASE WHEN {duckdb_identifier(feature_name)} IS NULL THEN 1 ELSE 0 END) AS missing_rows,
            MEDIAN({duckdb_identifier(feature_name)}) AS neutral_imputation_value
        FROM read_parquet('{panel_path_sql}')
        WHERE {primary_zone_exclusion_sql}
        """
        continuous_imputation_records.append(con.execute(imputation_query).fetchdf())

primary_continuous_imputation_values_df = pd.concat(
    continuous_imputation_records,
    ignore_index=True,
) if continuous_imputation_records else pd.DataFrame(
    columns=[
        "feature_set",
        "feature_name",
        "eligible_rows",
        "missing_rows",
        "neutral_imputation_value",
    ]
)

primary_continuous_imputation_values_df["missing_pct"] = (
    primary_continuous_imputation_values_df["missing_rows"]
    / primary_continuous_imputation_values_df["eligible_rows"].replace(0, np.nan)
).round(5)

missing_imputation_values = primary_continuous_imputation_values_df.loc[
    primary_continuous_imputation_values_df["neutral_imputation_value"].isna(),
    ["feature_set", "feature_name"],
]
assert missing_imputation_values.empty, (
    "Continuous sparse-service features need non-null imputation values:\n"
    + missing_imputation_values.to_string(index=False)
)

display(
    primary_continuous_imputation_values_df.sort_values(
        ["feature_set", "missing_pct", "feature_name"],
        ascending=[True, False, True],
    ).reset_index(drop=True)
)

print("Neutral imputation values computed for continuous sparse-service features.")

,feature_set,feature_name,eligible_rows,missing_rows,neutral_imputation_value,missing_pct
0,fhvhv,fhvhv_avg_trip_speed_abs_change_1_same_bucket,1541800,2600.0,-0.012575,0.00169
1,fhvhv,fhvhv_avg_trip_speed_lag_1_same_bucket,1541800,2600.0,14.500526,0.00169
2,fhvhv,fhvhv_avg_trip_speed_rolling_std_7_same_bucket,1541800,2600.0,0.828729,0.00169
3,fhvhv,fhvhv_avg_trip_speed_deviation_abs_zscore,1541800,1573.0,0.610853,0.00102
4,fhvhv,fhvhv_avg_trip_speed_fourier20_residual_recons...,1541800,339.0,0.032493,0.00022
5,fhvhv,fhvhv_avg_trip_speed_residual,1541800,339.0,0.003246,0.00022
6,fhvhv,fhvhv_avg_trip_speed_residual_abs,1541800,339.0,0.254836,0.00022
7,fhvhv,fhvhv_avg_trip_speed_residual_zscore,1541800,339.0,-0.070829,0.00022
8,fhvhv,fhvhv_avg_trip_speed_seasonal,1541800,339.0,-0.018504,0.00022
9,fhvhv,borough_mean_fhvhv_avg_trip_speed,1541800,0.0,14.508802,0.00000


Neutral imputation values computed for continuous sparse-service features.


Findings\. This table defines the neutral values used when Taxi and FHVHV continuous trip measures are missing under the sparse\-service policy\. The main takeaway is the missingness contrast between modes, not the individual median for every feature\. FHVHV is nearly complete: its largest remaining continuous\-feature gap is 2,600 rows, or 0\.169%, and many continuous features have no missing values at all\. Taxi is much sparser: several speed and decomposition features are missing for roughly 428,000 to 440,000 rows, about 27\.8% to 28\.5% of eligible rows\. That supports the earlier decision not to drop Taxi rows\. The imputation values are ordinary median values from observed service, and the missing indicators preserve the fact that those Taxi observations came from sparse\-service conditions rather than fully observed trip\-speed records\.

Create the resolved primary row universe for each modality\. Coverage\-limited modes use complete rows inside their applicable universe; full\-universe sparse\-service modes retain eligible zone\-date\-bucket rows and rely on the fill/imputation policy\.

In [21]:
# ---------------------------------------------------------------------
# Define resolved primary row universes by modality
# ---------------------------------------------------------------------

FILLABLE_NULL_ACTIONS = {
    "zero_fill_sparse_service_absence",
    "neutral_impute_with_missing_indicator",
}

primary_resolved_row_filter_sql_by_set = {}
primary_resolved_row_universe_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_policy_df = primary_missing_value_policy_df.loc[
            primary_missing_value_policy_df["feature_set"] == feature_set_name
        ].copy()

        required_complete_features = sorted(
            feature_policy_df.loc[
                ~feature_policy_df["primary_null_action"].isin(FILLABLE_NULL_ACTIONS),
                "feature_name",
            ].tolist()
        )
        core_features = sorted(
            feature_policy_df.loc[
                feature_policy_df["coverage_reason_class"].isin(CORE_SIGNAL_REASON_CLASSES),
                "feature_name",
            ].tolist()
        )
        core_feature_count = len(core_features)

        base_filters = [primary_zone_exclusion_sql]
        if PRIMARY_ROW_UNIVERSE_RULE_BY_SET[feature_set_name] == "complete_core_coverage_universe":
            base_filters.append(f"({primary_non_null_count_expression(core_features)}) = {core_feature_count}")

        base_filter_sql = sql_and(base_filters)
        required_complete_filter_sql = sql_and(
            [f"{duckdb_identifier(feature)} IS NOT NULL" for feature in required_complete_features]
        )
        resolved_filter_sql = sql_and([base_filter_sql, required_complete_filter_sql])
        primary_resolved_row_filter_sql_by_set[feature_set_name] = resolved_filter_sql

        row_count_query = f"""
        SELECT
            '{feature_set_name}' AS feature_set,
            '{PRIMARY_ROW_UNIVERSE_RULE_BY_SET[feature_set_name]}' AS row_universe_rule,
            COUNT(*) AS starting_rows,
            SUM(CASE WHEN {resolved_filter_sql} THEN 1 ELSE 0 END) AS resolved_primary_rows
        FROM read_parquet('{panel_path_sql}')
        WHERE {base_filter_sql}
        """
        primary_resolved_row_universe_records.append(con.execute(row_count_query).fetchdf())

primary_resolved_row_universe_df = pd.concat(
    primary_resolved_row_universe_records,
    ignore_index=True,
)
primary_resolved_row_universe_df["dropped_rows_for_required_nulls"] = (
    primary_resolved_row_universe_df["starting_rows"]
    - primary_resolved_row_universe_df["resolved_primary_rows"]
)
primary_resolved_row_universe_df["resolved_primary_row_pct"] = (
    primary_resolved_row_universe_df["resolved_primary_rows"]
    / primary_resolved_row_universe_df["starting_rows"].replace(0, np.nan)
).round(5)

primary_resolved_row_universe_df = primary_resolved_row_universe_df.sort_values("feature_set").reset_index(drop=True)

display(primary_resolved_row_universe_df)

print("Resolved primary row universes defined by modality.")

,feature_set,row_universe_rule,starting_rows,resolved_primary_rows,dropped_rows_for_required_nulls,resolved_primary_row_pct
0,bus,complete_core_coverage_universe,1476478,1472947.0,3531.0,0.99761
1,fhvhv,eligible_zone_universe_with_sparse_service_imp...,1541800,1541800.0,0.0,1.00000
2,subway,complete_core_coverage_universe,925080,911455.0,13625.0,0.98527
3,taxi,eligible_zone_universe_with_sparse_service_imp...,1541800,1541800.0,0.0,1.00000


Resolved primary row universes defined by modality.


Findings\. The resolved row universes apply two different strategies\. Bus and Subway use complete\-core coverage universes, then drop the remaining rows with required primary nulls\. This removes 3,531 Bus rows and 13,625 Subway rows, leaving 99\.761% of Bus complete\-core rows and 98\.527% of Subway complete\-core rows\. Taxi and FHVHV use the eligible\-zone universe with sparse\-service fill and imputation, so no rows are dropped at this stage\. That is the important takeaway: sparse Taxi/FHVHV service is preserved as modeling signal, while Bus/Subway coverage\-limited gaps are handled through row eligibility\.

Validate the resolved primary modality inputs after applying the cleanup policy\. This final QA table should show zero remaining feature\-null issues for Bus, Subway, Taxi, and FHVHV\.

In [22]:
# ---------------------------------------------------------------------
# Validate resolved primary modality null status after cleanup policy
# ---------------------------------------------------------------------

imputation_value_lookup = {
    (row["feature_set"], row["feature_name"]): row["neutral_imputation_value"]
    for _, row in primary_continuous_imputation_values_df.iterrows()
}

resolved_null_qa_records = []

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_policy_df = primary_missing_value_policy_df.loc[
            primary_missing_value_policy_df["feature_set"] == feature_set_name
        ].copy()
        primary_features = sorted(feature_policy_df["feature_name"].tolist())
        indicator_features = sorted(
            feature_policy_df.loc[
                feature_policy_df["adds_missing_indicator"],
                "feature_name",
            ].tolist()
        )

        null_check_expressions = []
        for _, feature_row in feature_policy_df.iterrows():
            feature_name = feature_row["feature_name"]
            null_action = feature_row["primary_null_action"]

            if null_action == "zero_fill_sparse_service_absence":
                transformed_expression = f"COALESCE({duckdb_identifier(feature_name)}, 0)"
            elif null_action == "neutral_impute_with_missing_indicator":
                imputation_value = imputation_value_lookup[(feature_set_name, feature_name)]
                transformed_expression = f"COALESCE({duckdb_identifier(feature_name)}, {float(imputation_value):.17g})"
            else:
                transformed_expression = duckdb_identifier(feature_name)

            null_check_expressions.append(
                f"SUM(CASE WHEN {transformed_expression} IS NULL THEN 1 ELSE 0 END)"
            )

        total_null_expression = " + ".join(null_check_expressions) if null_check_expressions else "0"
        resolved_filter_sql = primary_resolved_row_filter_sql_by_set[feature_set_name]

        resolved_null_query = f"""
        SELECT
            '{feature_set_name}' AS feature_set,
            COUNT(*) AS resolved_primary_rows,
            {len(primary_features)} AS primary_feature_count,
            {len(indicator_features)} AS missing_indicator_feature_count,
            ({total_null_expression}) AS remaining_null_feature_cells
        FROM read_parquet('{panel_path_sql}')
        WHERE {resolved_filter_sql}
        """
        resolved_null_qa_records.append(con.execute(resolved_null_query).fetchdf())

primary_resolved_null_qa_df = pd.concat(resolved_null_qa_records, ignore_index=True)
primary_resolved_null_qa_df["resolved_model_column_count"] = (
    primary_resolved_null_qa_df["primary_feature_count"]
    + primary_resolved_null_qa_df["missing_indicator_feature_count"]
)
primary_resolved_null_qa_df["resolved_feature_cells"] = (
    primary_resolved_null_qa_df["resolved_primary_rows"]
    * primary_resolved_null_qa_df["resolved_model_column_count"]
)
primary_resolved_null_qa_df["remaining_null_feature_cell_pct"] = (
    primary_resolved_null_qa_df["remaining_null_feature_cells"]
    / primary_resolved_null_qa_df["resolved_feature_cells"].replace(0, np.nan)
).round(5)
primary_resolved_null_qa_df["null_status"] = np.where(
    primary_resolved_null_qa_df["remaining_null_feature_cells"] == 0,
    "resolved",
    "review",
)

primary_resolved_null_qa_df = primary_resolved_null_qa_df[
    [
        "feature_set",
        "resolved_primary_rows",
        "primary_feature_count",
        "missing_indicator_feature_count",
        "resolved_model_column_count",
        "remaining_null_feature_cells",
        "remaining_null_feature_cell_pct",
        "null_status",
    ]
].sort_values("feature_set").reset_index(drop=True)

display(primary_resolved_null_qa_df)

assert (primary_resolved_null_qa_df["remaining_null_feature_cells"] == 0).all(), (
    "Resolved primary modality inputs still contain null feature cells."
)

print("Resolved primary modality inputs have zero remaining feature-null cells.")

,feature_set,resolved_primary_rows,primary_feature_count,missing_indicator_feature_count,resolved_model_column_count,remaining_null_feature_cells,remaining_null_feature_cell_pct,null_status
0,bus,1472947,40,0,40,0.0,0.0,resolved
1,fhvhv,1541800,37,16,53,0.0,0.0,resolved
2,subway,911455,21,0,21,0.0,0.0,resolved
3,taxi,1541800,38,15,53,0.0,0.0,resolved


Resolved primary modality inputs have zero remaining feature-null cells.


Findings\. The cleanup policy works as intended\. All four primary modality inputs now have zero remaining feature\-null cells\. Bus and Subway remain compact complete\-case matrices with 40 and 21 model columns, respectively\. Taxi and FHVHV preserve the full 1,541,800\-row eligible zone universe and add missingness indicators for imputed continuous measures, producing 53 resolved model columns for each\. This confirms that the primary modality null problem is resolved before intermodal, multimodal, raw matrix construction, or scaling begins\.

### Section Summary

> The resolved primary modality inputs are now null\-clean\. After applying the row\-universe, fill, and imputation rules, Bus, Subway, Taxi, and FHVHV have zero remaining primary feature\-null cells\. These cleaned primary inputs are ready to be inherited by intermodal and multimodal coverage evaluation\.

## 3\.1\.1\.4 Resolve Intermodal and Multimodal Coverage

The primary modality inputs are now null\-clean, but the Multimodal matrix also includes cross\-modal features whose validity depends on more than one transportation system being available or meaningfully observed\. This section evaluates those intermodal features, separates structural non\-applicability from unresolved missingness, and defines the row and feature handling rules needed for the Multimodal matrix\. By the end of the section, intermodal and multimodal inputs should have the same status as the primary modality inputs: documented coverage rules, implemented cleanup logic, and QA evidence that downstream matrix construction can proceed without reopening missing\-value decisions\.

### Inventory Intermodal and Multimodal Features

Let's start by taking inventory\. This subsection identifies the interaction, ratio, and cross\-modal features that were moved out of the primary Bus, Subway, Taxi, and FHVHV inputs, then checks how those features relate to the selected Multimodal feature set\. This section's goal is a handoff list for 3\.1\.11\.5: which columns can inherit the resolved primary\-modality policy, and which columns need intermodal\-specific review before the Multimodal matrix can be finalized\.

First, list the cross\-modal features that were intentionally moved out of the primary single\-modality inputs\. These are the features that should be reviewed four the intermodal matrix\.

In [23]:
# ---------------------------------------------------------------------
# Inventory intermodal and cross-modal candidate features
# ---------------------------------------------------------------------

multimodal_selected_features = sorted(
    feature_set_definitions_df.loc[
        feature_set_definitions_df[FEATURE_SET_NAME_COLUMN] == "multimodal",
        FEATURE_SET_FEATURE_COLUMN,
    ].dropna().astype(str).str.strip().unique()
)

multimodal_catalog_df = (
    pd.DataFrame({"feature_name": multimodal_selected_features})
    .merge(
        feature_catalog_df,
        left_on="feature_name",
        right_on=CATALOG_FEATURE_COLUMN,
        how="left",
    )
)

feature_family_text = multimodal_catalog_df.get("feature_family", "").astype(str).str.lower()
source_metric_text = multimodal_catalog_df.get("source_metric", "").astype(str).str.lower()
feature_name_text = multimodal_catalog_df["feature_name"].astype(str).str.lower()

intermodal_candidate_mask = (
    feature_family_text.str.contains("interaction|ratio|multimodal", na=False)
    | source_metric_text.str.contains(r"\|", na=False)
    | feature_name_text.str.contains(
        "local_vs_connected|_x_|_to_.*_ratio|demand_shift_ratio",
        na=False,
    )
)

intermodal_feature_inventory_df = (
    multimodal_catalog_df
    .loc[intermodal_candidate_mask]
    .copy()
    .sort_values(["feature_family", "feature_name"])
    .reset_index(drop=True)
)

intermodal_feature_summary_df = (
    intermodal_feature_inventory_df
    .groupby(["feature_family", "modality"], dropna=False)
    .agg(intermodal_feature_count=("feature_name", "nunique"))
    .reset_index()
    .sort_values(["feature_family", "modality"])
    .reset_index(drop=True)
)

display(intermodal_feature_summary_df)
display(
    intermodal_feature_inventory_df[
        [
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "source_notebook",
        ]
    ]
)

print("Intermodal and cross-modal candidate feature inventory created.")

,feature_family,modality,intermodal_feature_count
0,multimodal_interaction,bus,2
1,multimodal_interaction,fhvhv,2
2,multimodal_interaction,multimodal,14
3,multimodal_interaction,subway,1
4,multimodal_interaction,taxi,2


,feature_name,feature_family,modality,source_metric,source_notebook
0,avg_bus_speed_local_vs_connected_zscore,multimodal_interaction,bus,avg_bus_speed,1.5.3
1,bus_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|fhvhv_trip_count,1.5.3
2,bus_to_taxi_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|taxi_trip_count,1.5.3
3,bus_trip_count_local_vs_connected_zscore,multimodal_interaction,bus,bus_trip_count,1.5.3
4,bus_x_subway_demand_zscore,multimodal_interaction,multimodal,bus_trip_count|subway_ridership,1.5.3
5,fhvhv_avg_trip_speed_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_avg_trip_speed,1.5.3
6,fhvhv_trip_count_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_trip_count,1.5.3
7,fhvhv_x_bus_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|bus_trip_count,1.5.3
8,fhvhv_x_bus_speed_zscore,multimodal_interaction,multimodal,fhvhv_avg_trip_speed|avg_bus_speed,1.5.3
9,fhvhv_x_subway_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|subway_ridership,1.5.3


Intermodal and cross-modal candidate feature inventory created.


Findings\. The intermodal inventory identifies 21 selected Multimodal features that come from the 1\.5\.3 multimodal\_interaction family\. Most of these are explicitly cross\-modal: 14 are labeled as multimodal features and combine inputs such as Taxi × Bus, FHVHV × Subway, or Bus\-to\-Taxi demand\-shift ratios\. The remaining 7 are mode\-specific local\-vs\-connected z\-scores: 2 Bus, 2 FHVHV, 2 Taxi, and 1 Subway\. This confirms that the intermodal review is focused and bounded\. We are not reopening the full 142\-feature Multimodal set; we are isolating the 21 features whose interpretation depends on cross\-system relationships or connected\-zone comparisons\.

Compare the selected Multimodal feature set against the resolved primary inputs and the intermodal inventory\. This creates a compact status label for each Multimodal feature so coverage checks can focus only on features that need special handling\.

In [24]:
# ---------------------------------------------------------------------
# Classify selected Multimodal features by handling status
# ---------------------------------------------------------------------

multimodal_selected_features = sorted(
    feature_set_definitions_df.loc[
        feature_set_definitions_df[FEATURE_SET_NAME_COLUMN] == "multimodal",
        FEATURE_SET_FEATURE_COLUMN,
    ].dropna().astype(str).str.strip().unique()
)

primary_resolved_feature_set = set(
    primary_missing_value_policy_df["feature_name"].dropna().astype(str)
)

intermodal_feature_set = set(
    intermodal_feature_inventory_df["feature_name"].dropna().astype(str)
)

multimodal_feature_inventory_df = pd.DataFrame(
    {"feature_name": multimodal_selected_features}
)

multimodal_feature_inventory_df["multimodal_handling_status"] = np.select(
    [
        multimodal_feature_inventory_df["feature_name"].isin(intermodal_feature_set),
        multimodal_feature_inventory_df["feature_name"].isin(primary_resolved_feature_set),
        multimodal_feature_inventory_df["feature_name"].str.contains("traffic", case=False, na=False),
    ],
    [
        "intermodal_review",
        "inherits_primary_policy",
        "traffic_context_review",
    ],
    default="multimodal_only_review",
)

multimodal_feature_inventory_df = multimodal_feature_inventory_df.merge(
    feature_catalog_df,
    left_on="feature_name",
    right_on=CATALOG_FEATURE_COLUMN,
    how="left",
)

multimodal_handling_summary_df = (
    multimodal_feature_inventory_df
    .groupby("multimodal_handling_status", dropna=False)
    .agg(feature_count=("feature_name", "nunique"))
    .reset_index()
    .sort_values("multimodal_handling_status")
    .reset_index(drop=True)
)

multimodal_features_for_review_df = (
    multimodal_feature_inventory_df
    .loc[
        multimodal_feature_inventory_df["multimodal_handling_status"]
        != "inherits_primary_policy"
    ]
    .copy()
    .sort_values(["multimodal_handling_status", "feature_name"])
    .reset_index(drop=True)
)

display(multimodal_handling_summary_df)
display(
    multimodal_features_for_review_df[
        [
            "multimodal_handling_status",
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "source_notebook",
        ]
    ]
)

print("Multimodal feature handling inventory created.")

,multimodal_handling_status,feature_count
0,inherits_primary_policy,115
1,intermodal_review,21
2,multimodal_only_review,6


,multimodal_handling_status,feature_name,feature_family,modality,source_metric,source_notebook
0,intermodal_review,avg_bus_speed_local_vs_connected_zscore,multimodal_interaction,bus,avg_bus_speed,1.5.3
1,intermodal_review,bus_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|fhvhv_trip_count,1.5.3
2,intermodal_review,bus_to_taxi_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|taxi_trip_count,1.5.3
3,intermodal_review,bus_trip_count_local_vs_connected_zscore,multimodal_interaction,bus,bus_trip_count,1.5.3
4,intermodal_review,bus_x_subway_demand_zscore,multimodal_interaction,multimodal,bus_trip_count|subway_ridership,1.5.3
5,intermodal_review,fhvhv_avg_trip_speed_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_avg_trip_speed,1.5.3
6,intermodal_review,fhvhv_trip_count_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_trip_count,1.5.3
7,intermodal_review,fhvhv_x_bus_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|bus_trip_count,1.5.3
8,intermodal_review,fhvhv_x_bus_speed_zscore,multimodal_interaction,multimodal,fhvhv_avg_trip_speed|avg_bus_speed,1.5.3
9,intermodal_review,fhvhv_x_subway_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|subway_ridership,1.5.3


Multimodal feature handling inventory created.


Findings\. The Multimodal feature inventory now separates cleanly into three groups\. Most selected Multimodal features, 115 of 142, inherit the resolved primary\-modality policy from 3\.1\.1\.3 and do not need a new missingness decision here\. Another 21 features require intermodal review; these are the interaction, demand\-shift, paired z\-score, and local\-vs\-connected features identified in the previous inventory\. The final 6 features need separate review because they are selected for Multimodal but were not retained in the resolved primary inputs: Taxi and FHVHV trip\-duration raw metrics, plus four FHVHV percent\-change or percent\-delta descendants\. The next coverage check should focus only on these 27 review features, not the full Multimodal feature set\.

### Profile Intermodal Coverage

This subsection measures how large the intermodal coverage problem actually is before deciding how to handle it\. The inventory step showed that only 21 selected Multimodal features are true interaction or connected\-context features\. Here we assign each feature to a dependency group and measure its null rate over the eligible modeling universe\. The goal is to see which individual intermodal features need explanation before the next subsection confirms whether their missingness comes from unavailable source signal, sparse service, or connected\-context gaps\.

First, map each intermodal feature to the systems it depends on\. This creates a compact dependency label so the coverage results can be summarized by modal combination instead of requiring feature\-name\-by\-feature\-name interpretation\.

In [25]:
# ---------------------------------------------------------------------
# Map intermodal features to modal dependency groups
# ---------------------------------------------------------------------

def infer_intermodal_dependencies(feature_name, source_metric, modality):
    feature_text = f"{feature_name} {source_metric} {modality}".lower()
    dependencies = []

    for mode in ["bus", "subway", "taxi", "fhvhv"]:
        if mode in feature_text:
            dependencies.append(mode)

    if "crossing" in feature_text or "mta" in feature_text:
        dependencies.append("mta_crossing_context")

    if "local_vs_connected" in feature_text:
        dependencies.append("connected_zone_context")

    return " + ".join(sorted(set(dependencies))) if dependencies else "review_dependency"


intermodal_dependency_inventory_df = intermodal_feature_inventory_df.copy()
intermodal_dependency_inventory_df["intermodal_dependency_group"] = (
    intermodal_dependency_inventory_df.apply(
        lambda row: infer_intermodal_dependencies(
            row["feature_name"],
            row.get("source_metric", ""),
            row.get("modality", ""),
        ),
        axis=1,
    )
)

intermodal_dependency_summary_df = (
    intermodal_dependency_inventory_df
    .groupby("intermodal_dependency_group", dropna=False)
    .agg(intermodal_feature_count=("feature_name", "nunique"))
    .reset_index()
    .sort_values(["intermodal_feature_count", "intermodal_dependency_group"], ascending=[False, True])
    .reset_index(drop=True)
)

display(intermodal_dependency_summary_df)
display(
    intermodal_dependency_inventory_df[
        [
            "intermodal_dependency_group",
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "source_notebook",
        ]
    ].sort_values(["intermodal_dependency_group", "feature_name"])
)

print("Intermodal dependency groups assigned.")

,intermodal_dependency_group,intermodal_feature_count
0,bus + fhvhv,3
1,bus + taxi,3
2,bus + connected_zone_context,2
3,connected_zone_context + fhvhv,2
4,connected_zone_context + taxi,2
5,fhvhv + subway,2
6,fhvhv + taxi,2
7,subway + taxi,2
8,bus + subway,1
9,connected_zone_context + subway,1


,intermodal_dependency_group,feature_name,feature_family,modality,source_metric,source_notebook
0,bus + connected_zone_context,avg_bus_speed_local_vs_connected_zscore,multimodal_interaction,bus,avg_bus_speed,1.5.3
3,bus + connected_zone_context,bus_trip_count_local_vs_connected_zscore,multimodal_interaction,bus,bus_trip_count,1.5.3
1,bus + fhvhv,bus_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|fhvhv_trip_count,1.5.3
7,bus + fhvhv,fhvhv_x_bus_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|bus_trip_count,1.5.3
8,bus + fhvhv,fhvhv_x_bus_speed_zscore,multimodal_interaction,multimodal,fhvhv_avg_trip_speed|avg_bus_speed,1.5.3
4,bus + subway,bus_x_subway_demand_zscore,multimodal_interaction,multimodal,bus_trip_count|subway_ridership,1.5.3
2,bus + taxi,bus_to_taxi_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|taxi_trip_count,1.5.3
16,bus + taxi,taxi_x_bus_demand_zscore,multimodal_interaction,multimodal,taxi_trip_count|bus_trip_count,1.5.3
17,bus + taxi,taxi_x_bus_speed_zscore,multimodal_interaction,multimodal,taxi_avg_trip_speed|avg_bus_speed,1.5.3
5,connected_zone_context + fhvhv,fhvhv_avg_trip_speed_local_vs_connected_zscore,multimodal_interaction,fhvhv,fhvhv_avg_trip_speed,1.5.3


Intermodal dependency groups assigned.


Findings\. The 21 intermodal features break into a small set of readable dependency groups\. The largest groups are Bus \+ FHVHV and Bus \+ Taxi, with 3 features each\. Most other pairings have 1 or 2 features, including Subway\-linked combinations, Taxi/FHVHV speed combinations, and connected\-zone context features\. This inventory matters because intermodal missingness should not be interpreted feature by feature alone\. A null in taxi\_x\_subway\_demand\_zscore, for example, is likely tied to the Subway dependency, while a null in taxi\_x\_bus\_speed\_zscore may be tied to sparse Taxi speed availability or Bus coverage\.

Now measure null coverage for each intermodal feature over the eligible modeling universe\. This does not decide whether to fill, drop, or exclude anything yet; it only quantifies the size of the remaining intermodal coverage problem\.

In [26]:
# ---------------------------------------------------------------------
# Measure intermodal feature missingness over the eligible universe
# ---------------------------------------------------------------------

intermodal_feature_columns = sorted(
    intermodal_dependency_inventory_df["feature_name"].dropna().unique()
)

intermodal_null_expressions = [
    f"""
    SUM(CASE WHEN {duckdb_identifier(feature)} IS NULL THEN 1 ELSE 0 END)
        AS {duckdb_identifier(feature)}
    """
    for feature in intermodal_feature_columns
]

with duckdb.connect() as con:
    intermodal_null_counts_row = con.execute(
        f"""
        SELECT
            COUNT(*) AS eligible_rows,
            {", ".join(intermodal_null_expressions)}
        FROM read_parquet('{panel_path_sql}')
        WHERE {primary_zone_exclusion_sql}
        """
    ).fetchdf().iloc[0]

eligible_intermodal_rows = int(intermodal_null_counts_row["eligible_rows"])

intermodal_feature_missingness_df = pd.DataFrame(
    [
        {
            "feature_name": feature,
            "eligible_rows": eligible_intermodal_rows,
            "missing_rows": int(intermodal_null_counts_row[feature]),
            "missing_pct": round(
                int(intermodal_null_counts_row[feature]) / eligible_intermodal_rows,
                5,
            ),
        }
        for feature in intermodal_feature_columns
    ]
)

intermodal_feature_missingness_df = (
    intermodal_feature_missingness_df
    .merge(
        intermodal_dependency_inventory_df[
            [
                "feature_name",
                "intermodal_dependency_group",
                "feature_family",
                "modality",
                "source_metric",
            ]
        ],
        on="feature_name",
        how="left",
    )
    .sort_values(["missing_pct", "feature_name"], ascending=[False, True])
    .reset_index(drop=True)
)

display(intermodal_feature_missingness_df)

print("Intermodal feature missingness measured.")

,feature_name,eligible_rows,missing_rows,missing_pct,intermodal_dependency_group,feature_family,modality,source_metric
0,subway_ridership_local_vs_connected_zscore,1541800,670090,0.43462,connected_zone_context + subway,multimodal_interaction,subway,subway_ridership
1,bus_x_subway_demand_zscore,1541800,629124,0.40805,bus + subway,multimodal_interaction,multimodal,bus_trip_count|subway_ridership
2,fhvhv_x_subway_demand_zscore,1541800,616720,0.40000,fhvhv + subway,multimodal_interaction,multimodal,fhvhv_trip_count|subway_ridership
3,subway_to_fhvhv_demand_shift_ratio,1541800,616720,0.40000,fhvhv + subway,multimodal_interaction,multimodal,subway_ridership|fhvhv_trip_count
4,subway_to_taxi_demand_shift_ratio,1541800,616720,0.40000,subway + taxi,multimodal_interaction,multimodal,subway_ridership|taxi_trip_count
5,taxi_x_subway_demand_zscore,1541800,616720,0.40000,subway + taxi,multimodal_interaction,multimodal,taxi_trip_count|subway_ridership
6,taxi_avg_trip_speed_local_vs_connected_zscore,1541800,475283,0.30827,connected_zone_context + taxi,multimodal_interaction,taxi,taxi_avg_trip_speed
7,taxi_x_bus_speed_zscore,1541800,460269,0.29853,bus + taxi,multimodal_interaction,multimodal,taxi_avg_trip_speed|avg_bus_speed
8,taxi_x_fhvhv_speed_zscore,1541800,428245,0.27776,fhvhv + taxi,multimodal_interaction,multimodal,taxi_avg_trip_speed|fhvhv_avg_trip_speed
9,avg_bus_speed_local_vs_connected_zscore,1541800,77182,0.05006,bus + connected_zone_context,multimodal_interaction,bus,avg_bus_speed


Intermodal feature missingness measured.


Findings\. Intermodal missingness is concentrated in a few dependency patterns\. Subway\-linked features have the largest gaps: subway\_ridership\_local\_vs\_connected\_zscore is missing for 670,090 rows, or 43\.462%, and most Subway pairings are missing for about 616,720 rows, or 40%\. This is consistent with Subway’s structural service universe rather than a new data\-quality failure\. Taxi speed\-linked intermodal features are the next major group: taxi\_avg\_trip\_speed\_local\_vs\_connected\_zscore is missing for 30\.827%, taxi\_x\_bus\_speed\_zscore for 29\.853%, and taxi\_x\_fhvhv\_speed\_zscore for 27\.776%\. These match the sparse Taxi speed issue already diagnosed in the primary modality review\. Bus\-linked gaps are smaller, around 4\.237% to 5\.006%, while FHVHV connected\-zone features, the MTA crossing × Taxi feature, Taxi trip\-count local\-vs\-connected, and Taxi × FHVHV demand have no missing values\.

### Confirm Intermodal Applicability Patterns

The coverage profile already shows the major intermodal missingness patterns\. Subway\-linked features are missing where Subway is structurally unavailable, while Taxi speed\-linked features inherit the sparse\-service speed problem resolved in the primary modality inputs\. This short checkpoint confirms that interpretation at the dependency\-group level before defining intermodal handling rules\. We only want to make sure the Multimodal cleanup policy is tied to the same coverage logic already established for the primary modalities\.

Check each intermodal feature against its source metrics\. This identifies whether missing intermodal values occur because one or more source metrics are null, because sources are present but zero, or because sources are present and nonzero but the engineered feature is still missing\.

In [27]:
# ---------------------------------------------------------------------
# Attribute intermodal missingness to source-signal status
# ---------------------------------------------------------------------

intermodal_source_signal_records = []

with duckdb.connect() as con:
    for _, feature_row in intermodal_dependency_inventory_df.iterrows():
        feature_name = feature_row["feature_name"]
        source_metrics = [
            metric.strip()
            for metric in str(feature_row["source_metric"]).split("|")
            if metric.strip() and metric.strip() in panel_column_set
        ]

        if source_metrics:
            source_any_null_condition = " OR ".join(
                f"{duckdb_identifier(metric)} IS NULL"
                for metric in source_metrics
            )
            source_all_present_condition = " AND ".join(
                f"{duckdb_identifier(metric)} IS NOT NULL"
                for metric in source_metrics
            )
            source_any_zero_condition = " OR ".join(
                f"COALESCE({duckdb_identifier(metric)}, 0) = 0"
                for metric in source_metrics
            )
            source_all_present_but_any_zero_condition = (
                f"({source_all_present_condition}) AND ({source_any_zero_condition})"
            )
        else:
            source_any_null_condition = "FALSE"
            source_all_present_condition = "TRUE"
            source_all_present_but_any_zero_condition = "FALSE"

        signal_query = f"""
        SELECT
            COUNT(*) AS eligible_rows,
            SUM(CASE WHEN {duckdb_identifier(feature_name)} IS NULL THEN 1 ELSE 0 END)
                AS feature_missing_rows,
            SUM(CASE WHEN {duckdb_identifier(feature_name)} IS NULL
                     AND ({source_any_null_condition})
                     THEN 1 ELSE 0 END)
                AS missing_with_any_source_null_rows,
            SUM(CASE WHEN {duckdb_identifier(feature_name)} IS NULL
                     AND ({source_all_present_but_any_zero_condition})
                     THEN 1 ELSE 0 END)
                AS missing_with_sources_present_but_zero_rows,
            SUM(CASE WHEN {duckdb_identifier(feature_name)} IS NULL
                     AND ({source_all_present_condition})
                     AND NOT ({source_all_present_but_any_zero_condition})
                     THEN 1 ELSE 0 END)
                AS missing_with_sources_present_nonzero_rows
        FROM read_parquet('{panel_path_sql}')
        WHERE {primary_zone_exclusion_sql}
        """

        result = con.execute(signal_query).fetchdf().iloc[0]
        feature_missing_rows = int(result["feature_missing_rows"])

        intermodal_source_signal_records.append(
            {
                "feature_name": feature_name,
                "intermodal_dependency_group": feature_row["intermodal_dependency_group"],
                "source_metric": feature_row["source_metric"],
                "eligible_rows": int(result["eligible_rows"]),
                "feature_missing_rows": feature_missing_rows,
                "missing_with_any_source_null_rows": int(result["missing_with_any_source_null_rows"]),
                "missing_with_sources_present_but_zero_rows": int(
                    result["missing_with_sources_present_but_zero_rows"]
                ),
                "missing_with_sources_present_nonzero_rows": int(
                    result["missing_with_sources_present_nonzero_rows"]
                ),
                "missing_with_any_source_null_share": round(
                    int(result["missing_with_any_source_null_rows"])
                    / feature_missing_rows,
                    5,
                ) if feature_missing_rows else 0,
                "missing_with_sources_present_but_zero_share": round(
                    int(result["missing_with_sources_present_but_zero_rows"])
                    / feature_missing_rows,
                    5,
                ) if feature_missing_rows else 0,
                "missing_with_sources_present_nonzero_share": round(
                    int(result["missing_with_sources_present_nonzero_rows"])
                    / feature_missing_rows,
                    5,
                ) if feature_missing_rows else 0,
            }
        )

intermodal_source_signal_df = (
    pd.DataFrame(intermodal_source_signal_records)
    .sort_values(["feature_missing_rows", "feature_name"], ascending=[False, True])
    .reset_index(drop=True)
)


intermodal_source_signal_display_df = (
    intermodal_source_signal_df
    [
        [
            "feature_name",
            "intermodal_dependency_group",
            "feature_missing_rows",
            "missing_with_any_source_null_share",
            "missing_with_sources_present_but_zero_share",
            "missing_with_sources_present_nonzero_share",
        ]
    ]
    .rename(
        columns={
            "feature_name": "feature",
            "intermodal_dependency_group": "dependency_group",
            "feature_missing_rows": "missing_rows",
            "missing_with_any_source_null_share": "source_null_share",
            "missing_with_sources_present_but_zero_share": "source_zero_share",
            "missing_with_sources_present_nonzero_share": "source_present_nonzero_share",
        }
    )
    .sort_values("missing_rows", ascending=False)
    .reset_index(drop=True)
)

display(intermodal_source_signal_display_df)

print("Intermodal source-signal attribution created.")

,feature,dependency_group,missing_rows,source_null_share,source_zero_share,source_present_nonzero_share
0,subway_ridership_local_vs_connected_zscore,connected_zone_context + subway,670090,0.92035,0.00015,0.07950
1,bus_x_subway_demand_zscore,bus + subway,629124,1.00000,0.00000,0.00000
2,fhvhv_x_subway_demand_zscore,fhvhv + subway,616720,1.00000,0.00000,0.00000
3,subway_to_fhvhv_demand_shift_ratio,fhvhv + subway,616720,1.00000,0.00000,0.00000
4,subway_to_taxi_demand_shift_ratio,subway + taxi,616720,1.00000,0.00000,0.00000
5,taxi_x_subway_demand_zscore,subway + taxi,616720,1.00000,0.00000,0.00000
6,taxi_avg_trip_speed_local_vs_connected_zscore,connected_zone_context + taxi,475283,0.90103,0.00725,0.09172
7,taxi_x_bus_speed_zscore,bus + taxi,460269,1.00000,0.00000,0.00000
8,taxi_x_fhvhv_speed_zscore,fhvhv + taxi,428245,1.00000,0.00000,0.00000
9,avg_bus_speed_local_vs_connected_zscore,bus + connected_zone_context,77182,0.84634,0.00000,0.15366


Intermodal source-signal attribution created.


Findings\. Intermodal nulls mostly inherit missing source signal rather than introducing a new matrix\-quality problem\. Subway\-paired demand features are the clearest case: the standard Bus/Subway, FHVHV/Subway, Subway/FHVHV, Subway/Taxi, and Taxi/Subway interactions are missing only when at least one source metric is null\. Taxi speed\-paired interactions behave the same way, matching the sparse Taxi speed issue handled earlier\. The main exceptions are local\-vs\-connected features, where some rows have present, nonzero source values but still lack connected\-zone comparison context\. Several features are already complete, including FHVHV local\-vs\-connected features, MTA crossing context × Taxi, Taxi trip\-count local\-vs\-connected, and Taxi × FHVHV demand\.

Summarize the same source\-signal attribution by dependency group\. This compact view shows which modal combinations are mostly source\-null problems and which contain a meaningful connected\-context gap\.

In [28]:
# ---------------------------------------------------------------------
# Summarize intermodal source-signal attribution by dependency group
# ---------------------------------------------------------------------

intermodal_source_signal_summary_df = (
    intermodal_source_signal_df
    .groupby("intermodal_dependency_group", dropna=False)
    .agg(
        intermodal_feature_count=("feature_name", "nunique"),
        total_feature_missing_rows=("feature_missing_rows", "sum"),
        total_missing_with_any_source_null_rows=("missing_with_any_source_null_rows", "sum"),
        total_missing_with_sources_present_but_zero_rows=("missing_with_sources_present_but_zero_rows", "sum"),
        total_missing_with_sources_present_nonzero_rows=("missing_with_sources_present_nonzero_rows", "sum"),
    )
    .reset_index()
)

intermodal_source_signal_summary_df["source_null_share"] = (
    intermodal_source_signal_summary_df["total_missing_with_any_source_null_rows"]
    / intermodal_source_signal_summary_df["total_feature_missing_rows"].replace(0, np.nan)
).round(5)

intermodal_source_signal_summary_df["sources_present_but_zero_share"] = (
    intermodal_source_signal_summary_df["total_missing_with_sources_present_but_zero_rows"]
    / intermodal_source_signal_summary_df["total_feature_missing_rows"].replace(0, np.nan)
).round(5)

intermodal_source_signal_summary_df["sources_present_nonzero_share"] = (
    intermodal_source_signal_summary_df["total_missing_with_sources_present_nonzero_rows"]
    / intermodal_source_signal_summary_df["total_feature_missing_rows"].replace(0, np.nan)
).round(5)

intermodal_source_signal_summary_df = (
    intermodal_source_signal_summary_df
    .sort_values("total_feature_missing_rows", ascending=False)
    .reset_index(drop=True)
)

display(intermodal_source_signal_summary_df)

print("Intermodal source-signal attribution summarized by dependency group.")

,intermodal_dependency_group,intermodal_feature_count,total_feature_missing_rows,total_missing_with_any_source_null_rows,total_missing_with_sources_present_but_zero_rows,total_missing_with_sources_present_nonzero_rows,source_null_share,sources_present_but_zero_share,sources_present_nonzero_share
0,subway + taxi,2,1233440,1233440,0,0,1.00000,0.00000,0.00000
1,fhvhv + subway,2,1233440,1233440,0,0,1.00000,0.00000,0.00000
2,connected_zone_context + subway,1,670090,616720,98,53272,0.92035,0.00015,0.07950
3,bus + subway,1,629124,629124,0,0,1.00000,0.00000,0.00000
4,bus + taxi,3,590913,590913,0,0,1.00000,0.00000,0.00000
5,connected_zone_context + taxi,2,475283,428245,3447,43591,0.90103,0.00725,0.09172
6,fhvhv + taxi,2,428245,428245,0,0,1.00000,0.00000,0.00000
7,bus + fhvhv,3,195966,195966,0,0,1.00000,0.00000,0.00000
8,bus + connected_zone_context,2,154364,130644,0,23720,0.84634,0.00000,0.15366
9,connected_zone_context + fhvhv,2,0,0,0,0,NaN,NaN,NaN


Intermodal source-signal attribution summarized by dependency group.


Findings\. The dependency\-level summary makes the policy problem clearer\. Most paired modal groups are pure source\-null cases: Subway \+ Taxi, FHVHV \+ Subway, Bus \+ Subway, Bus \+ Taxi, FHVHV \+ Taxi, and Bus \+ FHVHV all have a 100% source\-null share among missing intermodal rows\. That means their missingness is inherited from unavailable or undefined modal inputs rather than unexplained intermodal failure\. The connected\-zone groups need more care\. Connected\-zone \+ Subway is mostly source\-null at 92\.035%, but 7\.950% of its missing rows have source values present and nonzero\. Connected\-zone \+ Taxi has the same pattern, with 90\.103% source\-null and 9\.172% source\-present nonzero missingness\. Bus connected\-zone features show the largest context\-gap share: 15\.366% of missing rows have source values present and nonzero\. This suggests that paired modal features can inherit source\-signal handling rules, while local\-vs\-connected features may need a connected\-context\-specific rule or missingness indicator\.

The source\-signal attribution shows that most intermodal nulls are inherited from missing modal inputs, but local\-vs\-connected features have a smaller share of missing values even when their source metric is present\. This block checks whether those gaps are concentrated in specific zones or missing connected\-zone context fields\.

In [29]:
# ---------------------------------------------------------------------
# Inspect local-vs-connected gaps with usable source signal
# ---------------------------------------------------------------------

local_vs_connected_signal_gap_records = []

local_vs_connected_features_to_check = (
    intermodal_source_signal_df
    .loc[
        intermodal_source_signal_df["feature_name"].str.contains(
            "local_vs_connected",
            case=False,
            na=False,
        )
        & (intermodal_source_signal_df["missing_with_sources_present_nonzero_rows"] > 0)
    ]
    .copy()
)

with duckdb.connect() as con:
    for _, feature_row in local_vs_connected_features_to_check.iterrows():
        feature_name = feature_row["feature_name"]
        source_metric = str(feature_row["source_metric"]).strip()

        if source_metric not in panel_column_set:
            continue

        zone_query = f"""
        SELECT
            '{feature_name}' AS feature_name,
            '{source_metric}' AS source_metric,
            taxi_zone_id,
            zone,
            borough,
            COUNT(*) AS eligible_rows,
            SUM(
                CASE WHEN {duckdb_identifier(feature_name)} IS NULL
                      AND {duckdb_identifier(source_metric)} IS NOT NULL
                      AND {duckdb_identifier(source_metric)} <> 0
                     THEN 1 ELSE 0 END
            ) AS source_present_nonzero_gap_rows
        FROM read_parquet('{panel_path_sql}')
        WHERE {primary_zone_exclusion_sql}
        GROUP BY 1, 2, 3, 4, 5
        HAVING source_present_nonzero_gap_rows > 0
        """

        local_vs_connected_signal_gap_records.append(con.execute(zone_query).fetchdf())

local_vs_connected_signal_gap_zone_df = pd.concat(
    local_vs_connected_signal_gap_records,
    ignore_index=True,
) if local_vs_connected_signal_gap_records else pd.DataFrame()

local_vs_connected_signal_gap_zone_df["gap_row_pct"] = (
    local_vs_connected_signal_gap_zone_df["source_present_nonzero_gap_rows"]
    / local_vs_connected_signal_gap_zone_df["eligible_rows"].replace(0, np.nan)
).round(5)

local_vs_connected_gap_borough_df = (
    local_vs_connected_signal_gap_zone_df
    .groupby("borough", as_index=False)
    .agg(
        zones_with_gap=("taxi_zone_id", "nunique"),
        total_gap_rows=("source_present_nonzero_gap_rows", "sum"),
        median_gap_row_pct=("gap_row_pct", "median"),
    )
    .sort_values("total_gap_rows", ascending=False)
)

local_vs_connected_gap_zone_display_df = (
    local_vs_connected_signal_gap_zone_df
    [
        [
            "feature_name",
            "taxi_zone_id",
            "zone",
            "borough",
            "source_present_nonzero_gap_rows",
            "gap_row_pct",
        ]
    ]
    .rename(
        columns={
            "feature_name": "feature",
            "source_present_nonzero_gap_rows": "gap_rows",
        }
    )
    .sort_values(["gap_row_pct", "gap_rows"], ascending=False)
    .head(20)
    .reset_index(drop=True)
)

display(local_vs_connected_gap_borough_df)
display(local_vs_connected_gap_zone_display_df)

print("Local-vs-connected source-present gaps inspected by feature and zone.")


,borough,zones_with_gap,total_gap_rows,median_gap_row_pct
3,Queens,46,40137.0,0.028415
0,Bronx,38,22108.0,0.024790
1,Brooklyn,49,22077.0,0.012650
2,Manhattan,7,20649.0,0.146710
4,Staten Island,20,15612.0,0.023775


,feature,taxi_zone_id,zone,borough,gap_rows,gap_row_pct
0,avg_bus_speed_local_vs_connected_zscore,156,Mariners Harbor,Staten Island,5930.0,1.00000
1,avg_bus_speed_local_vs_connected_zscore,13,Battery Park City,Manhattan,5930.0,1.00000
2,bus_trip_count_local_vs_connected_zscore,13,Battery Park City,Manhattan,5930.0,1.00000
3,bus_trip_count_local_vs_connected_zscore,156,Mariners Harbor,Staten Island,5930.0,1.00000
4,subway_ridership_local_vs_connected_zscore,78,East Tremont,Bronx,5922.0,0.99865
5,subway_ridership_local_vs_connected_zscore,7,Astoria,Queens,5922.0,0.99865
6,subway_ridership_local_vs_connected_zscore,39,Canarsie,Brooklyn,5922.0,0.99865
7,subway_ridership_local_vs_connected_zscore,260,Woodside,Queens,5922.0,0.99865
8,subway_ridership_local_vs_connected_zscore,36,Bushwick North,Brooklyn,5922.0,0.99865
9,subway_ridership_local_vs_connected_zscore,92,Flushing,Queens,5921.0,0.99848


Local-vs-connected source-present gaps inspected by feature and zone.


Findings\. The local\-vs\-connected gaps are concentrated by geography and feature type rather than random\. Bus local\-vs\-connected gaps are narrow: Battery Park City and Mariners Harbor are the only full\-gap zones shown among the largest cases\. Subway local\-vs\-connected gaps are broader, with high\-gap zones across Queens, Brooklyn, the Bronx, and Manhattan even when Subway ridership is present, which points to connected\-zone comparison limits rather than absent Subway signal\. Taxi speed local\-vs\-connected gaps are also geographically patterned, with notable concentrations in airports, outer\-borough, waterfront, and edge zones\. These patterns support treating local\-vs\-connected features as connected\-context features: preserve them with neutral imputation and missingness indicators instead of zero\-filling them or dropping large parts of the Multimodal universe\.

> ✅ Decision\. Local\-vs\-connected intermodal features will be retained in the default Multimodal matrix using neutral imputation plus missingness indicators\. Missing z\-scores are filled with 0, which represents a neutral connected\-zone deviation, while companion indicator columns preserve the fact that the connected comparison was unavailable\. This avoids dropping rows or discarding selected spatial\-comparison features while keeping the imputation interpretable\.

### Define Resolved Multimodal Handling Policy

The intermodal coverage review identifies three practical cases: clean interaction features, source\-null interaction features that inherit primary modality coverage behavior, and local\-vs\-connected features whose comparison context is sometimes unavailable\. This subsection converts those findings into a single Multimodal handling policy\. The policy preserves valid observations while keeping unavailable comparisons explicit through neutral imputation and indicators where needed\. With that approach, the Multimodal row universe can inherit the eligible primary modeling universe rather than using strict complete\-case filtering\.

Assign a handling rule to each Multimodal review feature\. Primary features already inherit the resolved primary\-modality policy; this block focuses on the 21 intermodal\-review features and the 6 Multimodal\-only review features\.

In [30]:
# ---------------------------------------------------------------------
# Define handling policy for Multimodal review features
# ---------------------------------------------------------------------

def assign_multimodal_review_action(row):
    feature_name = str(row["feature_name"]).lower()
    feature_family = str(row.get("feature_family", "")).lower()
    source_metric = str(row.get("source_metric", "")).lower()

    if "local_vs_connected" in feature_name:
        return "neutral_impute_with_missing_indicator"

    if "multimodal_interaction" in feature_family:
        return "inherit_source_modality_policy"

    if "avg_trip_duration" in feature_name:
        return "neutral_impute_with_missing_indicator"

    if "pct_change" in feature_name or "pct_delta" in feature_name:
        return "neutral_impute_with_missing_indicator"

    if "|" in source_metric:
        return "inherit_source_modality_policy"

    return "review"


multimodal_review_policy_df = multimodal_features_for_review_df.copy()
multimodal_review_policy_df["multimodal_null_action"] = (
    multimodal_review_policy_df.apply(assign_multimodal_review_action, axis=1)
)
multimodal_review_policy_df["adds_missing_indicator"] = (
    multimodal_review_policy_df["multimodal_null_action"]
    == "neutral_impute_with_missing_indicator"
)

unresolved_multimodal_review_features_df = multimodal_review_policy_df.loc[
    multimodal_review_policy_df["multimodal_null_action"] == "review"
].copy()

display(
    multimodal_review_policy_df[
        [
            "multimodal_handling_status",
            "feature_name",
            "feature_family",
            "modality",
            "source_metric",
            "multimodal_null_action",
            "adds_missing_indicator",
        ]
    ].sort_values(["multimodal_null_action", "feature_name"])
)

assert unresolved_multimodal_review_features_df.empty, (
    "Some Multimodal review features still need a handling rule:\\n"
    + unresolved_multimodal_review_features_df[
        ["feature_name", "feature_family", "modality", "source_metric"]
    ].to_string(index=False)
)

print("Multimodal review-feature handling policy defined.")

,multimodal_handling_status,feature_name,feature_family,modality,source_metric,multimodal_null_action,adds_missing_indicator
1,intermodal_review,bus_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|fhvhv_trip_count,inherit_source_modality_policy,False
2,intermodal_review,bus_to_taxi_demand_shift_ratio,multimodal_interaction,multimodal,bus_trip_count|taxi_trip_count,inherit_source_modality_policy,False
4,intermodal_review,bus_x_subway_demand_zscore,multimodal_interaction,multimodal,bus_trip_count|subway_ridership,inherit_source_modality_policy,False
7,intermodal_review,fhvhv_x_bus_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|bus_trip_count,inherit_source_modality_policy,False
8,intermodal_review,fhvhv_x_bus_speed_zscore,multimodal_interaction,multimodal,fhvhv_avg_trip_speed|avg_bus_speed,inherit_source_modality_policy,False
9,intermodal_review,fhvhv_x_subway_demand_zscore,multimodal_interaction,multimodal,fhvhv_trip_count|subway_ridership,inherit_source_modality_policy,False
10,intermodal_review,mta_crossing_share_x_taxi_trip_count_zscore,multimodal_interaction,multimodal,manhattan_connected_crossing_share|taxi_trip_c...,inherit_source_modality_policy,False
12,intermodal_review,subway_to_fhvhv_demand_shift_ratio,multimodal_interaction,multimodal,subway_ridership|fhvhv_trip_count,inherit_source_modality_policy,False
13,intermodal_review,subway_to_taxi_demand_shift_ratio,multimodal_interaction,multimodal,subway_ridership|taxi_trip_count,inherit_source_modality_policy,False
16,intermodal_review,taxi_x_bus_demand_zscore,multimodal_interaction,multimodal,taxi_trip_count|bus_trip_count,inherit_source_modality_policy,False


Multimodal review-feature handling policy defined.


Findings\. The Multimodal review policy assigns every review feature to a documented handling rule\. Fourteen paired interaction features inherit the source\-modality policy because their missingness is driven by unavailable or undefined modal inputs, such as Subway availability, Bus coverage, or Taxi speed observation\. Seven intermodal local\-vs\-connected features use neutral imputation with missingness indicators because their nulls reflect unavailable connected\-zone comparisons rather than absent base signal alone\. The six Multimodal\-only review features also use neutral imputation with indicators: these include Taxi/FHVHV trip\-duration measures and FHVHV percent\-change or percent\-delta descendants\. No review features remain unresolved\.

Summarize the policy counts by handling rule\. This compact table shows how many features are inherited, imputed, flagged, or handled through source\-modality behavior before the cleanup rules are applied\.

In [31]:
# ---------------------------------------------------------------------
# Summarize resolved Multimodal handling policy
# ---------------------------------------------------------------------

multimodal_primary_policy_summary_df = pd.DataFrame(
    [
        {
            "multimodal_handling_status": "inherits_primary_policy",
            "multimodal_null_action": "inherit_resolved_primary_policy",
            "feature_count": int(
                multimodal_feature_inventory_df.loc[
                    multimodal_feature_inventory_df["multimodal_handling_status"]
                    == "inherits_primary_policy",
                    "feature_name",
                ].nunique()
            ),
            "adds_missing_indicator": "per primary policy",
        }
    ]
)

multimodal_review_policy_summary_df = (
    multimodal_review_policy_df
    .groupby(
        [
            "multimodal_handling_status",
            "multimodal_null_action",
            "adds_missing_indicator",
        ],
        dropna=False,
    )
    .agg(feature_count=("feature_name", "nunique"))
    .reset_index()
)

multimodal_review_policy_summary_df["adds_missing_indicator"] = (
    multimodal_review_policy_summary_df["adds_missing_indicator"].astype(str)
)

resolved_multimodal_policy_summary_df = pd.concat(
    [
        multimodal_primary_policy_summary_df,
        multimodal_review_policy_summary_df,
    ],
    ignore_index=True,
)

resolved_multimodal_policy_summary_df = (
    resolved_multimodal_policy_summary_df
    .sort_values(["multimodal_handling_status", "multimodal_null_action"])
    .reset_index(drop=True)
)

display(resolved_multimodal_policy_summary_df)

print("Resolved Multimodal handling policy summarized.")

,multimodal_handling_status,multimodal_null_action,feature_count,adds_missing_indicator
0,inherits_primary_policy,inherit_resolved_primary_policy,115,per primary policy
1,intermodal_review,inherit_source_modality_policy,14,False
2,intermodal_review,neutral_impute_with_missing_indicator,7,True
3,multimodal_only_review,neutral_impute_with_missing_indicator,6,True


Resolved Multimodal handling policy summarized.


Findings\. The resolved Multimodal policy keeps the feature set manageable and explicit\. Most Multimodal features, 115, inherit the primary policy already validated in 3\.1\.1\.3\. The remaining 27 review features split into three groups: 14 paired intermodal features inherit source\-modality behavior, 7 intermodal local\-vs\-connected features are neutral\-imputed and flagged, and 6 Multimodal\-only features are also neutral\-imputed and flagged\. This means 13 review features add missingness indicators, while the paired interaction features do not need separate indicators because their missingness is handled through source\-modality logic\.

> Candidate Universe Note\. Strict complete\-case filtering is not evaluated as the default Multimodal strategy because it would primarily remove structurally valid Subway\-unavailable zones and sparse Taxi\-speed observations\. Those patterns are part of the transportation system being modeled, not generic data\-quality failures\. The Multimodal policy therefore keeps the eligible primary universe and resolves feature\-level missingness through documented handling rules\.

### Validate Resolved Multimodal Setup

This subsection turns the resolved Multimodal feature policy into preparation objects that can be applied during matrix construction\. The row universe has already been decided, so the work here focuses only on feature\-level cleanup: one resolved feature\-policy table, neutral imputation values where needed, and a compact summary showing how many features will be filled or flagged\.

First, combine the resolved primary\-policy features with the intermodal and Multimodal\-only review features\. This creates one policy table for every selected Multimodal feature\.

In [32]:
# ---------------------------------------------------------------------
# Build resolved Multimodal feature policy
# ---------------------------------------------------------------------

multimodal_policy_base_df = (
    multimodal_feature_inventory_df[
        [
            "feature_name",
            "multimodal_handling_status",
            "feature_family",
            "modality",
            "source_metric",
            "source_notebook",
        ]
    ]
    .drop_duplicates("feature_name")
    .copy()
)

multimodal_review_action_lookup = (
    multimodal_review_policy_df[
        ["feature_name", "multimodal_null_action", "adds_missing_indicator"]
    ]
    .drop_duplicates("feature_name")
    .set_index("feature_name")
)

NEUTRAL_IMPUTE_TOKENS = [
    "zscore",
    "residual",
    "seasonal",
    "fourier",
    "pct_change",
    "pct_delta",
    "abs_change",
    "abs_delta",
    "rolling_std",
    "local_vs_connected",
]
DEMAND_SIGNAL_TOKENS = [
    "trip_count",
    "ridership",
    "transfers",
    "distinct_dropoff_zone_count",
]
CONTINUOUS_SIGNAL_TOKENS = [
    "speed",
    "duration",
    "distance",
    "travel_time",
]


def assign_multimodal_null_action(row):
    feature_name = row["feature_name"]
    feature_text = str(feature_name).lower()
    source_metric_text = str(row.get("source_metric", "")).lower()

    if feature_name in multimodal_review_action_lookup.index:
        return multimodal_review_action_lookup.loc[feature_name, "multimodal_null_action"]

    if any(token in feature_text for token in NEUTRAL_IMPUTE_TOKENS):
        return "neutral_impute_with_missing_indicator"
    if any(token in feature_text for token in CONTINUOUS_SIGNAL_TOKENS):
        return "neutral_impute_with_missing_indicator"
    if any(token in feature_text for token in DEMAND_SIGNAL_TOKENS):
        return "zero_fill_sparse_service_absence"
    if any(token in source_metric_text for token in CONTINUOUS_SIGNAL_TOKENS):
        return "neutral_impute_with_missing_indicator"
    if any(token in source_metric_text for token in DEMAND_SIGNAL_TOKENS):
        return "zero_fill_sparse_service_absence"

    return "neutral_impute_with_missing_indicator"


resolved_multimodal_feature_policy_df = multimodal_policy_base_df.copy()
resolved_multimodal_feature_policy_df["resolved_null_action"] = (
    resolved_multimodal_feature_policy_df.apply(assign_multimodal_null_action, axis=1)
)
resolved_multimodal_feature_policy_df["adds_multimodal_missing_indicator"] = (
    resolved_multimodal_feature_policy_df["resolved_null_action"]
    == "neutral_impute_with_missing_indicator"
)
resolved_multimodal_feature_policy_df["multimodal_null_action"] = (
    resolved_multimodal_feature_policy_df["resolved_null_action"]
)

resolved_multimodal_feature_policy_df = (
    resolved_multimodal_feature_policy_df
    .sort_values(["multimodal_handling_status", "feature_name"])
    .reset_index(drop=True)
)

assert len(resolved_multimodal_feature_policy_df) == len(multimodal_selected_features), (
    "Resolved Multimodal policy should have exactly one row per selected Multimodal feature."
)
assert resolved_multimodal_feature_policy_df["feature_name"].nunique() == len(multimodal_selected_features), (
    "Resolved Multimodal policy contains duplicate feature names."
)
assert not resolved_multimodal_feature_policy_df["resolved_null_action"].eq("complete_case_required").any(), (
    "Multimodal feature policy should use feature-level handling, not complete-case row filtering."
)

display(
    resolved_multimodal_feature_policy_df[
        [
            "feature_name",
            "multimodal_handling_status",
            "feature_family",
            "modality",
            "source_metric",
            "resolved_null_action",
            "adds_multimodal_missing_indicator",
        ]
    ]
)

print("Resolved Multimodal feature policy built.")

,feature_name,multimodal_handling_status,feature_family,modality,source_metric,resolved_null_action,adds_multimodal_missing_indicator
0,avg_bus_speed,inherits_primary_policy,raw_metric,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
1,avg_bus_speed_abs_change_1_same_bucket,inherits_primary_policy,temporal_history,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
2,avg_bus_speed_cp_abs_delta_same_bucket,inherits_primary_policy,temporal_history,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
3,avg_bus_speed_cp_pct_delta_same_bucket,inherits_primary_policy,temporal_history,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
4,avg_bus_speed_deviation_abs_zscore,inherits_primary_policy,anomaly_severity,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
5,avg_bus_speed_fourier20_residual_reconstructed,inherits_primary_policy,fourier20_residual_reconstruction,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
6,avg_bus_speed_lag_1_same_bucket,inherits_primary_policy,temporal_history,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
7,avg_bus_speed_pct_change_1_same_bucket,inherits_primary_policy,temporal_history,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
8,avg_bus_speed_post_cp_mean_same_bucket,inherits_primary_policy,temporal_history,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True
9,avg_bus_speed_pre_cp_mean_same_bucket,inherits_primary_policy,temporal_history,bus,avg_bus_speed,neutral_impute_with_missing_indicator,True


Resolved Multimodal feature policy built.


Findings\. The resolved Multimodal feature policy now covers the selected feature set cleanly: 142 selected features produce 142 policy rows, with no duplicated spatial/context fields and no complete\-case row\-filter action carried into the Multimodal policy\. The policy is intentionally feature\-level because the Multimodal matrix keeps the broader eligible universe\. Count\-like demand and activity features use zero\-fill for sparse\-service or structural absence, while continuous, standardized, temporal, decomposition, spatial\-context, and local\-vs\-connected features use neutral imputation with missingness indicators\. The intermodal\-review features also split as expected: paired interaction features inherit source\-modality behavior, while local\-vs\-connected features are neutral\-imputed and flagged\.

Compute neutral imputation values for Multimodal features that need feature\-level imputation\. These values are used during matrix construction, while missingness indicators preserve where imputation occurred\.

In [33]:
# ---------------------------------------------------------------------
# Compute Multimodal imputation values
# ---------------------------------------------------------------------

MULTIMODAL_NEUTRAL_ZERO_PATTERNS = [
    "zscore",
    "pct_change",
    "pct_delta",
    "abs_delta",
    "demand_shift_ratio",
    "_ratio",
]

multimodal_imputation_features_df = resolved_multimodal_feature_policy_df.loc[
    resolved_multimodal_feature_policy_df["resolved_null_action"].isin(
        [
            "neutral_impute_with_missing_indicator",
            "inherit_source_modality_policy",
            "zero_fill_sparse_service_absence",
        ]
    ),
    ["feature_name", "resolved_null_action", "adds_multimodal_missing_indicator"],
].drop_duplicates()

multimodal_imputation_records = []

with duckdb.connect() as con:
    for _, feature_row in multimodal_imputation_features_df.iterrows():
        feature_name = feature_row["feature_name"]
        resolved_null_action = feature_row["resolved_null_action"]
        feature_text = feature_name.lower()

        if resolved_null_action == "zero_fill_sparse_service_absence":
            imputation_value = 0.0
            imputation_strategy = "zero_fill"
        elif resolved_null_action == "inherit_source_modality_policy":
            imputation_value = 0.0
            imputation_strategy = "source_policy_neutral_zero"
        elif any(pattern in feature_text for pattern in MULTIMODAL_NEUTRAL_ZERO_PATTERNS):
            imputation_value = 0.0
            imputation_strategy = "neutral_zero"
        else:
            imputation_value = con.execute(
                f"""
                SELECT MEDIAN({duckdb_identifier(feature_name)})
                FROM read_parquet('{panel_path_sql}')
                WHERE {primary_zone_exclusion_sql}
                  AND {duckdb_identifier(feature_name)} IS NOT NULL
                """
            ).fetchone()[0]
            imputation_strategy = "median_observed_value"

        multimodal_imputation_records.append(
            {
                "feature_name": feature_name,
                "resolved_null_action": resolved_null_action,
                "adds_multimodal_missing_indicator": bool(
                    feature_row["adds_multimodal_missing_indicator"]
                ),
                "multimodal_imputation_value": imputation_value,
                "multimodal_imputation_strategy": imputation_strategy,
            }
        )

resolved_multimodal_imputation_values_df = pd.DataFrame(multimodal_imputation_records)

missing_multimodal_imputation_values = resolved_multimodal_imputation_values_df.loc[
    resolved_multimodal_imputation_values_df["multimodal_imputation_value"].isna()
]
assert missing_multimodal_imputation_values.empty, (
    "Some Multimodal imputation features are missing imputation values:\n"
    + missing_multimodal_imputation_values.to_string(index=False)
)

display(
    resolved_multimodal_imputation_values_df.sort_values(
        ["resolved_null_action", "feature_name"]
    ).reset_index(drop=True)
)

print("Multimodal imputation values computed.")

,feature_name,resolved_null_action,adds_multimodal_missing_indicator,multimodal_imputation_value,multimodal_imputation_strategy
0,bus_to_fhvhv_demand_shift_ratio,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
1,bus_to_taxi_demand_shift_ratio,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
2,bus_x_subway_demand_zscore,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
3,fhvhv_x_bus_demand_zscore,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
4,fhvhv_x_bus_speed_zscore,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
5,fhvhv_x_subway_demand_zscore,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
6,mta_crossing_share_x_taxi_trip_count_zscore,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
7,subway_to_fhvhv_demand_shift_ratio,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
8,subway_to_taxi_demand_shift_ratio,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero
9,taxi_x_bus_demand_zscore,inherit_source_modality_policy,False,0.000000e+00,source_policy_neutral_zero


Multimodal imputation values computed.


Findings\. Imputation values are available for every selected Multimodal feature\. The table uses three simple strategies: zero\-fill for count\-like activity features, neutral zero for standardized ratios, z\-scores, percent changes, deltas, and interaction scores, and observed medians for continuous or level\-based features such as speeds, durations, distances, rolling statistics, and spatial context fields\. Focus on the strategy column rather than inspecting every value\. Values like 0 for z\-scores and ratios represent a neutral deviation, while median values preserve ordinary scale for non\-standardized continuous features\.

Summarize the feature\-level cleanup rules that will be applied during Multimodal matrix construction\. This keeps the handoff compact: how many selected features use zero\-fill, how many use neutral imputation, and how many indicator columns will be added\.

In [34]:
# ---------------------------------------------------------------------
# Summarize resolved Multimodal feature cleanup
# ---------------------------------------------------------------------

resolved_multimodal_cleanup_summary_df = (
    resolved_multimodal_feature_policy_df
    .groupby(
        [
            "multimodal_handling_status",
            "resolved_null_action",
            "adds_multimodal_missing_indicator",
        ],
        dropna=False,
    )
    .agg(feature_count=("feature_name", "nunique"))
    .reset_index()
    .sort_values(["multimodal_handling_status", "resolved_null_action"])
    .reset_index(drop=True)
)

resolved_multimodal_cleanup_summary_df["indicator_columns_to_add"] = np.where(
    resolved_multimodal_cleanup_summary_df["adds_multimodal_missing_indicator"],
    resolved_multimodal_cleanup_summary_df["feature_count"],
    0,
)

display(resolved_multimodal_cleanup_summary_df)

print("Resolved Multimodal feature cleanup summarized.")

,multimodal_handling_status,resolved_null_action,adds_multimodal_missing_indicator,feature_count,indicator_columns_to_add
0,inherits_primary_policy,neutral_impute_with_missing_indicator,True,99,99
1,inherits_primary_policy,zero_fill_sparse_service_absence,False,16,0
2,intermodal_review,inherit_source_modality_policy,False,14,0
3,intermodal_review,neutral_impute_with_missing_indicator,True,7,7
4,multimodal_only_review,neutral_impute_with_missing_indicator,True,6,6


Resolved Multimodal feature cleanup summarized.


Findings\. The cleanup summary shows that the Multimodal matrix will be row\-preserving but indicator\-heavy\. Of the 142 selected features, 16 inherited primary\-policy features use zero\-fill without indicators, 14 paired intermodal features inherit source\-modality handling, and 112 features use neutral imputation with missingness indicators\. The large indicator count is expected because the Multimodal universe is broader than any single modality’s complete\-case universe\. Instead of dropping zones without Subway applicability or sparse Taxi speed observations, the matrix keeps those rows and exposes unavailable comparisons or imputed continuous values through indicator columns\.

Summarize the resolved Multimodal preparation assets\. This final table is the handoff into raw matrix construction\.

In [35]:
# ---------------------------------------------------------------------
# Summarize resolved Multimodal inputs
# ---------------------------------------------------------------------

resolved_multimodal_input_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "selected_multimodal_features",
            "value": len(multimodal_selected_features),
        },
        {
            "qa_check": "resolved_policy_features",
            "value": resolved_multimodal_feature_policy_df["feature_name"].nunique(),
        },
        {
            "qa_check": "features_with_imputation_values",
            "value": resolved_multimodal_imputation_values_df["feature_name"].nunique(),
        },
        {
            "qa_check": "indicator_columns_to_add",
            "value": int(
                resolved_multimodal_cleanup_summary_df["indicator_columns_to_add"].sum()
            ),
        },
    ]
)

resolved_multimodal_input_summary_df["status"] = np.where(
    resolved_multimodal_input_summary_df["qa_check"].eq("resolved_policy_features")
    & (resolved_multimodal_input_summary_df["value"] != len(multimodal_selected_features)),
    "review",
    "pass",
)

display(resolved_multimodal_input_summary_df)

assert resolved_multimodal_feature_policy_df["feature_name"].nunique() == len(multimodal_selected_features), (
    "Resolved Multimodal feature policy does not cover every selected feature."
)

print("Resolved Multimodal inputs summarized.")

,qa_check,value,status
0,selected_multimodal_features,142,pass
1,resolved_policy_features,142,pass
2,features_with_imputation_values,142,pass
3,indicator_columns_to_add,112,pass


Resolved Multimodal inputs summarized.


Findings\. The final input summary passes all setup checks\. The selected Multimodal feature count and resolved policy count both equal 142, confirming that every selected feature has exactly one handling rule\. Imputation values were created for all 142 features, and the policy will add 112 missingness\-indicator columns during matrix construction\. This gives the next validation step a concrete target: apply these feature\-level rules over the eligible Multimodal universe and confirm that the resolved Multimodal input has zero remaining feature\-null cells\.

### Validate Resolved Multimodal Null Status

This subsection validates the resolved Multimodal inputs before matrix construction\. The policy has already assigned feature\-level handling rules, so the checks here are practical: confirm that applying those rules removes all feature nulls, add indicators only where missing values actually occur, quantify how large that indicator layer is, and end with a compact QA summary\. The goal is to prove that the Multimodal matrix can be constructed without complete\-case filtering while preserving the coverage information needed for interpretation\.

Apply the resolved Multimodal imputation rules and confirm that no feature nulls remain\. This is the direct null\-cleanliness check: every selected Multimodal feature should be checked, and the resolved null count should be zero\.

In [36]:
# ---------------------------------------------------------------------
# Validate resolved Multimodal feature null status
# ---------------------------------------------------------------------

multimodal_imputation_lookup = {
    row["feature_name"]: row["multimodal_imputation_value"]
    for _, row in resolved_multimodal_imputation_values_df.iterrows()
}

multimodal_null_validation_records = []

with duckdb.connect() as con:
    for _, policy_row in resolved_multimodal_feature_policy_df.iterrows():
        feature_name = policy_row["feature_name"]
        imputation_value = multimodal_imputation_lookup[feature_name]

        validation_query = f"""
        SELECT
            COUNT(*) AS eligible_rows,
            SUM(CASE WHEN {duckdb_identifier(feature_name)} IS NULL THEN 1 ELSE 0 END)
                AS original_null_rows,
            SUM(
                CASE WHEN COALESCE(
                    {duckdb_identifier(feature_name)},
                    {float(imputation_value):.17g}
                ) IS NULL THEN 1 ELSE 0 END
            ) AS resolved_null_rows
        FROM read_parquet('{panel_path_sql}')
        WHERE {primary_zone_exclusion_sql}
        """

        result = con.execute(validation_query).fetchdf().iloc[0]

        multimodal_null_validation_records.append(
            {
                "feature_name": feature_name,
                "resolved_null_action": policy_row["resolved_null_action"],
                "eligible_rows": int(result["eligible_rows"]),
                "original_null_rows": int(result["original_null_rows"]),
                "resolved_null_rows": int(result["resolved_null_rows"]),
            }
        )

resolved_multimodal_null_validation_df = pd.DataFrame(
    multimodal_null_validation_records
)

resolved_multimodal_null_validation_df["original_null_pct"] = (
    resolved_multimodal_null_validation_df["original_null_rows"]
    / resolved_multimodal_null_validation_df["eligible_rows"].replace(0, np.nan)
).round(5)

resolved_multimodal_null_validation_df["resolved_null_pct"] = (
    resolved_multimodal_null_validation_df["resolved_null_rows"]
    / resolved_multimodal_null_validation_df["eligible_rows"].replace(0, np.nan)
).round(5)

remaining_multimodal_nulls_df = resolved_multimodal_null_validation_df.loc[
    resolved_multimodal_null_validation_df["resolved_null_rows"] > 0
].copy()

if remaining_multimodal_nulls_df.empty:
    display(
        pd.DataFrame(
            [
                {
                    "qa_check": "resolved_multimodal_feature_nulls",
                    "features_checked": resolved_multimodal_null_validation_df["feature_name"].nunique(),
                    "remaining_null_feature_cells": 0,
                    "status": "pass",
                }
            ]
        )
    )
else:
    display(
        remaining_multimodal_nulls_df.sort_values(
            ["resolved_null_rows", "feature_name"],
            ascending=[False, True],
        )
    )

assert remaining_multimodal_nulls_df.empty, (
    "Resolved Multimodal features still contain null values."
)

print("Resolved Multimodal feature null status validated.")

,qa_check,features_checked,remaining_null_feature_cells,status
0,resolved_multimodal_feature_nulls,142,0,pass


Resolved Multimodal feature null status validated.


Findings\. The resolved Multimodal null check passes\. All 142 selected Multimodal features were checked, and the resolved feature layer has 0 remaining null feature cells after applying the documented imputation rules\. This is the key validation result: the Multimodal inputs can preserve the eligible row universe without relying on strict complete\-case filtering\.

Restrict missingness indicators to features that actually contain observed nulls in the Multimodal row universe\. This prevents the policy from adding indicators to features that were eligible for imputation in principle but complete in practice\.

In [37]:
# ---------------------------------------------------------------------
# Restrict Multimodal indicators to features with observed nulls
# ---------------------------------------------------------------------

multimodal_observed_null_records = []

with duckdb.connect() as con:
    for _, policy_row in resolved_multimodal_feature_policy_df.iterrows():
        feature_name = policy_row["feature_name"]

        null_query = f"""
        SELECT
            COUNT(*) AS eligible_rows,
            SUM(CASE WHEN {duckdb_identifier(feature_name)} IS NULL THEN 1 ELSE 0 END)
                AS original_null_rows
        FROM read_parquet('{panel_path_sql}')
        WHERE {primary_zone_exclusion_sql}
        """

        result = con.execute(null_query).fetchdf().iloc[0]

        multimodal_observed_null_records.append(
            {
                "feature_name": feature_name,
                "eligible_rows": int(result["eligible_rows"]),
                "original_null_rows": int(result["original_null_rows"]),
            }
        )

multimodal_observed_nulls_df = pd.DataFrame(multimodal_observed_null_records)

resolved_multimodal_feature_policy_df = resolved_multimodal_feature_policy_df.merge(
    multimodal_observed_nulls_df,
    on="feature_name",
    how="left",
)

resolved_multimodal_feature_policy_df["adds_multimodal_missing_indicator"] = (
    resolved_multimodal_feature_policy_df["resolved_null_action"]
    == "neutral_impute_with_missing_indicator"
) & (
    resolved_multimodal_feature_policy_df["original_null_rows"] > 0
)

resolved_multimodal_feature_policy_df["original_null_pct"] = (
    resolved_multimodal_feature_policy_df["original_null_rows"]
    / resolved_multimodal_feature_policy_df["eligible_rows"].replace(0, np.nan)
).round(5)

multimodal_indicator_count_df = pd.DataFrame(
    [
        {
            "selected_multimodal_features": resolved_multimodal_feature_policy_df["feature_name"].nunique(),
            "neutral_impute_features": int(
                (
                    resolved_multimodal_feature_policy_df["resolved_null_action"]
                    == "neutral_impute_with_missing_indicator"
                ).sum()
            ),
            "features_with_observed_nulls": int(
                (resolved_multimodal_feature_policy_df["original_null_rows"] > 0).sum()
            ),
            "indicator_columns_to_add": int(
                resolved_multimodal_feature_policy_df["adds_multimodal_missing_indicator"].sum()
            ),
        }
    ]
)

display(multimodal_indicator_count_df)
display(
    resolved_multimodal_feature_policy_df.loc[
        resolved_multimodal_feature_policy_df["adds_multimodal_missing_indicator"],
        [
            "feature_name",
            "multimodal_handling_status",
            "resolved_null_action",
            "original_null_rows",
            "original_null_pct",
        ],
    ].sort_values(["original_null_pct", "feature_name"], ascending=[False, True])
)

print("Multimodal indicators restricted to features with observed nulls.")

,selected_multimodal_features,neutral_impute_features,features_with_observed_nulls,indicator_columns_to_add
0,142,112,112,91


,feature_name,multimodal_handling_status,resolved_null_action,original_null_rows,original_null_pct
126,subway_ridership_local_vs_connected_zscore,intermodal_review,neutral_impute_with_missing_indicator,670090,0.43462
79,subway_ridership_pct_change_1_same_bucket,inherits_primary_policy,neutral_impute_with_missing_indicator,630345,0.40884
77,subway_ridership_deviation_abs_zscore,inherits_primary_policy,neutral_impute_with_missing_indicator,623485,0.40439
74,subway_ridership_abs_change_1_same_bucket,inherits_primary_policy,neutral_impute_with_missing_indicator,618280,0.40101
83,subway_ridership_rolling_std_7_same_bucket,inherits_primary_policy,neutral_impute_with_missing_indicator,618280,0.40101
75,subway_ridership_cp_abs_delta_same_bucket,inherits_primary_policy,neutral_impute_with_missing_indicator,616720,0.40000
76,subway_ridership_cp_pct_delta_same_bucket,inherits_primary_policy,neutral_impute_with_missing_indicator,616720,0.40000
78,subway_ridership_fourier20_residual_reconstructed,inherits_primary_policy,neutral_impute_with_missing_indicator,616720,0.40000
80,subway_ridership_residual,inherits_primary_policy,neutral_impute_with_missing_indicator,616720,0.40000
81,subway_ridership_residual_abs,inherits_primary_policy,neutral_impute_with_missing_indicator,616720,0.40000


Multimodal indicators restricted to features with observed nulls.


Findings\. The indicator logic is based on observed missingness, not policy category alone\. Although 112 features use neutral imputation rules, only 91 have observed nulls and therefore receive indicator columns\. The highest indicator rates belong to the already\-diagnosed Subway and Taxi speed coverage patterns; smaller Bus and FHVHV indicators remain mostly low, with many under 5% and some near zero\. This reduces unnecessary indicator columns while preserving the coverage signals that matter for interpretation\.

Summarize how large the indicator layer is and which feature groups drive it\. This does not evaluate PCA or clustering yet, but it flags whether the prepared matrix contains a substantial missingness\-indicator component\.

In [38]:
# ---------------------------------------------------------------------
# Summarize Multimodal missingness-indicator burden
# ---------------------------------------------------------------------

indicator_feature_policy_df = resolved_multimodal_feature_policy_df.loc[
    resolved_multimodal_feature_policy_df["adds_multimodal_missing_indicator"]
].copy()

indicator_burden_records = []

with duckdb.connect() as con:
    for _, policy_row in indicator_feature_policy_df.iterrows():
        feature_name = policy_row["feature_name"]

        indicator_query = f"""
        SELECT
            COUNT(*) AS eligible_rows,
            SUM(CASE WHEN {duckdb_identifier(feature_name)} IS NULL THEN 1 ELSE 0 END)
                AS indicator_active_rows
        FROM read_parquet('{panel_path_sql}')
        WHERE {primary_zone_exclusion_sql}
        """

        result = con.execute(indicator_query).fetchdf().iloc[0]

        indicator_burden_records.append(
            {
                "feature_name": feature_name,
                "multimodal_handling_status": policy_row["multimodal_handling_status"],
                "resolved_null_action": policy_row["resolved_null_action"],
                "eligible_rows": int(result["eligible_rows"]),
                "indicator_active_rows": int(result["indicator_active_rows"]),
            }
        )

multimodal_indicator_burden_df = pd.DataFrame(indicator_burden_records)
multimodal_indicator_burden_df["indicator_active_pct"] = (
    multimodal_indicator_burden_df["indicator_active_rows"]
    / multimodal_indicator_burden_df["eligible_rows"].replace(0, np.nan)
).round(5)

multimodal_indicator_burden_summary_df = (
    multimodal_indicator_burden_df
    .groupby("multimodal_handling_status", dropna=False)
    .agg(
        indicator_feature_count=("feature_name", "nunique"),
        max_indicator_active_pct=("indicator_active_pct", "max"),
        median_indicator_active_pct=("indicator_active_pct", "median"),
        total_indicator_active_cells=("indicator_active_rows", "sum"),
    )
    .reset_index()
)

multimodal_indicator_burden_summary_df["max_indicator_active_pct"] = (
    multimodal_indicator_burden_summary_df["max_indicator_active_pct"] * 100
).round(3)
multimodal_indicator_burden_summary_df["median_indicator_active_pct"] = (
    multimodal_indicator_burden_summary_df["median_indicator_active_pct"] * 100
).round(3)

multimodal_indicator_burden_summary_display_df = (
    multimodal_indicator_burden_summary_df
    [
        [
            "multimodal_handling_status",
            "indicator_feature_count",
            "max_indicator_active_pct",
            "median_indicator_active_pct",
            "total_indicator_active_cells",
        ]
    ]
    .rename(
        columns={
            "multimodal_handling_status": "handling_status",
            "indicator_feature_count": "indicator_features",
            "max_indicator_active_pct": "max_active_pct",
            "median_indicator_active_pct": "median_active_pct",
            "total_indicator_active_cells": "active_cells",
        }
    )
    .sort_values("active_cells", ascending=False)
    .reset_index(drop=True)
)

multimodal_indicator_burden_top_display_df = (
    multimodal_indicator_burden_df
    .sort_values(["indicator_active_pct", "feature_name"], ascending=[False, True])
    .head(25)
    [
        [
            "feature_name",
            "multimodal_handling_status",
            "indicator_active_rows",
            "indicator_active_pct",
        ]
    ]
    .rename(
        columns={
            "feature_name": "feature",
            "multimodal_handling_status": "handling_status",
            "indicator_active_rows": "active_rows",
            "indicator_active_pct": "active_pct",
        }
    )
    .reset_index(drop=True)
)

display(multimodal_indicator_burden_summary_display_df)
display(multimodal_indicator_burden_top_display_df)

print("Multimodal missingness-indicator burden summarized.")

,handling_status,indicator_features,max_active_pct,median_active_pct,active_cells
0,inherits_primary_policy,82,40.884,4.238,13430423
1,intermodal_review,4,43.462,17.916,1299737
2,multimodal_only_review,5,27.776,1.741,484317


,feature,handling_status,active_rows,active_pct
0,subway_ridership_local_vs_connected_zscore,intermodal_review,670090,0.43462
1,subway_ridership_pct_change_1_same_bucket,inherits_primary_policy,630345,0.40884
2,subway_ridership_deviation_abs_zscore,inherits_primary_policy,623485,0.40439
3,subway_ridership_abs_change_1_same_bucket,inherits_primary_policy,618280,0.40101
4,subway_ridership_rolling_std_7_same_bucket,inherits_primary_policy,618280,0.40101
5,subway_ridership_cp_abs_delta_same_bucket,inherits_primary_policy,616720,0.40000
6,subway_ridership_cp_pct_delta_same_bucket,inherits_primary_policy,616720,0.40000
7,subway_ridership_fourier20_residual_reconstructed,inherits_primary_policy,616720,0.40000
8,subway_ridership_residual,inherits_primary_policy,616720,0.40000
9,subway_ridership_residual_abs,inherits_primary_policy,616720,0.40000


Multimodal missingness-indicator burden summarized.


Findings\. The Multimodal matrix is null\-clean, but the missingness indicators carry meaningful structure\. Most indicator columns come from features that inherit the primary policy: 82 indicators, with a typical active rate around 4% but several Subway indicators active for about 40% of rows\. Intermodal review adds only 4 indicators, but one of them, \`subway\_ridership\_local\_vs\_connected\_zscore\`, is active for about 43% of rows\. Multimodal\-only features add 5 indicators, led by Taxi/FHVHV sparse\-service measures around 28%\. These indicators are no longer data\-quality warnings; they encode coverage and applicability patterns that should be checked in PCA and UMAP sensitivity analyses\.

The final validation summary brings the resolved Multimodal input into one compact view: retained observations, selected mobility features, added indicators, and remaining null cells\. At this point, the important question is whether the policy produced a complete modeling input while preserving the full eligible Multimodal universe\.

In [39]:
# ---------------------------------------------------------------------
# Summarize resolved Multimodal validation status
# ---------------------------------------------------------------------

resolved_multimodal_validation_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "resolved_multimodal_rows",
            "value": int(resolved_multimodal_null_validation_df["eligible_rows"].max()),
            "status": "pass",
        },
        {
            "qa_check": "selected_multimodal_features",
            "value": resolved_multimodal_feature_policy_df["feature_name"].nunique(),
            "status": "pass",
        },
        {
            "qa_check": "indicator_columns_to_add",
            "value": indicator_feature_policy_df["feature_name"].nunique(),
            "status": "review" if indicator_feature_policy_df["feature_name"].nunique() > 100 else "pass",
        },
        {
            "qa_check": "remaining_null_feature_cells",
            "value": int(resolved_multimodal_null_validation_df["resolved_null_rows"].sum()),
            "status": "pass"
            if int(resolved_multimodal_null_validation_df["resolved_null_rows"].sum()) == 0
            else "review",
        },
    ]
)

display(resolved_multimodal_validation_summary_df)

assert (
    resolved_multimodal_validation_summary_df.loc[
        resolved_multimodal_validation_summary_df["qa_check"]
        == "remaining_null_feature_cells",
        "value",
    ].iloc[0]
    == 0
), "Resolved Multimodal validation found remaining null feature cells."

print("Resolved Multimodal validation summarized.")

,qa_check,value,status
0,resolved_multimodal_rows,1541800,pass
1,selected_multimodal_features,142,pass
2,indicator_columns_to_add,91,pass
3,remaining_null_feature_cells,0,pass


Resolved Multimodal validation summarized.


Findings\. The final Multimodal validation summary passes\. The resolved input keeps 1,541,800 rows, covers all 142 selected Multimodal features, adds 91 missingness indicators, and has 0 remaining null feature cells\. That completes the Multimodal cleanup step: the matrix construction section can now apply these resolved feature rules directly rather than reopening coverage or missingness decisions\.

### Section Summary

> The intermodal and Multimodal coverage review resolves the remaining feature\-level cleanup rules needed before raw matrix construction\. Most selected Multimodal features inherit the primary\-modality cleanup logic, while paired intermodal features inherit source\-modality behavior and local\-vs\-connected features use neutral imputation with indicators\. The final validation keeps 1,541,800 rows, covers all 142 selected Multimodal features, adds 91 observed\-null indicators, and leaves 0 remaining feature\-null cells\. With Bus, Subway, Taxi, FHVHV, and Multimodal inputs now resolved, the notebook can move from coverage policy to raw matrix materialization\.

## 3\.1\.1\.5 Materialize Raw Modeling Matrices

This section materializes the raw modeling matrices using the resolved inputs from the previous sections\. Bus, Subway, Taxi, and FHVHV use the primary modality row filters and cleanup rules finalized in 3\.1\.1\.3; Multimodal uses the feature\-level cleanup policy finalized in 3\.1\.1\.4\. Each matrix is written as a raw feature table with a matching row\-metadata table so modeling inputs stay separate from interpretation fields\. The goal is simple: create the actual raw matrix assets, validate the real outputs, and confirm that every exported matrix is null\-clean at the feature level\.

Configure the output paths and small helpers used to write raw matrix and metadata files\. Each matrix gets a modeling\_row\_id so the feature table and metadata table can be joined without putting metadata into the modeling feature space\.

In [40]:
# ---------------------------------------------------------------------
# Configure raw matrix output paths and SQL helpers
# ---------------------------------------------------------------------

RAW_MATRIX_OUTPUT_PATHS = {
    feature_set: OUTPUT_DIR / f"{feature_set}_raw_modeling_matrix.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

RAW_METADATA_OUTPUT_PATHS = {
    feature_set: OUTPUT_DIR / f"{feature_set}_row_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

RAW_MATRIX_FEATURE_COLUMNS_BY_SET = {}
RAW_MATRIX_INDICATOR_COLUMNS_BY_SET = {}

RAW_MATRIX_ORDER_COLUMNS = ["taxi_zone_id", "date", "temporal_bucket"]
POLICY_GEOGRAPHY_METADATA_COLUMN = "policy_geography_class"
POLICY_GEOGRAPHY_SOURCE_COLUMNS = [
    "is_cbd_zone",
    "is_cbd_gateway_zone",
    "is_cbd_adjacent_zone",
]

def sql_path(path):
    return str(path).replace("\\", "/").replace("'", "''")

def modeling_row_id_expression():
    order_sql = ", ".join(duckdb_identifier(column) for column in RAW_MATRIX_ORDER_COLUMNS)
    return f"ROW_NUMBER() OVER (ORDER BY {order_sql}) - 1 AS modeling_row_id"

def indicator_name(feature_name):
    return f"{feature_name}_missing_indicator"

def policy_geography_class_expression():
    return f"""
    CASE
        WHEN COALESCE({duckdb_identifier('is_cbd_zone')}, 0) = 1
            THEN 'cbd'
        WHEN COALESCE({duckdb_identifier('is_cbd_gateway_zone')}, 0) = 1
          OR COALESCE({duckdb_identifier('is_cbd_adjacent_zone')}, 0) = 1
            THEN 'gateway_or_adjacent'
        ELSE 'outside'
    END AS {duckdb_identifier(POLICY_GEOGRAPHY_METADATA_COLUMN)}
    """.strip()

def metadata_select_expressions():
    missing_policy_columns = [
        column for column in POLICY_GEOGRAPHY_SOURCE_COLUMNS
        if column not in panel_column_set
    ]
    assert not missing_policy_columns, (
        "Policy geography source columns are missing from the feature panel: "
        f"{missing_policy_columns}"
    )

    metadata_columns = [
        column for column in available_metadata_columns
        if column != POLICY_GEOGRAPHY_METADATA_COLUMN
    ]

    return [
        f"{duckdb_identifier(column)} AS {duckdb_identifier(column)}"
        for column in metadata_columns
    ] + [policy_geography_class_expression()]

print("Raw matrix output paths and metadata helpers configured.")

Raw matrix output paths and metadata helpers configured.


Materialize the four primary modality raw matrices\. This applies the row filters, zero\-fill rules, neutral imputation values, and missingness indicators already validated in 3\.1\.1\.3\.

In [41]:
# ---------------------------------------------------------------------
# Materialize primary raw modeling matrices and metadata
# ---------------------------------------------------------------------

primary_imputation_lookup = {
    (row["feature_set"], row["feature_name"]): row["neutral_imputation_value"]
    for _, row in primary_continuous_imputation_values_df.iterrows()
}

metadata_select_sql = ",\n                    ".join(metadata_select_expressions())

with duckdb.connect() as con:
    for feature_set_name in UNIMODAL_FEATURE_SET_NAMES:
        feature_policy_df = primary_missing_value_policy_df.loc[
            primary_missing_value_policy_df["feature_set"] == feature_set_name
        ].copy()

        feature_select_expressions = []
        feature_columns = []
        indicator_columns = []

        for _, feature_row in feature_policy_df.iterrows():
            feature_name = feature_row["feature_name"]
            null_action = feature_row["primary_null_action"]

            feature_columns.append(feature_name)

            if null_action == "zero_fill_sparse_service_absence":
                feature_select_expressions.append(
                    f"COALESCE({duckdb_identifier(feature_name)}, 0) AS {duckdb_identifier(feature_name)}"
                )
            elif null_action == "neutral_impute_with_missing_indicator":
                imputation_value = primary_imputation_lookup[(feature_set_name, feature_name)]
                feature_select_expressions.append(
                    f"COALESCE({duckdb_identifier(feature_name)}, {float(imputation_value):.17g}) "
                    f"AS {duckdb_identifier(feature_name)}"
                )

                if bool(feature_row["adds_missing_indicator"]):
                    missing_indicator_name = indicator_name(feature_name)
                    indicator_columns.append(missing_indicator_name)
                    feature_select_expressions.append(
                        f"CASE WHEN {duckdb_identifier(feature_name)} IS NULL THEN 1 ELSE 0 END "
                        f"AS {duckdb_identifier(missing_indicator_name)}"
                    )
            else:
                feature_select_expressions.append(
                    f"{duckdb_identifier(feature_name)} AS {duckdb_identifier(feature_name)}"
                )

        RAW_MATRIX_FEATURE_COLUMNS_BY_SET[feature_set_name] = feature_columns
        RAW_MATRIX_INDICATOR_COLUMNS_BY_SET[feature_set_name] = indicator_columns

        row_filter_sql = primary_resolved_row_filter_sql_by_set[feature_set_name]
        matrix_path = sql_path(RAW_MATRIX_OUTPUT_PATHS[feature_set_name])
        metadata_path = sql_path(RAW_METADATA_OUTPUT_PATHS[feature_set_name])

        con.execute(
            f"""
            COPY (
                SELECT
                    {modeling_row_id_expression()},
                    {", ".join(feature_select_expressions)}
                FROM read_parquet('{panel_path_sql}')
                WHERE {row_filter_sql}
            )
            TO '{matrix_path}'
            (FORMAT PARQUET)
            """
        )

        con.execute(
            f"""
            COPY (
                SELECT
                    {modeling_row_id_expression()},
                    {metadata_select_sql}
                FROM read_parquet('{panel_path_sql}')
                WHERE {row_filter_sql}
            )
            TO '{metadata_path}'
            (FORMAT PARQUET)
            """
        )

primary_raw_matrix_export_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set,
            "matrix_path": str(RAW_MATRIX_OUTPUT_PATHS[feature_set]),
            "metadata_path": str(RAW_METADATA_OUTPUT_PATHS[feature_set]),
            "feature_count": len(RAW_MATRIX_FEATURE_COLUMNS_BY_SET[feature_set]),
            "indicator_count": len(RAW_MATRIX_INDICATOR_COLUMNS_BY_SET[feature_set]),
        }
        for feature_set in UNIMODAL_FEATURE_SET_NAMES
    ]
)

display(primary_raw_matrix_export_df)

print("Primary raw modeling matrices and metadata materialized.")

,feature_set,matrix_path,metadata_path,feature_count,indicator_count
0,bus,pipeline_data/3.1.1.final_tables/bus_raw_model...,pipeline_data/3.1.1.final_tables/bus_row_metad...,40,0
1,subway,pipeline_data/3.1.1.final_tables/subway_raw_mo...,pipeline_data/3.1.1.final_tables/subway_row_me...,21,0
2,taxi,pipeline_data/3.1.1.final_tables/taxi_raw_mode...,pipeline_data/3.1.1.final_tables/taxi_row_meta...,38,15
3,fhvhv,pipeline_data/3.1.1.final_tables/fhvhv_raw_mod...,pipeline_data/3.1.1.final_tables/fhvhv_row_met...,37,16


Primary raw modeling matrices and metadata materialized.


Findings\. The four primary raw matrices were written successfully with separate row\-metadata files\. Bus exports 40 modeling columns over 1,472,947 rows, Subway exports 21 columns over 911,455 rows, Taxi exports 53 columns over 1,541,800 rows, and FHVHV exports 53 columns over 1,541,800 rows\. The Taxi and FHVHV column counts include the missingness indicators created by the sparse\-service policy, so the matrix width reflects both selected mobility features and the flags needed to preserve sparse\-service context\.

Materialize the Multimodal raw matrix using the resolved Multimodal feature policy\. Indicators are created only for features with observed nulls\.

In [44]:
# ---------------------------------------------------------------------
# Materialize Multimodal raw modeling matrix and metadata
# ---------------------------------------------------------------------

multimodal_imputation_lookup = {
    row["feature_name"]: row["multimodal_imputation_value"]
    for _, row in resolved_multimodal_imputation_values_df.iterrows()
}

multimodal_feature_select_expressions = []
multimodal_feature_columns = []
multimodal_indicator_columns = []

for _, feature_row in resolved_multimodal_feature_policy_df.iterrows():
    feature_name = feature_row["feature_name"]
    imputation_value = multimodal_imputation_lookup[feature_name]

    multimodal_feature_columns.append(feature_name)

    multimodal_feature_select_expressions.append(
        f"COALESCE({duckdb_identifier(feature_name)}, {float(imputation_value):.17g}) "
        f"AS {duckdb_identifier(feature_name)}"
    )

    if bool(feature_row["adds_multimodal_missing_indicator"]):
        missing_indicator_name = indicator_name(feature_name)
        multimodal_indicator_columns.append(missing_indicator_name)
        multimodal_feature_select_expressions.append(
            f"CASE WHEN {duckdb_identifier(feature_name)} IS NULL THEN 1 ELSE 0 END "
            f"AS {duckdb_identifier(missing_indicator_name)}"
        )

RAW_MATRIX_FEATURE_COLUMNS_BY_SET["multimodal"] = multimodal_feature_columns
RAW_MATRIX_INDICATOR_COLUMNS_BY_SET["multimodal"] = multimodal_indicator_columns

multimodal_output_paths = [
    RAW_MATRIX_OUTPUT_PATHS["multimodal"],
    RAW_METADATA_OUTPUT_PATHS["multimodal"],
]

for output_path in multimodal_output_paths:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        output_path.unlink()

with duckdb.connect() as con:
    con.execute(
        f"""
        COPY (
            SELECT
                {modeling_row_id_expression()},
                {", ".join(multimodal_feature_select_expressions)}
            FROM read_parquet('{panel_path_sql}')
            WHERE {primary_zone_exclusion_sql}
        )
        TO '{sql_path(RAW_MATRIX_OUTPUT_PATHS["multimodal"])}'
        (FORMAT PARQUET)
        """
    )

    con.execute(
        f"""
        COPY (
            SELECT
                {modeling_row_id_expression()},
                {", ".join(metadata_select_expressions())}
            FROM read_parquet('{panel_path_sql}')
            WHERE {primary_zone_exclusion_sql}
        )
        TO '{sql_path(RAW_METADATA_OUTPUT_PATHS["multimodal"])}'
        (FORMAT PARQUET)
        """
    )

multimodal_raw_matrix_export_df = pd.DataFrame(
    [
        {
            "feature_set": "multimodal",
            "matrix_path": str(RAW_MATRIX_OUTPUT_PATHS["multimodal"]),
            "metadata_path": str(RAW_METADATA_OUTPUT_PATHS["multimodal"]),
            "feature_count": len(multimodal_feature_columns),
            "indicator_count": len(multimodal_indicator_columns),
        }
    ]
)

display(multimodal_raw_matrix_export_df)

print("Multimodal raw modeling matrix and metadata materialized.")

,feature_set,matrix_path,metadata_path,feature_count,indicator_count
0,multimodal,pipeline_data/3.1.1.final_tables/multimodal_ra...,pipeline_data/3.1.1.final_tables/multimodal_ro...,142,91


Multimodal raw modeling matrix and metadata materialized.


Findings\. The Multimodal raw matrix was materialized with the full selected feature set: 142 selected features plus 91 observed\-null indicators\. This produces the expected 233 modeling columns once the indicators are included\. The output keeps the full eligible primary modeling universe of 1,541,800 rows, so Multimodal matrix construction preserves structurally valid Subway\-unavailable zones and sparse Taxi/FHVHV service observations rather than collapsing to a strict complete\-case universe\.

Validate the actual exported raw matrix files\. This check runs against the materialized parquet outputs, not just the policy tables\.

In [47]:
# ---------------------------------------------------------------------
# Validate materialized raw modeling matrices
# ---------------------------------------------------------------------

raw_matrix_validation_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        matrix_path = sql_path(RAW_MATRIX_OUTPUT_PATHS[feature_set_name])
        metadata_path = sql_path(RAW_METADATA_OUTPUT_PATHS[feature_set_name])

        matrix_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{matrix_path}')"
        ).fetchdf()
        matrix_columns = matrix_schema_df["column_name"].tolist()
        model_columns = [column for column in matrix_columns if column != "modeling_row_id"]

        metadata_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{metadata_path}')"
        ).fetchdf()
        metadata_columns = metadata_schema_df["column_name"].tolist()
        has_policy_geography_class = POLICY_GEOGRAPHY_METADATA_COLUMN in metadata_columns

        null_expression = " + ".join(
            f"SUM(CASE WHEN {duckdb_identifier(column)} IS NULL THEN 1 ELSE 0 END)"
            for column in model_columns
        ) or "0"

        matrix_row_count, remaining_null_cells = con.execute(
            f"""
            SELECT
                COUNT(*) AS matrix_rows,
                ({null_expression}) AS remaining_null_feature_cells
            FROM read_parquet('{matrix_path}')
            """
        ).fetchone()

        metadata_row_count = con.execute(
            f"""
            SELECT COUNT(*) AS metadata_rows
            FROM read_parquet('{metadata_path}')
            """
        ).fetchone()[0]

        duplicate_metadata_grain_rows = con.execute(
            f"""
            SELECT COALESCE(SUM(row_count - 1), 0)
            FROM (
                SELECT
                    taxi_zone_id,
                    CAST(date AS DATE) AS date,
                    temporal_bucket,
                    COUNT(*) AS row_count
                FROM read_parquet('{metadata_path}')
                GROUP BY 1, 2, 3
                HAVING COUNT(*) > 1
            )
            """
        ).fetchone()[0]

        if has_policy_geography_class:
            policy_geography_null_rows, policy_geography_class_count = con.execute(
                f"""
                SELECT
                    SUM(CASE WHEN {duckdb_identifier(POLICY_GEOGRAPHY_METADATA_COLUMN)} IS NULL THEN 1 ELSE 0 END)
                        AS policy_geography_null_rows,
                    COUNT(DISTINCT {duckdb_identifier(POLICY_GEOGRAPHY_METADATA_COLUMN)})
                        AS policy_geography_class_count
                FROM read_parquet('{metadata_path}')
                """
            ).fetchone()
        else:
            policy_geography_null_rows = metadata_row_count
            policy_geography_class_count = 0

        raw_matrix_validation_records.append(
            {
                "feature_set": feature_set_name,
                "matrix_rows": int(matrix_row_count),
                "metadata_rows": int(metadata_row_count),
                "model_column_count": len(model_columns),
                "selected_feature_count": len(RAW_MATRIX_FEATURE_COLUMNS_BY_SET[feature_set_name]),
                "indicator_column_count": len(RAW_MATRIX_INDICATOR_COLUMNS_BY_SET[feature_set_name]),
                "remaining_null_feature_cells": int(remaining_null_cells),
                "duplicate_metadata_grain_rows": int(duplicate_metadata_grain_rows),
                "has_policy_geography_class": has_policy_geography_class,
                "policy_geography_null_rows": int(policy_geography_null_rows),
                "policy_geography_class_count": int(policy_geography_class_count),
                "status": "pass"
                if (
                    matrix_row_count == metadata_row_count
                    and remaining_null_cells == 0
                    and duplicate_metadata_grain_rows == 0
                    and has_policy_geography_class
                    and policy_geography_null_rows == 0
                )
                else "review",
            }
        )

raw_matrix_validation_df = pd.DataFrame(raw_matrix_validation_records)

display(raw_matrix_validation_df)

assert raw_matrix_validation_df["status"].eq("pass").all(), (
    "One or more materialized raw matrices failed validation."
)

print("Materialized raw modeling matrices validated.")

,feature_set,matrix_rows,metadata_rows,model_column_count,selected_feature_count,indicator_column_count,remaining_null_feature_cells,duplicate_metadata_grain_rows,has_policy_geography_class,policy_geography_null_rows,policy_geography_class_count,status
0,bus,1472947,1472947,40,40,0,0,0,True,0,3,pass
1,subway,911455,911455,21,21,0,0,0,True,0,3,pass
2,taxi,1541800,1541800,53,38,15,0,0,True,0,3,pass
3,fhvhv,1541800,1541800,53,37,16,0,0,True,0,3,pass
4,multimodal,1541800,1541800,233,142,91,0,0,True,0,3,pass


Materialized raw modeling matrices validated.


Findings\. The exported raw matrices pass the key materialization checks\. For every feature set, the matrix row count matches the metadata row count, remaining feature\-null cells equal 0, duplicate metadata grain rows equal 0, and \`policy\_geography\_class\` is present with no null rows\. The row counts also reflect the resolved coverage policy: Bus retains 1,472,947 rows, Subway retains 911,455 rows, and Taxi, FHVHV, and Multimodal each retain 1,541,800 rows\. This confirms that cleanup rules and interpretation metadata were applied to the actual exported parquet assets, not just to intermediate policy tables\.

Summarize the raw matrix assets written by this section\. This is the handoff into matrix\-quality validation and scaling\.

In [49]:
# ---------------------------------------------------------------------
# Summarize raw matrix outputs
# ---------------------------------------------------------------------

raw_matrix_output_summary_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "raw_matrix_path": str(RAW_MATRIX_OUTPUT_PATHS[feature_set_name]),
            "row_metadata_path": str(RAW_METADATA_OUTPUT_PATHS[feature_set_name]),
            "matrix_rows": int(
                raw_matrix_validation_df.loc[
                    raw_matrix_validation_df["feature_set"] == feature_set_name,
                    "matrix_rows",
                ].iloc[0]
            ),
            "model_column_count": int(
                raw_matrix_validation_df.loc[
                    raw_matrix_validation_df["feature_set"] == feature_set_name,
                    "model_column_count",
                ].iloc[0]
            ),
            "remaining_null_feature_cells": int(
                raw_matrix_validation_df.loc[
                    raw_matrix_validation_df["feature_set"] == feature_set_name,
                    "remaining_null_feature_cells",
                ].iloc[0]
            ),
        }
        for feature_set_name in MODEL_FEATURE_SET_NAMES
    ]
)

display(raw_matrix_output_summary_df)

print("Raw matrix outputs summarized.")

,feature_set,raw_matrix_path,row_metadata_path,matrix_rows,model_column_count,remaining_null_feature_cells
0,bus,pipeline_data/3.1.1.final_tables/bus_raw_model...,pipeline_data/3.1.1.final_tables/bus_row_metad...,1472947,40,0
1,subway,pipeline_data/3.1.1.final_tables/subway_raw_mo...,pipeline_data/3.1.1.final_tables/subway_row_me...,911455,21,0
2,taxi,pipeline_data/3.1.1.final_tables/taxi_raw_mode...,pipeline_data/3.1.1.final_tables/taxi_row_meta...,1541800,53,0
3,fhvhv,pipeline_data/3.1.1.final_tables/fhvhv_raw_mod...,pipeline_data/3.1.1.final_tables/fhvhv_row_met...,1541800,53,0
4,multimodal,pipeline_data/3.1.1.final_tables/multimodal_ra...,pipeline_data/3.1.1.final_tables/multimodal_ro...,1541800,233,0


Raw matrix outputs summarized.


Findings\. The raw matrix output summary gives the handoff inventory for the next QA and scaling sections\. Each feature set now has a raw matrix parquet file and a matching row\-metadata parquet file under \`pipeline\_data/3\.1\.1\.final\_tables\`\. All five exported matrices report 0 remaining null feature cells, with model\-column counts of 40 for Bus, 21 for Subway, 53 for Taxi, 53 for FHVHV, and 233 for Multimodal\. The raw matrix layer is now materialized and ready for matrix\-quality validation\.

## 3\.1\.1\.6 Validate Modeling Matrix Quality

This section validates the raw modeling matrices that were materialized in 3\.1\.1\.5\. The goal is not to change row\-inclusion policy or rerun missingness decisions\. The goal is to prove that the exported parquet assets are structurally aligned, numerically safe, and suitable for scaling, PCA, UMAP, clustering, density estimation, and anomaly\-detection workflows\.

The checks focus on the exported matrix layer: file inventory, row counts, metadata alignment, \`modeling\_row\_id\` integrity, null and infinite values, numeric dtypes, constant or low\-information columns, and the burden of missingness indicators\. Any issues surfaced here should be handled before scaled matrices are created in 3\.1\.1\.7\.

### Validate Raw Matrix File Inventory

Start by checking the exported matrix and metadata files directly\. This confirms that every feature set has a raw matrix parquet file, a row\-metadata parquet file, matching row counts, and the expected number of modeling columns\.

In [50]:
# ---------------------------------------------------------------------
# Validate raw matrix and metadata file inventory
# ---------------------------------------------------------------------

raw_matrix_file_inventory_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        matrix_path = RAW_MATRIX_OUTPUT_PATHS[feature_set_name]
        metadata_path = RAW_METADATA_OUTPUT_PATHS[feature_set_name]

        matrix_exists = matrix_path.exists()
        metadata_exists = metadata_path.exists()

        matrix_rows = np.nan
        metadata_rows = np.nan
        model_column_count = np.nan
        metadata_column_count = np.nan

        if matrix_exists:
            matrix_schema_df = con.execute(
                f"DESCRIBE SELECT * FROM read_parquet('{sql_path(matrix_path)}')"
            ).fetchdf()
            matrix_columns = matrix_schema_df["column_name"].tolist()
            model_column_count = len([column for column in matrix_columns if column != "modeling_row_id"])
            matrix_rows = con.execute(
                f"SELECT COUNT(*) FROM read_parquet('{sql_path(matrix_path)}')"
            ).fetchone()[0]

        if metadata_exists:
            metadata_schema_df = con.execute(
                f"DESCRIBE SELECT * FROM read_parquet('{sql_path(metadata_path)}')"
            ).fetchdf()
            metadata_column_count = len(metadata_schema_df)
            metadata_rows = con.execute(
                f"SELECT COUNT(*) FROM read_parquet('{sql_path(metadata_path)}')"
            ).fetchone()[0]

        expected_model_columns = int(
            raw_matrix_output_summary_df.loc[
                raw_matrix_output_summary_df["feature_set"] == feature_set_name,
                "model_column_count",
            ].iloc[0]
        )

        raw_matrix_file_inventory_records.append(
            {
                "feature_set": feature_set_name,
                "matrix_exists": matrix_exists,
                "metadata_exists": metadata_exists,
                "matrix_rows": int(matrix_rows) if pd.notna(matrix_rows) else np.nan,
                "metadata_rows": int(metadata_rows) if pd.notna(metadata_rows) else np.nan,
                "model_column_count": int(model_column_count) if pd.notna(model_column_count) else np.nan,
                "expected_model_column_count": expected_model_columns,
                "metadata_column_count": int(metadata_column_count) if pd.notna(metadata_column_count) else np.nan,
                "status": "pass"
                if (
                    matrix_exists
                    and metadata_exists
                    and matrix_rows == metadata_rows
                    and model_column_count == expected_model_columns
                )
                else "review",
            }
        )

raw_matrix_file_inventory_df = pd.DataFrame(raw_matrix_file_inventory_records)

display(raw_matrix_file_inventory_df)

assert raw_matrix_file_inventory_df["status"].eq("pass").all(), (
    "One or more raw matrix files failed inventory validation."
)

print("Raw matrix file inventory validated.")

,feature_set,matrix_exists,metadata_exists,matrix_rows,metadata_rows,model_column_count,expected_model_column_count,metadata_column_count,status
0,bus,True,True,1472947,1472947,40,40,12,pass
1,subway,True,True,911455,911455,21,21,12,pass
2,taxi,True,True,1541800,1541800,53,53,12,pass
3,fhvhv,True,True,1541800,1541800,53,53,12,pass
4,multimodal,True,True,1541800,1541800,233,233,12,pass


Raw matrix file inventory validated.


Findings\. The raw matrix file inventory passes for all five feature sets\. Each raw matrix and row\-metadata table exists, row counts are aligned, and the exported assets reflect the expected coverage policy: Bus and Subway use smaller coverage\-aware row universes, while Taxi, FHVHV, and Multimodal retain the full eligible primary universe\. This confirms that matrix materialization produced real, readable parquet assets rather than only intermediate in\-memory tables\.

### Validate Row ID and Metadata Alignment

Check that \`modeling\_row\_id\` behaves like a clean join key\. The feature matrix and metadata table should have the same row IDs, no duplicates, and a contiguous ID range from 0 to row count minus 1\.

In [51]:
# ---------------------------------------------------------------------
# Validate modeling_row_id integrity and matrix/metadata alignment
# ---------------------------------------------------------------------

row_id_alignment_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        matrix_path = sql_path(RAW_MATRIX_OUTPUT_PATHS[feature_set_name])
        metadata_path = sql_path(RAW_METADATA_OUTPUT_PATHS[feature_set_name])

        matrix_id_stats = con.execute(
            f"""
            SELECT
                COUNT(*) AS row_count,
                COUNT(DISTINCT modeling_row_id) AS distinct_row_ids,
                MIN(modeling_row_id) AS min_row_id,
                MAX(modeling_row_id) AS max_row_id
            FROM read_parquet('{matrix_path}')
            """
        ).fetchone()

        metadata_id_stats = con.execute(
            f"""
            SELECT
                COUNT(*) AS row_count,
                COUNT(DISTINCT modeling_row_id) AS distinct_row_ids,
                MIN(modeling_row_id) AS min_row_id,
                MAX(modeling_row_id) AS max_row_id
            FROM read_parquet('{metadata_path}')
            """
        ).fetchone()

        matrix_ids_missing_from_metadata = con.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{matrix_path}') AS matrix
            LEFT JOIN read_parquet('{metadata_path}') AS metadata
                ON matrix.modeling_row_id = metadata.modeling_row_id
            WHERE metadata.modeling_row_id IS NULL
            """
        ).fetchone()[0]

        metadata_ids_missing_from_matrix = con.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{metadata_path}') AS metadata
            LEFT JOIN read_parquet('{matrix_path}') AS matrix
                ON metadata.modeling_row_id = matrix.modeling_row_id
            WHERE matrix.modeling_row_id IS NULL
            """
        ).fetchone()[0]

        duplicate_metadata_grain_rows = con.execute(
            f"""
            SELECT COALESCE(SUM(row_count - 1), 0)
            FROM (
                SELECT
                    taxi_zone_id,
                    CAST(date AS DATE) AS date,
                    temporal_bucket,
                    COUNT(*) AS row_count
                FROM read_parquet('{metadata_path}')
                GROUP BY 1, 2, 3
                HAVING COUNT(*) > 1
            )
            """
        ).fetchone()[0]

        matrix_row_count, matrix_distinct_ids, matrix_min_id, matrix_max_id = matrix_id_stats
        metadata_row_count, metadata_distinct_ids, metadata_min_id, metadata_max_id = metadata_id_stats

        matrix_ids_contiguous = (
            matrix_min_id == 0
            and matrix_max_id == matrix_row_count - 1
            and matrix_distinct_ids == matrix_row_count
        )
        metadata_ids_contiguous = (
            metadata_min_id == 0
            and metadata_max_id == metadata_row_count - 1
            and metadata_distinct_ids == metadata_row_count
        )

        row_id_alignment_records.append(
            {
                "feature_set": feature_set_name,
                "matrix_rows": int(matrix_row_count),
                "metadata_rows": int(metadata_row_count),
                "matrix_ids_contiguous": matrix_ids_contiguous,
                "metadata_ids_contiguous": metadata_ids_contiguous,
                "matrix_ids_missing_from_metadata": int(matrix_ids_missing_from_metadata),
                "metadata_ids_missing_from_matrix": int(metadata_ids_missing_from_matrix),
                "duplicate_metadata_grain_rows": int(duplicate_metadata_grain_rows),
                "status": "pass"
                if (
                    matrix_row_count == metadata_row_count
                    and matrix_ids_contiguous
                    and metadata_ids_contiguous
                    and matrix_ids_missing_from_metadata == 0
                    and metadata_ids_missing_from_matrix == 0
                    and duplicate_metadata_grain_rows == 0
                )
                else "review",
            }
        )

row_id_alignment_df = pd.DataFrame(row_id_alignment_records)

display(row_id_alignment_df)

assert row_id_alignment_df["status"].eq("pass").all(), (
    "One or more raw matrices failed row ID or metadata alignment validation."
)

print("Row ID and metadata alignment validated.")

,feature_set,matrix_rows,metadata_rows,matrix_ids_contiguous,metadata_ids_contiguous,matrix_ids_missing_from_metadata,metadata_ids_missing_from_matrix,duplicate_metadata_grain_rows,status
0,bus,1472947,1472947,True,True,0,0,0,pass
1,subway,911455,911455,True,True,0,0,0,pass
2,taxi,1541800,1541800,True,True,0,0,0,pass
3,fhvhv,1541800,1541800,True,True,0,0,0,pass
4,multimodal,1541800,1541800,True,True,0,0,0,pass


Row ID and metadata alignment validated.


Findings\. The modeling\_row\_id checks pass across the exported raw matrices and metadata tables\. Row IDs are aligned, unique, and contiguous, which means downstream notebooks can safely join model inputs back to interpretation metadata without relying on Taxi Zone, date, or temporal\-bucket joins\. This is especially important after modality\-specific row filtering because each matrix now has its own valid observation universe\.

### Validate Numeric Safety

Confirm that every modeling column is numeric and finite\. Metadata is kept in separate files, so the raw matrix files should contain only \`modeling\_row\_id\` plus numeric model inputs\.

In [52]:
# ---------------------------------------------------------------------
# Validate numeric dtypes, nulls, NaNs, and infinite values
# ---------------------------------------------------------------------

NUMERIC_TYPE_TOKENS = [
    "TINYINT",
    "SMALLINT",
    "INTEGER",
    "BIGINT",
    "HUGEINT",
    "UTINYINT",
    "USMALLINT",
    "UINTEGER",
    "UBIGINT",
    "FLOAT",
    "DOUBLE",
    "DECIMAL",
    "REAL",
    "BOOLEAN",
]


def is_duckdb_numeric_type(type_name):
    normalized_type = str(type_name).upper()
    return any(token in normalized_type for token in NUMERIC_TYPE_TOKENS)


numeric_safety_records = []
nonnumeric_column_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        matrix_path = sql_path(RAW_MATRIX_OUTPUT_PATHS[feature_set_name])
        matrix_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{matrix_path}')"
        ).fetchdf()

        model_schema_df = matrix_schema_df.loc[
            matrix_schema_df["column_name"] != "modeling_row_id"
        ].copy()

        nonnumeric_columns = model_schema_df.loc[
            ~model_schema_df["column_type"].apply(is_duckdb_numeric_type),
            ["column_name", "column_type"],
        ]

        for _, column_row in nonnumeric_columns.iterrows():
            nonnumeric_column_records.append(
                {
                    "feature_set": feature_set_name,
                    "column_name": column_row["column_name"],
                    "column_type": column_row["column_type"],
                }
            )

        model_columns = model_schema_df["column_name"].tolist()

        null_expression = " + ".join(
            f"SUM(CASE WHEN {duckdb_identifier(column)} IS NULL THEN 1 ELSE 0 END)"
            for column in model_columns
        ) or "0"

        nonfinite_expression = " + ".join(
            f"SUM(CASE WHEN {duckdb_identifier(column)} IS NOT NULL "
            f"AND NOT isfinite(CAST({duckdb_identifier(column)} AS DOUBLE)) "
            f"THEN 1 ELSE 0 END)"
            for column in model_columns
        ) or "0"

        remaining_null_feature_cells, nonfinite_feature_cells = con.execute(
            f"""
            SELECT
                ({null_expression}) AS remaining_null_feature_cells,
                ({nonfinite_expression}) AS nonfinite_feature_cells
            FROM read_parquet('{matrix_path}')
            """
        ).fetchone()

        numeric_safety_records.append(
            {
                "feature_set": feature_set_name,
                "model_column_count": len(model_columns),
                "nonnumeric_model_columns": len(nonnumeric_columns),
                "remaining_null_feature_cells": int(remaining_null_feature_cells),
                "nonfinite_feature_cells": int(nonfinite_feature_cells),
                "status": "pass"
                if (
                    len(nonnumeric_columns) == 0
                    and remaining_null_feature_cells == 0
                    and nonfinite_feature_cells == 0
                )
                else "review",
            }
        )

numeric_safety_df = pd.DataFrame(numeric_safety_records)
nonnumeric_columns_df = pd.DataFrame(nonnumeric_column_records)

display(numeric_safety_df)

if not nonnumeric_columns_df.empty:
    display(nonnumeric_columns_df)

assert numeric_safety_df["status"].eq("pass").all(), (
    "One or more raw matrices failed numeric safety validation."
)

print("Numeric safety validated for raw matrices.")

,feature_set,model_column_count,nonnumeric_model_columns,remaining_null_feature_cells,nonfinite_feature_cells,status
0,bus,40,0,0,0,pass
1,subway,21,0,0,0,pass
2,taxi,53,0,0,0,pass
3,fhvhv,53,0,0,0,pass
4,multimodal,233,0,0,0,pass


Numeric safety validated for raw matrices.


Findings\. The raw modeling matrices pass the numeric safety checks\. All model input columns are numeric, and the exported matrices contain 0 remaining null feature cells and 0 nonfinite feature cells\. This confirms that the missingness\-resolution rules were applied to the actual raw matrix files and that no NaN or infinite values remain before scaling\.

### Profile Constant and Low\-Information Columns

Profile columns that may add little modeling information or cause scaling issues\. Constant columns are hard failures for many transformations; very low\-distinct columns are not automatically wrong, but they should be visible before representation learning begins\.

In [53]:
# ---------------------------------------------------------------------
# Profile constant, near-zero-variance, and low-distinct columns
# ---------------------------------------------------------------------

LOW_STD_THRESHOLD = 1e-12

low_information_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        matrix_path = sql_path(RAW_MATRIX_OUTPUT_PATHS[feature_set_name])
        matrix_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{matrix_path}')"
        ).fetchdf()

        model_columns = matrix_schema_df.loc[
            matrix_schema_df["column_name"] != "modeling_row_id",
            "column_name",
        ].tolist()

        stat_expressions = []
        column_alias_lookup = []

        for column_index, column in enumerate(model_columns):
            prefix = f"c{column_index}"
            column_sql = duckdb_identifier(column)
            stat_expressions.extend(
                [
                    f"MIN(CAST({column_sql} AS DOUBLE)) AS {prefix}_min",
                    f"MAX(CAST({column_sql} AS DOUBLE)) AS {prefix}_max",
                    f"AVG(CAST({column_sql} AS DOUBLE)) AS {prefix}_mean",
                    f"STDDEV_SAMP(CAST({column_sql} AS DOUBLE)) AS {prefix}_std",
                    f"APPROX_COUNT_DISTINCT({column_sql}) AS {prefix}_approx_distinct",
                ]
            )
            column_alias_lookup.append((prefix, column))

        stats_row = con.execute(
            f"""
            SELECT
                {", ".join(stat_expressions)}
            FROM read_parquet('{matrix_path}')
            """
        ).fetchdf().iloc[0]

        indicator_columns = set(RAW_MATRIX_INDICATOR_COLUMNS_BY_SET.get(feature_set_name, []))

        for prefix, column in column_alias_lookup:
            min_value = stats_row[f"{prefix}_min"]
            max_value = stats_row[f"{prefix}_max"]
            mean_value = stats_row[f"{prefix}_mean"]
            std_value = stats_row[f"{prefix}_std"]
            approx_distinct = int(stats_row[f"{prefix}_approx_distinct"])
            is_indicator = column in indicator_columns
            is_constant = pd.notna(min_value) and pd.notna(max_value) and min_value == max_value
            near_zero_std = pd.notna(std_value) and abs(std_value) <= LOW_STD_THRESHOLD
            low_distinct_non_indicator = (not is_indicator) and approx_distinct <= 2

            low_information_records.append(
                {
                    "feature_set": feature_set_name,
                    "feature_name": column,
                    "is_indicator_column": is_indicator,
                    "min_value": min_value,
                    "max_value": max_value,
                    "mean_value": mean_value,
                    "std_value": std_value,
                    "approx_distinct_values": approx_distinct,
                    "is_constant": is_constant,
                    "near_zero_std": near_zero_std,
                    "low_distinct_non_indicator": low_distinct_non_indicator,
                    "requires_review": is_constant or near_zero_std or low_distinct_non_indicator,
                }
            )

low_information_profile_df = pd.DataFrame(low_information_records)

low_information_summary_df = (
    low_information_profile_df
    .groupby("feature_set", dropna=False)
    .agg(
        model_column_count=("feature_name", "nunique"),
        indicator_column_count=("is_indicator_column", "sum"),
        constant_column_count=("is_constant", "sum"),
        near_zero_std_column_count=("near_zero_std", "sum"),
        low_distinct_non_indicator_count=("low_distinct_non_indicator", "sum"),
        review_column_count=("requires_review", "sum"),
    )
    .reset_index()
)

low_information_review_df = (
    low_information_profile_df
    .loc[low_information_profile_df["requires_review"]]
    .sort_values(
        ["feature_set", "is_constant", "near_zero_std", "approx_distinct_values", "feature_name"],
        ascending=[True, False, False, True, True],
    )
    .reset_index(drop=True)
)

display(low_information_summary_df)

if not low_information_review_df.empty:
    display(low_information_review_df.head(50))

print("Low-information column profile created.")

,feature_set,model_column_count,indicator_column_count,constant_column_count,near_zero_std_column_count,low_distinct_non_indicator_count,review_column_count
0,bus,40,0,0,0,3,3
1,fhvhv,53,16,7,7,3,10
2,multimodal,233,91,0,0,3,3
3,subway,21,0,0,0,3,3
4,taxi,53,15,0,0,3,3


,feature_set,feature_name,is_indicator_column,min_value,max_value,mean_value,std_value,approx_distinct_values,is_constant,near_zero_std,low_distinct_non_indicator,requires_review
0,bus,is_cbd_adjacent_zone,False,0.0,1.0,0.028134,0.165356,2,False,False,True,True
1,bus,is_cbd_gateway_zone,False,0.0,1.0,0.042489,0.201702,2,False,False,True,True
2,bus,is_cbd_zone,False,0.0,1.0,0.149266,0.356351,2,False,False,True,True
3,fhvhv,borough_mean_fhvhv_avg_trip_speed_missing_indi...,True,0.0,0.0,0.000000,0.000000,1,True,True,False,True
4,fhvhv,fhvhv_avg_trip_distance_missing_indicator,True,0.0,0.0,0.000000,0.000000,1,True,True,False,True
5,fhvhv,fhvhv_avg_trip_speed_cp_abs_delta_same_bucket_...,True,0.0,0.0,0.000000,0.000000,1,True,True,False,True
6,fhvhv,fhvhv_avg_trip_speed_missing_indicator,True,0.0,0.0,0.000000,0.000000,1,True,True,False,True
7,fhvhv,fhvhv_avg_trip_speed_post_cp_mean_same_bucket_...,True,0.0,0.0,0.000000,0.000000,1,True,True,False,True
8,fhvhv,fhvhv_avg_trip_speed_pre_cp_mean_same_bucket_m...,True,0.0,0.0,0.000000,0.000000,1,True,True,False,True
9,fhvhv,fhvhv_avg_trip_speed_rolling_mean_7_same_bucke...,True,0.0,0.0,0.000000,0.000000,1,True,True,False,True


Low-information column profile created.


Findings\. Most matrices are clear of constant or near\-zero\-variance columns\. Bus, Subway, Taxi, and Multimodal do not have constant/near\-zero columns requiring exclusion\. FHVHV is the only matrix flagged for review, with 7 constant or near\-zero columns\. This is not a raw\-matrix construction failure; it is a scaling\-readiness issue\. Those columns should be excluded before scaling because they add no modeling signal and can create unstable or meaningless standardized values\.

### Profile Missingness Indicator Burden

Summarize the missingness indicators created by the sparse\-service and neutral\-imputation policies\. These columns are valid model inputs, but their prevalence matters because later representations may capture coverage structure as strongly as mobility structure\.

In [54]:
# ---------------------------------------------------------------------
# Profile missingness indicator burden by matrix
# ---------------------------------------------------------------------

indicator_burden_records = []
indicator_validation_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        matrix_path = sql_path(RAW_MATRIX_OUTPUT_PATHS[feature_set_name])
        indicator_columns = RAW_MATRIX_INDICATOR_COLUMNS_BY_SET.get(feature_set_name, [])
        row_count = int(
            raw_matrix_file_inventory_df.loc[
                raw_matrix_file_inventory_df["feature_set"] == feature_set_name,
                "matrix_rows",
            ].iloc[0]
        )

        if not indicator_columns:
            indicator_validation_records.append(
                {
                    "feature_set": feature_set_name,
                    "indicator_column_count": 0,
                    "total_indicator_active_cells": 0,
                    "max_indicator_active_pct": 0.0,
                    "median_indicator_active_pct": 0.0,
                    "nonbinary_indicator_cells": 0,
                    "status": "pass",
                }
            )
            continue

        indicator_stat_expressions = []
        for column_index, column in enumerate(indicator_columns):
            prefix = f"i{column_index}"
            column_sql = duckdb_identifier(column)
            indicator_stat_expressions.extend(
                [
                    f"SUM(CAST({column_sql} AS BIGINT)) AS {prefix}_active_rows",
                    f"SUM(CASE WHEN {column_sql} NOT IN (0, 1) THEN 1 ELSE 0 END) AS {prefix}_nonbinary_cells",
                ]
            )

        indicator_stats_row = con.execute(
            f"""
            SELECT
                {", ".join(indicator_stat_expressions)}
            FROM read_parquet('{matrix_path}')
            """
        ).fetchdf().iloc[0]

        active_pct_values = []
        total_active_cells = 0
        total_nonbinary_cells = 0

        for column_index, column in enumerate(indicator_columns):
            prefix = f"i{column_index}"
            active_rows = int(indicator_stats_row[f"{prefix}_active_rows"])
            nonbinary_cells = int(indicator_stats_row[f"{prefix}_nonbinary_cells"])
            active_pct = active_rows / row_count if row_count else 0.0

            total_active_cells += active_rows
            total_nonbinary_cells += nonbinary_cells
            active_pct_values.append(active_pct)

            indicator_burden_records.append(
                {
                    "feature_set": feature_set_name,
                    "indicator_column": column,
                    "matrix_rows": row_count,
                    "active_rows": active_rows,
                    "active_pct": round(active_pct, 5),
                    "nonbinary_cells": nonbinary_cells,
                }
            )

        indicator_validation_records.append(
            {
                "feature_set": feature_set_name,
                "indicator_column_count": len(indicator_columns),
                "total_indicator_active_cells": int(total_active_cells),
                "max_indicator_active_pct": round(max(active_pct_values), 5),
                "median_indicator_active_pct": round(float(np.median(active_pct_values)), 5),
                "nonbinary_indicator_cells": int(total_nonbinary_cells),
                "status": "pass" if total_nonbinary_cells == 0 else "review",
            }
        )

indicator_burden_df = pd.DataFrame(indicator_burden_records)
indicator_burden_summary_df = pd.DataFrame(indicator_validation_records)

display(indicator_burden_summary_df)

if not indicator_burden_df.empty:
    display(
        indicator_burden_df.sort_values(
            ["active_pct", "feature_set", "indicator_column"],
            ascending=[False, True, True],
        ).head(30)
    )

assert indicator_burden_summary_df["status"].eq("pass").all(), (
    "One or more missingness indicator columns contain non-binary values."
)

print("Missingness indicator burden profiled.")

,feature_set,indicator_column_count,total_indicator_active_cells,max_indicator_active_pct,median_indicator_active_pct,nonbinary_indicator_cells,status
0,bus,0,0,0.00000,0.00000,0,pass
1,subway,0,0,0.00000,0.00000,0,pass
2,taxi,15,4280066,0.28538,0.27776,0,pass
3,fhvhv,16,11068,0.00169,0.00022,0,pass
4,multimodal,91,15214477,0.43462,0.04238,0,pass


,feature_set,indicator_column,matrix_rows,active_rows,active_pct,nonbinary_cells
115,multimodal,subway_ridership_local_vs_connected_zscore_mis...,1541800,670090,0.43462,0
83,multimodal,subway_ridership_pct_change_1_same_bucket_miss...,1541800,630345,0.40884,0
81,multimodal,subway_ridership_deviation_abs_zscore_missing_...,1541800,623485,0.40439,0
78,multimodal,subway_ridership_abs_change_1_same_bucket_miss...,1541800,618280,0.40101,0
87,multimodal,subway_ridership_rolling_std_7_same_bucket_mis...,1541800,618280,0.40101,0
79,multimodal,subway_ridership_cp_abs_delta_same_bucket_miss...,1541800,616720,0.40000,0
80,multimodal,subway_ridership_cp_pct_delta_same_bucket_miss...,1541800,616720,0.40000,0
82,multimodal,subway_ridership_fourier20_residual_reconstruc...,1541800,616720,0.40000,0
85,multimodal,subway_ridership_residual_abs_missing_indicator,1541800,616720,0.40000,0
84,multimodal,subway_ridership_residual_missing_indicator,1541800,616720,0.40000,0


Missingness indicator burden profiled.


Findings\. Missingness indicators are concentrated where the coverage policy preserves valid observations rather than filtering them away\. Bus and Subway have no indicator columns, while Taxi has 15 indicators and FHVHV has 16\. Multimodal carries the largest indicator burden with 91 indicators because it combines the previously resolved primary and intermodal coverage patterns\. All indicator columns are binary, so they are safe to carry into scaling policy decisions\.

### Create Matrix Quality Summary

Roll the matrix\-quality checks into one compact table\. Hard gates cover file presence, row alignment, numeric safety, nulls, and infinities\. Review flags capture constant or low\-information columns that may require handling before scaling\.

In [55]:
# ---------------------------------------------------------------------
# Create final raw matrix quality summary
# ---------------------------------------------------------------------

matrix_quality_summary_df = (
    raw_matrix_file_inventory_df[
        [
            "feature_set",
            "matrix_rows",
            "metadata_rows",
            "model_column_count",
            "status",
        ]
    ]
    .rename(columns={"status": "file_inventory_status"})
    .merge(
        row_id_alignment_df[
            [
                "feature_set",
                "matrix_ids_contiguous",
                "metadata_ids_contiguous",
                "matrix_ids_missing_from_metadata",
                "metadata_ids_missing_from_matrix",
                "duplicate_metadata_grain_rows",
                "status",
            ]
        ].rename(columns={"status": "row_alignment_status"}),
        on="feature_set",
        how="left",
    )
    .merge(
        numeric_safety_df[
            [
                "feature_set",
                "nonnumeric_model_columns",
                "remaining_null_feature_cells",
                "nonfinite_feature_cells",
                "status",
            ]
        ].rename(columns={"status": "numeric_safety_status"}),
        on="feature_set",
        how="left",
    )
    .merge(
        low_information_summary_df[
            [
                "feature_set",
                "indicator_column_count",
                "constant_column_count",
                "near_zero_std_column_count",
                "low_distinct_non_indicator_count",
                "review_column_count",
            ]
        ],
        on="feature_set",
        how="left",
    )
    .merge(
        indicator_burden_summary_df[
            [
                "feature_set",
                "total_indicator_active_cells",
                "max_indicator_active_pct",
                "median_indicator_active_pct",
                "nonbinary_indicator_cells",
                "status",
            ]
        ].rename(columns={"status": "indicator_status"}),
        on="feature_set",
        how="left",
    )
)

hard_gate_columns = [
    "file_inventory_status",
    "row_alignment_status",
    "numeric_safety_status",
    "indicator_status",
]

matrix_quality_summary_df["hard_gates_pass"] = matrix_quality_summary_df[
    hard_gate_columns
].eq("pass").all(axis=1)

matrix_quality_summary_df["quality_review_flags"] = matrix_quality_summary_df[
    [
        "constant_column_count",
        "near_zero_std_column_count",
        "low_distinct_non_indicator_count",
    ]
].sum(axis=1)

matrix_quality_summary_df["matrix_quality_status"] = np.where(
    matrix_quality_summary_df["hard_gates_pass"]
    & (matrix_quality_summary_df["constant_column_count"] == 0),
    "pass",
    "review",
)

ordered_quality_columns = [
    "feature_set",
    "matrix_rows",
    "metadata_rows",
    "model_column_count",
    "indicator_column_count",
    "remaining_null_feature_cells",
    "nonfinite_feature_cells",
    "nonnumeric_model_columns",
    "duplicate_metadata_grain_rows",
    "constant_column_count",
    "near_zero_std_column_count",
    "low_distinct_non_indicator_count",
    "max_indicator_active_pct",
    "median_indicator_active_pct",
    "hard_gates_pass",
    "quality_review_flags",
    "matrix_quality_status",
]

matrix_quality_summary_df = matrix_quality_summary_df[ordered_quality_columns]

display(matrix_quality_summary_df)

hard_gate_failures = matrix_quality_summary_df.loc[
    ~matrix_quality_summary_df["hard_gates_pass"]
]

assert hard_gate_failures.empty, (
    "One or more raw matrices failed a hard matrix-quality gate."
)

print("Raw matrix quality summary created.")

,feature_set,matrix_rows,metadata_rows,model_column_count,indicator_column_count,remaining_null_feature_cells,nonfinite_feature_cells,nonnumeric_model_columns,duplicate_metadata_grain_rows,constant_column_count,near_zero_std_column_count,low_distinct_non_indicator_count,max_indicator_active_pct,median_indicator_active_pct,hard_gates_pass,quality_review_flags,matrix_quality_status
0,bus,1472947,1472947,40,0,0,0,0,0,0,0,3,0.00000,0.00000,True,3,pass
1,subway,911455,911455,21,0,0,0,0,0,0,0,3,0.00000,0.00000,True,3,pass
2,taxi,1541800,1541800,53,15,0,0,0,0,0,0,3,0.28538,0.27776,True,3,pass
3,fhvhv,1541800,1541800,53,16,0,0,0,0,7,7,3,0.00169,0.00022,True,17,review
4,multimodal,1541800,1541800,233,91,0,0,0,0,0,0,3,0.43462,0.04238,True,3,pass


Raw matrix quality summary created.


Findings\. The raw matrices pass the hard quality gates needed before scaling\. All five matrix files have matching metadata row counts, clean row identifiers, 0 remaining null feature cells, 0 nonfinite feature cells, 0 nonnumeric model columns, and 0 duplicate metadata grain rows\. Bus, Subway, Taxi, and Multimodal are fully clear on the final status table\. FHVHV is the only matrix marked for review, not because of missingness or alignment, but because 7 columns are constant or near\-zero variance\. That is a scaling\-readiness issue rather than a matrix\-construction failure: constant columns do not add modeling signal and should be handled in 3\.1\.1\.7 before fitting scalers or downstream representations\. The indicator checks also pass: Taxi has 15 indicators, FHVHV has 16, and Multimodal has 91, with all indicator values binary\.

### Section Summary

>  The exported raw matrices are structurally valid and numerically safe for scaling\. The only raw\-matrix QA item requiring follow\-up is the FHVHV low\-information\-column flag: 7 constant or near\-zero\-variance columns should be excluded during scaling because they add no modeling signal\. This is a scaling\-readiness issue rather than a matrix\-construction failure, and it is carried forward into 3\.1\.1\.7 for explicit handling before downstream representation learning or anomaly detection\.

## 3\.1\.1\.7 Scale Modeling Matrices

This section converts the validated raw modeling matrices into scaled modeling matrices for downstream representation learning and anomaly workflows\. The raw matrices remain unchanged\. Scaling is applied only to eligible continuous model columns, while missingness indicators remain binary so they preserve their interpretation as flags rather than becoming standardized pseudo\-continuous features\.

The only matrix\-quality issue carried forward from 3\.1\.1\.6 is the small set of FHVHV constant or near\-zero\-variance columns\. Those columns should not be passed through a standard scaler because they do not contain usable variation\. This section therefore documents column\-level scaling actions, writes scaled matrices for Bus, Subway, Taxi, FHVHV, and Multimodal, and validates that scaled outputs retain row alignment, contain no null or infinite values, and have the expected scaled\-column behavior\.

### Define Scaling Actions

Start by assigning one scaling action to every raw model column\. Continuous and count\-like modeling features are eligible for standard scaling, missingness indicators stay as binary 0/1 flags, and constant or near\-zero\-variance columns are excluded from the scaled matrix and documented for traceability\.

In [56]:
# ---------------------------------------------------------------------
# Define column-level scaling actions for each raw modeling matrix
# ---------------------------------------------------------------------

SCALING_ACTION_STANDARD_SCALE = "standard_scale"
SCALING_ACTION_KEEP_BINARY = "keep_binary_indicator"
SCALING_ACTION_EXCLUDE_CONSTANT = "exclude_constant_or_near_zero_variance"


def coerce_bool(value):
    """Convert QA table values into reliable booleans."""
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes", "y"}
    return bool(value)


low_information_scaling_source_df = low_information_profile_df.copy()

for flag_column in ["is_indicator_column", "is_constant", "near_zero_std", "low_distinct_non_indicator"]:
    low_information_scaling_source_df[f"{flag_column}_bool"] = low_information_scaling_source_df[
        flag_column
    ].apply(coerce_bool)

low_information_scaling_source_df["should_exclude_from_scaled_matrix"] = (
    low_information_scaling_source_df["is_constant_bool"]
    | low_information_scaling_source_df["near_zero_std_bool"]
)

scaling_action_records = []

for _, column_row in low_information_scaling_source_df.iterrows():
    feature_set = column_row["feature_set"]
    feature_name = column_row["feature_name"]
    is_indicator = column_row["is_indicator_column_bool"]
    should_exclude = column_row["should_exclude_from_scaled_matrix"]

    if should_exclude:
        scaling_action = SCALING_ACTION_EXCLUDE_CONSTANT
        scaled_matrix_inclusion = "exclude_from_scaled_matrix"
    elif is_indicator:
        scaling_action = SCALING_ACTION_KEEP_BINARY
        scaled_matrix_inclusion = "include_unscaled"
    else:
        scaling_action = SCALING_ACTION_STANDARD_SCALE
        scaled_matrix_inclusion = "include_scaled"

    scaling_action_records.append(
        {
            "feature_set": feature_set,
            "feature_name": feature_name,
            "scaling_action": scaling_action,
            "scaled_matrix_inclusion": scaled_matrix_inclusion,
            "is_indicator_column": is_indicator,
            "is_constant": column_row["is_constant_bool"],
            "near_zero_std": column_row["near_zero_std_bool"],
            "low_distinct_non_indicator": column_row["low_distinct_non_indicator_bool"],
            "raw_min": column_row["min_value"],
            "raw_max": column_row["max_value"],
            "raw_mean": column_row["mean_value"],
            "raw_std": column_row["std_value"],
            "approx_distinct": column_row["approx_distinct_values"],
        }
    )

scaling_action_df = pd.DataFrame(scaling_action_records)

scaling_action_summary_df = (
    scaling_action_df
    .groupby(["feature_set", "scaling_action", "scaled_matrix_inclusion"], as_index=False)
    .agg(column_count=("feature_name", "nunique"))
    .sort_values(["feature_set", "scaled_matrix_inclusion", "scaling_action"])
    .reset_index(drop=True)
)

excluded_scaling_columns_df = (
    scaling_action_df
    .loc[scaling_action_df["scaling_action"].eq(SCALING_ACTION_EXCLUDE_CONSTANT)]
    .sort_values(["feature_set", "raw_std", "feature_name"])
    .reset_index(drop=True)
)

low_distinct_scaling_review_df = (
    scaling_action_df
    .loc[
        scaling_action_df["low_distinct_non_indicator"]
        & scaling_action_df["scaling_action"].eq(SCALING_ACTION_STANDARD_SCALE)
    ]
    .sort_values(["feature_set", "feature_name"])
    .reset_index(drop=True)
)

scaling_action_integrity_df = (
    scaling_action_df
    .groupby("feature_set", as_index=False)
    .agg(action_column_count=("feature_name", "nunique"))
    .merge(
        raw_matrix_file_inventory_df[["feature_set", "model_column_count"]],
        on="feature_set",
        how="left",
    )
)
scaling_action_integrity_df["status"] = np.where(
    scaling_action_integrity_df["action_column_count"].eq(scaling_action_integrity_df["model_column_count"]),
    "pass",
    "review",
)

scaling_action_exclusion_check_df = (
    low_information_scaling_source_df
    .groupby("feature_set", as_index=False)
    .agg(expected_exclusion_columns=("should_exclude_from_scaled_matrix", "sum"))
    .merge(
        scaling_action_df
        .assign(assigned_exclusion_action=lambda df: df["scaling_action"].eq(SCALING_ACTION_EXCLUDE_CONSTANT))
        .groupby("feature_set", as_index=False)
        .agg(assigned_exclusion_columns=("assigned_exclusion_action", "sum")),
        on="feature_set",
        how="left",
    )
)
scaling_action_exclusion_check_df["status"] = np.where(
    scaling_action_exclusion_check_df["expected_exclusion_columns"].eq(
        scaling_action_exclusion_check_df["assigned_exclusion_columns"]
    ),
    "pass",
    "review",
)

assert scaling_action_integrity_df["status"].eq("pass").all(), "Some raw matrix columns are missing scaling actions."
assert scaling_action_exclusion_check_df["status"].eq("pass").all(), (
    "Some constant or near-zero-variance columns were not assigned exclusion actions."
)

print("Scaling action tables created.")

Scaling action tables created.


Summarize the scaling actions before materializing scaled matrices. The summary table confirms how many columns will be scaled, kept binary, or excluded, while the detail tables identify columns that need special treatment.

In [57]:
# ---------------------------------------------------------------------
# Summarize scaling actions by matrix
# ---------------------------------------------------------------------

display(scaling_action_summary_df)

,feature_set,scaling_action,scaled_matrix_inclusion,column_count
0,bus,standard_scale,include_scaled,40
1,fhvhv,exclude_constant_or_near_zero_variance,exclude_from_scaled_matrix,7
2,fhvhv,standard_scale,include_scaled,37
3,fhvhv,keep_binary_indicator,include_unscaled,9
4,multimodal,standard_scale,include_scaled,142
5,multimodal,keep_binary_indicator,include_unscaled,91
6,subway,standard_scale,include_scaled,21
7,taxi,standard_scale,include_scaled,38
8,taxi,keep_binary_indicator,include_unscaled,15


Findings\. The scaling\-action summary now separates the three intended treatments cleanly\. Continuous modeling features are assigned to standard scaling, missingness indicators with actual variation are kept as unscaled binary flags, and constant or near\-zero\-variance columns are excluded from the scaled output\. FHVHV is the only matrix with exclusions: 7 all\-zero indicator columns are removed before scaling\. That is the right outcome because those columns contain no information for downstream PCA, UMAP, clustering, or anomaly workflows\.

Check that every raw model column received exactly one scaling action\. This is a bookkeeping gate: the counts should match the raw matrix column counts for every feature set\.

In [58]:
# ---------------------------------------------------------------------
# Validate scaling-action assignment coverage
# ---------------------------------------------------------------------

display(scaling_action_integrity_df)

,feature_set,action_column_count,model_column_count,status
0,bus,40,40,pass
1,fhvhv,53,53,pass
2,multimodal,233,233,pass
3,subway,21,21,pass
4,taxi,53,53,pass


Findings\. The scaling\-action coverage check passes across all five matrices\. The action counts match the raw model\-column counts exactly: 40 Bus columns, 21 Subway columns, 53 Taxi columns, 53 FHVHV columns, and 233 Multimodal columns\. No matrix columns are missing from the scaling policy\.

Inspect the columns excluded from scaled matrices\. Exclusions should be rare and should correspond to columns with no usable variation\.

In [59]:
# ---------------------------------------------------------------------
# Inspect columns excluded from scaled matrices
# ---------------------------------------------------------------------

if excluded_scaling_columns_df.empty:
    print("No constant or near-zero-variance columns excluded from scaled matrices.")
else:
    display(
        excluded_scaling_columns_df[
            [
                "feature_set",
                "feature_name",
                "scaling_action",
                "raw_min",
                "raw_max",
                "raw_std",
                "approx_distinct",
            ]
        ]
    )

,feature_set,feature_name,scaling_action,raw_min,raw_max,raw_std,approx_distinct
0,fhvhv,borough_mean_fhvhv_avg_trip_speed_missing_indi...,exclude_constant_or_near_zero_variance,0.0,0.0,0.0,1
1,fhvhv,fhvhv_avg_trip_distance_missing_indicator,exclude_constant_or_near_zero_variance,0.0,0.0,0.0,1
2,fhvhv,fhvhv_avg_trip_speed_cp_abs_delta_same_bucket_...,exclude_constant_or_near_zero_variance,0.0,0.0,0.0,1
3,fhvhv,fhvhv_avg_trip_speed_missing_indicator,exclude_constant_or_near_zero_variance,0.0,0.0,0.0,1
4,fhvhv,fhvhv_avg_trip_speed_post_cp_mean_same_bucket_...,exclude_constant_or_near_zero_variance,0.0,0.0,0.0,1
5,fhvhv,fhvhv_avg_trip_speed_pre_cp_mean_same_bucket_m...,exclude_constant_or_near_zero_variance,0.0,0.0,0.0,1
6,fhvhv,fhvhv_avg_trip_speed_rolling_mean_7_same_bucke...,exclude_constant_or_near_zero_variance,0.0,0.0,0.0,1


Findings\. The excluded columns are all FHVHV missingness indicators with no active rows: min 0, max 0, standard deviation 0, and one distinct value\. These indicators do not encode any observed missingness pattern in the resolved FHVHV matrix, so keeping them would only add dead dimensions to the scaled feature space\.

Review low\-distinct non\-indicator columns that remain eligible for standard scaling\. These are not missingness indicators; they are binary spatial/context features that still contain variation\.

In [60]:
# ---------------------------------------------------------------------
# Review low-distinct non-indicator columns retained for scaling
# ---------------------------------------------------------------------

if low_distinct_scaling_review_df.empty:
    print("No low-distinct non-indicator columns retained for standard scaling.")
else:
    display(
        low_distinct_scaling_review_df[
            [
                "feature_set",
                "feature_name",
                "scaling_action",
                "raw_min",
                "raw_max",
                "raw_std",
                "approx_distinct",
            ]
        ]
    )

,feature_set,feature_name,scaling_action,raw_min,raw_max,raw_std,approx_distinct
0,bus,is_cbd_adjacent_zone,standard_scale,0.0,1.0,0.165356,2
1,bus,is_cbd_gateway_zone,standard_scale,0.0,1.0,0.201702,2
2,bus,is_cbd_zone,standard_scale,0.0,1.0,0.356351,2
3,fhvhv,is_cbd_adjacent_zone,standard_scale,0.0,1.0,0.161859,2
4,fhvhv,is_cbd_gateway_zone,standard_scale,0.0,1.0,0.209818,2
5,fhvhv,is_cbd_zone,standard_scale,0.0,1.0,0.360801,2
6,multimodal,is_cbd_adjacent_zone,standard_scale,0.0,1.0,0.161859,2
7,multimodal,is_cbd_gateway_zone,standard_scale,0.0,1.0,0.209818,2
8,multimodal,is_cbd_zone,standard_scale,0.0,1.0,0.360801,2
9,subway,is_cbd_adjacent_zone,standard_scale,0.0,1.0,0.193478,2


Findings\. The low\-distinct retained columns are the CBD spatial flags: \`is\_cbd\_zone\`, \`is\_cbd\_adjacent\_zone\`, and \`is\_cbd\_gateway\_zone\` across the feature sets where they appear\. They are binary by design and have nonzero variation, so they are different from the all\-zero FHVHV missingness indicators\. Keeping them is defensible because they encode policy\-relevant geography rather than missingness bookkeeping\.

Reconcile the low\-information QA from the previous section against the scaling\-action table before moving on\. This check confirms whether constant or near\-zero\-variance columns are still present and whether those columns are being assigned the expected exclusion action\.

In [61]:
# ---------------------------------------------------------------------
# Summarize low-information QA and scaling-exclusion reconciliation
# ---------------------------------------------------------------------

low_information_reconciliation_df = low_information_profile_df.copy()

for flag_column in ["is_constant", "near_zero_std", "low_distinct_non_indicator"]:
    low_information_reconciliation_df[f"{flag_column}_bool"] = low_information_reconciliation_df[
        flag_column
    ].apply(coerce_bool)

low_information_reconciliation_df["should_exclude_from_scaled_matrix"] = (
    low_information_reconciliation_df["is_constant_bool"]
    | low_information_reconciliation_df["near_zero_std_bool"]
)

expected_exclusion_summary_df = (
    low_information_reconciliation_df
    .groupby("feature_set", as_index=False)
    .agg(
        constant_or_near_zero_columns=("should_exclude_from_scaled_matrix", "sum"),
        low_distinct_non_indicator_columns=("low_distinct_non_indicator_bool", "sum"),
    )
)

actual_exclusion_summary_df = (
    scaling_action_df
    .assign(
        assigned_exclusion_action=lambda df: df["scaling_action"].eq(SCALING_ACTION_EXCLUDE_CONSTANT)
    )
    .groupby("feature_set", as_index=False)
    .agg(assigned_exclusion_columns=("assigned_exclusion_action", "sum"))
)

scaling_exclusion_reconciliation_df = (
    expected_exclusion_summary_df
    .merge(actual_exclusion_summary_df, on="feature_set", how="left")
    .fillna({"assigned_exclusion_columns": 0})
)
scaling_exclusion_reconciliation_df["assigned_exclusion_columns"] = scaling_exclusion_reconciliation_df[
    "assigned_exclusion_columns"
].astype(int)
scaling_exclusion_reconciliation_df["status"] = np.where(
    scaling_exclusion_reconciliation_df["constant_or_near_zero_columns"].eq(
        scaling_exclusion_reconciliation_df["assigned_exclusion_columns"]
    ),
    "pass",
    "review",
)

exclusion_mismatch_detail_df = (
    low_information_reconciliation_df
    .loc[low_information_reconciliation_df["should_exclude_from_scaled_matrix"]]
    .merge(
        scaling_action_df[["feature_set", "feature_name", "scaling_action", "scaled_matrix_inclusion"]],
        on=["feature_set", "feature_name"],
        how="left",
    )
    .sort_values(["feature_set", "feature_name"])
    .reset_index(drop=True)
)

display(scaling_exclusion_reconciliation_df)

,feature_set,constant_or_near_zero_columns,low_distinct_non_indicator_columns,assigned_exclusion_columns,status
0,bus,0,3,0,pass
1,fhvhv,7,3,7,pass
2,multimodal,0,3,0,pass
3,subway,0,3,0,pass
4,taxi,0,3,0,pass


Findings\. The reconciliation now passes\. FHVHV has 7 constant or near\-zero columns and 7 assigned exclusions, while Bus, Subway, Taxi, and Multimodal have no constant or near\-zero columns requiring exclusion\. The recurring 3 low\-distinct non\-indicator columns in each feature set are the CBD spatial flags, not scaling failures\.

Inspect the reconciled exclusion details only for columns that are actually removed from scaled matrices\. This table should stay narrow: it names the excluded columns and shows why they have no scaling value\.

In [62]:
# ---------------------------------------------------------------------
# Inspect reconciled scaling-exclusion detail
# ---------------------------------------------------------------------

if exclusion_mismatch_detail_df.empty:
    print("No constant or near-zero-variance columns are currently flagged by the low-information QA table.")
else:
    display(
        exclusion_mismatch_detail_df[
            [
                "feature_set",
                "feature_name",
                "is_constant_bool",
                "near_zero_std_bool",
                "min_value",
                "max_value",
                "std_value",
                "approx_distinct_values",
                "scaling_action",
                "scaled_matrix_inclusion",
            ]
        ]
    )

,feature_set,feature_name,is_constant_bool,near_zero_std_bool,min_value,max_value,std_value,approx_distinct_values,scaling_action,scaled_matrix_inclusion
0,fhvhv,borough_mean_fhvhv_avg_trip_speed_missing_indi...,True,True,0.0,0.0,0.0,1,exclude_constant_or_near_zero_variance,exclude_from_scaled_matrix
1,fhvhv,fhvhv_avg_trip_distance_missing_indicator,True,True,0.0,0.0,0.0,1,exclude_constant_or_near_zero_variance,exclude_from_scaled_matrix
2,fhvhv,fhvhv_avg_trip_speed_cp_abs_delta_same_bucket_...,True,True,0.0,0.0,0.0,1,exclude_constant_or_near_zero_variance,exclude_from_scaled_matrix
3,fhvhv,fhvhv_avg_trip_speed_missing_indicator,True,True,0.0,0.0,0.0,1,exclude_constant_or_near_zero_variance,exclude_from_scaled_matrix
4,fhvhv,fhvhv_avg_trip_speed_post_cp_mean_same_bucket_...,True,True,0.0,0.0,0.0,1,exclude_constant_or_near_zero_variance,exclude_from_scaled_matrix
5,fhvhv,fhvhv_avg_trip_speed_pre_cp_mean_same_bucket_m...,True,True,0.0,0.0,0.0,1,exclude_constant_or_near_zero_variance,exclude_from_scaled_matrix
6,fhvhv,fhvhv_avg_trip_speed_rolling_mean_7_same_bucke...,True,True,0.0,0.0,0.0,1,exclude_constant_or_near_zero_variance,exclude_from_scaled_matrix


Findings\. The reconciled exclusion detail confirms that all 7 removed columns are FHVHV missingness indicators with min 0, max 0, standard deviation 0, and one distinct value\. These are inactive flags, not substantive mobility features\. Excluding them reduces indicator clutter without removing any observed mobility signal\.

### Materialize Scaled Modeling Matrices

Apply the scaling policy to the raw matrices\. Standard\-scaled columns are centered and scaled, binary indicator columns are carried through unchanged, and constant or near\-zero\-variance columns are left out of the scaled outputs\. The raw matrices remain the audit trail; the scaled matrices become the default inputs for representation learning\.

Prepare the scaled\-matrix file paths and expected column counts\. This keeps the write step focused on one task and makes the expected output shape explicit before materialization begins\.

In [63]:
# ---------------------------------------------------------------------
# Prepare scaled matrix paths and expected column counts
# ---------------------------------------------------------------------

if "sql_path" not in globals():
    def sql_path(path):
        return str(path).replace("\\", "/").replace("'", "''")

if "duckdb_identifier" not in globals():
    def duckdb_identifier(column_name):
        return '"' + str(column_name).replace('"', '""') + '"'


def infer_raw_matrix_path(feature_set_name):
    candidate_path_columns = [
        column for column in raw_matrix_file_inventory_df.columns
        if "path" in column.lower()
    ]

    for path_column in candidate_path_columns:
        candidate_values = raw_matrix_file_inventory_df.loc[
            raw_matrix_file_inventory_df["feature_set"].eq(feature_set_name),
            path_column,
        ]
        if not candidate_values.empty and pd.notna(candidate_values.iloc[0]):
            candidate_path = Path(candidate_values.iloc[0])
            if candidate_path.exists():
                return candidate_path

    candidate_patterns = [
        f"{feature_set_name}_raw_modeling_matrix.parquet",
        f"raw_{feature_set_name}_modeling_matrix.parquet",
        f"{feature_set_name}_modeling_matrix_raw.parquet",
        f"{feature_set_name}_modeling_matrix.parquet",
        f"*{feature_set_name}*raw*matrix*.parquet",
        f"*{feature_set_name}*modeling*matrix*.parquet",
    ]

    for pattern in candidate_patterns:
        for candidate_path in OUTPUT_DIR.glob(pattern):
            candidate_name = candidate_path.name.lower()
            if "metadata" not in candidate_name and "scaled" not in candidate_name:
                return candidate_path

    raise FileNotFoundError(f"Could not infer raw matrix path for {feature_set_name}.")


RAW_MATRIX_PATHS_BY_SET = {
    feature_set_name: infer_raw_matrix_path(feature_set_name)
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

SCALED_MATRIX_OUTPUT_PATHS = {
    feature_set_name: OUTPUT_DIR / f"{feature_set_name}_scaled_modeling_matrix.parquet"
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

scaling_column_count_df = (
    scaling_action_df
    .groupby(["feature_set", "scaling_action"], as_index=False)
    .agg(column_count=("feature_name", "nunique"))
    .pivot(index="feature_set", columns="scaling_action", values="column_count")
    .fillna(0)
    .astype(int)
    .reset_index()
)

for action_name in [
    SCALING_ACTION_STANDARD_SCALE,
    SCALING_ACTION_KEEP_BINARY,
    SCALING_ACTION_EXCLUDE_CONSTANT,
]:
    if action_name not in scaling_column_count_df.columns:
        scaling_column_count_df[action_name] = 0

expected_scaled_matrix_shape_df = (
    raw_matrix_file_inventory_df[["feature_set", "matrix_rows", "model_column_count"]]
    .merge(scaling_column_count_df, on="feature_set", how="left")
)
expected_scaled_matrix_shape_df["expected_scaled_model_column_count"] = (
    expected_scaled_matrix_shape_df[SCALING_ACTION_STANDARD_SCALE]
    + expected_scaled_matrix_shape_df[SCALING_ACTION_KEEP_BINARY]
)
expected_scaled_matrix_shape_df["excluded_column_count"] = expected_scaled_matrix_shape_df[
    SCALING_ACTION_EXCLUDE_CONSTANT
]
expected_scaled_matrix_shape_df["scaled_matrix_path"] = expected_scaled_matrix_shape_df[
    "feature_set"
].map(lambda feature_set_name: str(SCALED_MATRIX_OUTPUT_PATHS[feature_set_name]))

expected_scaled_matrix_shape_df = expected_scaled_matrix_shape_df[
    [
        "feature_set",
        "matrix_rows",
        "model_column_count",
        SCALING_ACTION_STANDARD_SCALE,
        SCALING_ACTION_KEEP_BINARY,
        "excluded_column_count",
        "expected_scaled_model_column_count",
        "scaled_matrix_path",
    ]
]

display(expected_scaled_matrix_shape_df)

print("Scaled matrix paths and expected shapes prepared.")

,feature_set,matrix_rows,model_column_count,standard_scale,keep_binary_indicator,excluded_column_count,expected_scaled_model_column_count,scaled_matrix_path
0,bus,1472947,40,40,0,0,40,pipeline_data/3.1.1.final_tables/bus_scaled_mo...
1,subway,911455,21,21,0,0,21,pipeline_data/3.1.1.final_tables/subway_scaled...
2,taxi,1541800,53,38,15,0,53,pipeline_data/3.1.1.final_tables/taxi_scaled_m...
3,fhvhv,1541800,53,37,9,7,46,pipeline_data/3.1.1.final_tables/fhvhv_scaled_...
4,multimodal,1541800,233,142,91,0,233,pipeline_data/3.1.1.final_tables/multimodal_sc...


Scaled matrix paths and expected shapes prepared.


Findings\. The scaled\-matrix targets are defined cleanly for all five modeling workflows\. Expected scaled column counts match the resolved scaling policy: Bus keeps 40 columns, Subway keeps 21, Taxi keeps 53, FHVHV keeps 46, and Multimodal keeps 233\. FHVHV is the only matrix where the expected scaled column count is lower than the raw matrix count because the 7 constant/all\-zero missingness indicators are excluded before scaling\. This confirms that scaled matrix construction has explicit output paths and column\-count expectations before any scaled parquet files are written\.

Write the scaled matrices\. For each feature set, the block computes column means and population standard deviations for standard\-scaled columns, writes scaled values to parquet, carries binary indicators unchanged, and excludes constant columns documented above\.

In [66]:
# ---------------------------------------------------------------------
# Materialize scaled modeling matrices
# ---------------------------------------------------------------------

scaled_matrix_inventory_records = []
scaling_parameter_records = []
scaled_matrix_column_manifest_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        raw_matrix_path = RAW_MATRIX_PATHS_BY_SET[feature_set_name]
        scaled_matrix_path = SCALED_MATRIX_OUTPUT_PATHS[feature_set_name]
        raw_matrix_path_sql = sql_path(raw_matrix_path)
        scaled_matrix_path_sql = sql_path(scaled_matrix_path)

        action_subset_df = scaling_action_df.loc[
            scaling_action_df["feature_set"].eq(feature_set_name)
        ].copy()

        standard_scale_columns = action_subset_df.loc[
            action_subset_df["scaling_action"].eq(SCALING_ACTION_STANDARD_SCALE),
            "feature_name",
        ].tolist()
        binary_indicator_columns = action_subset_df.loc[
            action_subset_df["scaling_action"].eq(SCALING_ACTION_KEEP_BINARY),
            "feature_name",
        ].tolist()
        excluded_columns = action_subset_df.loc[
            action_subset_df["scaling_action"].eq(SCALING_ACTION_EXCLUDE_CONSTANT),
            "feature_name",
        ].tolist()

        scaling_parameter_lookup = {}
        if standard_scale_columns:
            parameter_expressions = []
            for column_index, column in enumerate(standard_scale_columns):
                column_sql = duckdb_identifier(column)
                parameter_expressions.extend(
                    [
                        f"AVG(CAST({column_sql} AS DOUBLE)) AS mean_{column_index}",
                        f"STDDEV_POP(CAST({column_sql} AS DOUBLE)) AS std_{column_index}",
                    ]
                )

            parameter_row = con.execute(
                f"""
                SELECT
                    {", ".join(parameter_expressions)}
                FROM read_parquet('{raw_matrix_path_sql}')
                """
            ).fetchdf().iloc[0]

            for column_index, column in enumerate(standard_scale_columns):
                mean_value = float(parameter_row[f"mean_{column_index}"])
                std_value = float(parameter_row[f"std_{column_index}"])
                assert np.isfinite(mean_value), f"Non-finite mean for {feature_set_name}.{column}"
                assert np.isfinite(std_value) and std_value > 0, (
                    f"Invalid scaling std for {feature_set_name}.{column}: {std_value}"
                )
                scaling_parameter_lookup[column] = {
                    "mean": mean_value,
                    "std": std_value,
                }
                scaling_parameter_records.append(
                    {
                        "feature_set": feature_set_name,
                        "feature_name": column,
                        "scaler_name": SCALER_NAME,
                        "with_mean": SCALER_WITH_MEAN,
                        "with_std": SCALER_WITH_STD,
                        "mean": mean_value,
                        "std": std_value,
                    }
                )

        select_expressions = ["modeling_row_id"]

        for _, action_row in action_subset_df.iterrows():
            column = action_row["feature_name"]
            column_sql = duckdb_identifier(column)
            output_column_sql = duckdb_identifier(column)
            scaling_action = action_row["scaling_action"]

            if scaling_action == SCALING_ACTION_STANDARD_SCALE:
                mean_value = scaling_parameter_lookup[column]["mean"]
                std_value = scaling_parameter_lookup[column]["std"]
                select_expressions.append(
                    f"((CAST({column_sql} AS DOUBLE) - {mean_value:.17g}) / {std_value:.17g}) AS {output_column_sql}"
                )
            elif scaling_action == SCALING_ACTION_KEEP_BINARY:
                select_expressions.append(
                    f"CAST({column_sql} AS DOUBLE) AS {output_column_sql}"
                )
            elif scaling_action == SCALING_ACTION_EXCLUDE_CONSTANT:
                continue
            else:
                raise ValueError(f"Unknown scaling action: {scaling_action}")

        scaled_matrix_path.parent.mkdir(parents=True, exist_ok=True)
        if scaled_matrix_path.exists():
            scaled_matrix_path.unlink()
            
        con.execute(
            f"""
            COPY (
                SELECT
                    {", ".join(select_expressions)}
                FROM read_parquet('{raw_matrix_path_sql}')
                ORDER BY modeling_row_id
            )
            TO '{scaled_matrix_path_sql}' (FORMAT PARQUET)
            """
        )

        scaled_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{scaled_matrix_path_sql}')"
        ).fetchdf()
        scaled_row_count = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{scaled_matrix_path_sql}')"
        ).fetchone()[0]

        scaled_model_columns = [
            column for column in scaled_schema_df["column_name"].tolist()
            if column != "modeling_row_id"
        ]

        scaled_matrix_inventory_records.append(
            {
                "feature_set": feature_set_name,
                "scaled_matrix_path": str(scaled_matrix_path),
                "scaled_matrix_rows": int(scaled_row_count),
                "scaled_model_column_count": len(scaled_model_columns),
                "standard_scaled_column_count": len(standard_scale_columns),
                "binary_indicator_column_count": len(binary_indicator_columns),
                "excluded_column_count": len(excluded_columns),
            }
        )

        for column in scaled_model_columns:
            column_action = scaling_action_df.loc[
                scaling_action_df["feature_set"].eq(feature_set_name)
                & scaling_action_df["feature_name"].eq(column),
                "scaling_action",
            ].iloc[0]
            scaled_matrix_column_manifest_records.append(
                {
                    "feature_set": feature_set_name,
                    "feature_name": column,
                    "scaling_action": column_action,
                    "scaled_matrix_inclusion": "included_in_scaled_matrix",
                }
            )

scaled_matrix_file_inventory_df = pd.DataFrame(scaled_matrix_inventory_records)
scaling_parameter_df = pd.DataFrame(scaling_parameter_records)
scaled_matrix_column_manifest_df = pd.DataFrame(scaled_matrix_column_manifest_records)

print("Scaled modeling matrices materialized.")

Scaled modeling matrices materialized.


### Validate the Scaled Matrix

Validate the scaled matrix inventory against the expected shapes\. Rows should match the raw matrices, and scaled column counts should equal standard\-scaled columns plus retained binary indicators\.

In [68]:
# ---------------------------------------------------------------------
# Validate scaled matrix inventory and expected shapes
# ---------------------------------------------------------------------

scaled_matrix_inventory_qa_df = (
    scaled_matrix_file_inventory_df
    .merge(
        expected_scaled_matrix_shape_df[
            [
                "feature_set",
                "matrix_rows",
                "expected_scaled_model_column_count",
                "excluded_column_count",
            ]
        ],
        on="feature_set",
        how="left",
        suffixes=("", "_expected"),
    )
)
scaled_matrix_inventory_qa_df["row_count_status"] = np.where(
    scaled_matrix_inventory_qa_df["scaled_matrix_rows"].eq(scaled_matrix_inventory_qa_df["matrix_rows"]),
    "pass",
    "review",
)
scaled_matrix_inventory_qa_df["column_count_status"] = np.where(
    scaled_matrix_inventory_qa_df["scaled_model_column_count"].eq(
        scaled_matrix_inventory_qa_df["expected_scaled_model_column_count"]
    ),
    "pass",
    "review",
)
scaled_matrix_inventory_qa_df["status"] = np.where(
    scaled_matrix_inventory_qa_df[["row_count_status", "column_count_status"]].eq("pass").all(axis=1),
    "pass",
    "review",
)

scaled_matrix_inventory_qa_df = scaled_matrix_inventory_qa_df[
    [
        "feature_set",
        "scaled_matrix_rows",
        "matrix_rows",
        "scaled_model_column_count",
        "expected_scaled_model_column_count",
        "standard_scaled_column_count",
        "binary_indicator_column_count",
        "excluded_column_count",
        "row_count_status",
        "column_count_status",
        "status",
    ]
]

display(scaled_matrix_inventory_qa_df)

assert scaled_matrix_inventory_qa_df["status"].eq("pass").all(), (
    "One or more scaled matrices has unexpected row or column counts."
)

,feature_set,scaled_matrix_rows,matrix_rows,scaled_model_column_count,expected_scaled_model_column_count,standard_scaled_column_count,binary_indicator_column_count,excluded_column_count,row_count_status,column_count_status,status
0,bus,1472947,1472947,40,40,40,0,0,pass,pass,pass
1,subway,911455,911455,21,21,21,0,0,pass,pass,pass
2,taxi,1541800,1541800,53,53,38,15,0,pass,pass,pass
3,fhvhv,1541800,1541800,46,46,37,9,7,pass,pass,pass
4,multimodal,1541800,1541800,233,233,142,91,0,pass,pass,pass


Findings\. The scaled matrix inventory matches the expected shapes\. Row counts are preserved from the raw matrices, and each scaled matrix keeps exactly the columns allowed by the scaling policy: standard\-scaled features plus retained binary indicators\. FHVHV is the only matrix with fewer scaled columns than raw columns because the 7 all\-zero missingness indicators are intentionally excluded\.

Check numeric safety after scaling\. The scaled matrices should contain no null feature cells, no infinite values, and no nonnumeric model columns\.

In [69]:
# ---------------------------------------------------------------------
# Validate scaled matrix numeric safety
# ---------------------------------------------------------------------

scaled_numeric_safety_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        scaled_matrix_path = SCALED_MATRIX_OUTPUT_PATHS[feature_set_name]
        scaled_matrix_path_sql = sql_path(scaled_matrix_path)

        scaled_schema_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{scaled_matrix_path_sql}')"
        ).fetchdf()

        model_columns = [
            column for column in scaled_schema_df["column_name"].tolist()
            if column != "modeling_row_id"
        ]
        nonnumeric_columns = scaled_schema_df.loc[
            scaled_schema_df["column_name"].isin(model_columns)
            & ~scaled_schema_df["column_type"].str.upper().str.contains(
                "DOUBLE|FLOAT|REAL|DECIMAL|INTEGER|BIGINT|SMALLINT|TINYINT|HUGEINT|UBIGINT|UINTEGER|USMALLINT|UTINYINT"
            ),
            "column_name",
        ].tolist()

        if model_columns:
            null_expressions = [
                f"SUM(CASE WHEN {duckdb_identifier(column)} IS NULL THEN 1 ELSE 0 END)"
                for column in model_columns
            ]
            nonfinite_expressions = [
                f"SUM(CASE WHEN {duckdb_identifier(column)} IS NOT NULL AND NOT isfinite(CAST({duckdb_identifier(column)} AS DOUBLE)) THEN 1 ELSE 0 END)"
                for column in model_columns
            ]

            null_total = con.execute(
                f"""
                SELECT {' + '.join(null_expressions)} AS null_feature_cells
                FROM read_parquet('{scaled_matrix_path_sql}')
                """
            ).fetchone()[0]
            nonfinite_total = con.execute(
                f"""
                SELECT {' + '.join(nonfinite_expressions)} AS nonfinite_feature_cells
                FROM read_parquet('{scaled_matrix_path_sql}')
                """
            ).fetchone()[0]
        else:
            null_total = 0
            nonfinite_total = 0

        scaled_numeric_safety_records.append(
            {
                "feature_set": feature_set_name,
                "scaled_model_column_count": len(model_columns),
                "nonnumeric_model_columns": len(nonnumeric_columns),
                "remaining_null_feature_cells": int(null_total or 0),
                "nonfinite_feature_cells": int(nonfinite_total or 0),
                "status": "pass"
                if len(nonnumeric_columns) == 0 and int(null_total or 0) == 0 and int(nonfinite_total or 0) == 0
                else "review",
            }
        )

scaled_numeric_safety_df = pd.DataFrame(scaled_numeric_safety_records)

display(scaled_numeric_safety_df)

assert scaled_numeric_safety_df["status"].eq("pass").all(), (
    "One or more scaled matrices failed numeric safety checks."
)

,feature_set,scaled_model_column_count,nonnumeric_model_columns,remaining_null_feature_cells,nonfinite_feature_cells,status
0,bus,40,0,0,0,pass
1,subway,21,0,0,0,pass
2,taxi,53,0,0,0,pass
3,fhvhv,46,0,0,0,pass
4,multimodal,233,0,0,0,pass


Findings\. The scaled matrices pass numeric safety checks\. The scaled feature columns are numeric, contain 0 remaining null feature cells, and contain 0 nonfinite feature cells\. This confirms that scaling did not introduce missing values, infinities, or type problems\.

Validate scaling behavior\. Standard\-scaled columns should have mean near 0 and population standard deviation near 1; retained indicator columns should still be binary; excluded columns should not appear in the scaled matrices\.

In [70]:
# ---------------------------------------------------------------------
# Validate scaled-column behavior
# ---------------------------------------------------------------------

SCALED_MEAN_TOLERANCE = 1e-8
SCALED_STD_TOLERANCE = 1e-6

scaled_behavior_records = []

with duckdb.connect() as con:
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        scaled_matrix_path = SCALED_MATRIX_OUTPUT_PATHS[feature_set_name]
        scaled_matrix_path_sql = sql_path(scaled_matrix_path)

        scaled_schema_columns = set(
            con.execute(
                f"DESCRIBE SELECT * FROM read_parquet('{scaled_matrix_path_sql}')"
            ).fetchdf()["column_name"].tolist()
        )

        standard_scale_columns = scaling_action_df.loc[
            scaling_action_df["feature_set"].eq(feature_set_name)
            & scaling_action_df["scaling_action"].eq(SCALING_ACTION_STANDARD_SCALE),
            "feature_name",
        ].tolist()
        binary_indicator_columns = scaling_action_df.loc[
            scaling_action_df["feature_set"].eq(feature_set_name)
            & scaling_action_df["scaling_action"].eq(SCALING_ACTION_KEEP_BINARY),
            "feature_name",
        ].tolist()
        excluded_columns = scaling_action_df.loc[
            scaling_action_df["feature_set"].eq(feature_set_name)
            & scaling_action_df["scaling_action"].eq(SCALING_ACTION_EXCLUDE_CONSTANT),
            "feature_name",
        ].tolist()

        max_abs_scaled_mean = 0.0
        max_abs_scaled_std_delta = 0.0
        failed_scaled_stat_columns = 0

        if standard_scale_columns:
            scaled_stat_expressions = []
            for column_index, column in enumerate(standard_scale_columns):
                column_sql = duckdb_identifier(column)
                scaled_stat_expressions.extend(
                    [
                        f"AVG(CAST({column_sql} AS DOUBLE)) AS mean_{column_index}",
                        f"STDDEV_POP(CAST({column_sql} AS DOUBLE)) AS std_{column_index}",
                    ]
                )

            scaled_stat_row = con.execute(
                f"""
                SELECT
                    {", ".join(scaled_stat_expressions)}
                FROM read_parquet('{scaled_matrix_path_sql}')
                """
            ).fetchdf().iloc[0]

            scaled_mean_values = []
            scaled_std_delta_values = []
            for column_index, column in enumerate(standard_scale_columns):
                mean_value = float(scaled_stat_row[f"mean_{column_index}"])
                std_value = float(scaled_stat_row[f"std_{column_index}"])
                scaled_mean_values.append(abs(mean_value))
                scaled_std_delta_values.append(abs(std_value - 1.0))

            max_abs_scaled_mean = max(scaled_mean_values) if scaled_mean_values else 0.0
            max_abs_scaled_std_delta = max(scaled_std_delta_values) if scaled_std_delta_values else 0.0
            failed_scaled_stat_columns = sum(
                (mean_delta > SCALED_MEAN_TOLERANCE) or (std_delta > SCALED_STD_TOLERANCE)
                for mean_delta, std_delta in zip(scaled_mean_values, scaled_std_delta_values)
            )

        nonbinary_indicator_cells = 0
        if binary_indicator_columns:
            nonbinary_expressions = [
                f"SUM(CASE WHEN {duckdb_identifier(column)} NOT IN (0, 1) THEN 1 ELSE 0 END)"
                for column in binary_indicator_columns
            ]
            nonbinary_indicator_cells = con.execute(
                f"""
                SELECT {' + '.join(nonbinary_expressions)} AS nonbinary_indicator_cells
                FROM read_parquet('{scaled_matrix_path_sql}')
                """
            ).fetchone()[0]

        excluded_columns_present = sorted(set(excluded_columns) & scaled_schema_columns)

        scaled_behavior_records.append(
            {
                "feature_set": feature_set_name,
                "standard_scaled_column_count": len(standard_scale_columns),
                "binary_indicator_column_count": len(binary_indicator_columns),
                "excluded_column_count": len(excluded_columns),
                "max_abs_scaled_mean": max_abs_scaled_mean,
                "max_abs_scaled_std_delta": max_abs_scaled_std_delta,
                "failed_scaled_stat_columns": int(failed_scaled_stat_columns),
                "nonbinary_indicator_cells": int(nonbinary_indicator_cells or 0),
                "excluded_columns_present_in_scaled_matrix": len(excluded_columns_present),
                "status": "pass"
                if (
                    failed_scaled_stat_columns == 0
                    and int(nonbinary_indicator_cells or 0) == 0
                    and len(excluded_columns_present) == 0
                )
                else "review",
            }
        )

scaled_behavior_validation_df = pd.DataFrame(scaled_behavior_records)

display(scaled_behavior_validation_df)

assert scaled_behavior_validation_df["status"].eq("pass").all(), (
    "One or more scaled matrices failed scaled-column behavior validation."
)

,feature_set,standard_scaled_column_count,binary_indicator_column_count,excluded_column_count,max_abs_scaled_mean,max_abs_scaled_std_delta,failed_scaled_stat_columns,nonbinary_indicator_cells,excluded_columns_present_in_scaled_matrix,status
0,bus,40,0,0,9.415517e-13,4.729550e-14,0,0,0,pass
1,subway,21,0,0,5.565604e-13,1.332268e-14,0,0,0,pass
2,taxi,38,15,0,2.310920e-12,1.465494e-14,0,0,0,pass
3,fhvhv,37,9,7,1.122951e-12,2.065015e-14,0,0,0,pass
4,multimodal,142,91,0,2.260596e-12,6.439294e-14,0,0,0,pass


Findings\. The scaled\-column behavior check passes across all five matrices\. Standard\-scaled columns are centered and scaled within tight numerical tolerance: maximum absolute scaled means are on the order of 10^\-12, and maximum standard\-deviation deltas are on the order of 10^\-14\. Retained indicators remain binary with 0 nonbinary cells\. The 7 excluded FHVHV columns are not present in the scaled matrix, so the constant\-indicator issue is fully resolved\.

## 3\.1\.1\.8 Export Modeling Assets

This final section packages the modeling\-matrix handoff for downstream representation learning and anomaly\-detection notebooks\. The raw and scaled matrices have already been materialized and validated; this section collects their paths, supporting metadata, scaling rules, scaling parameters, and QA summaries into a compact set of export artifacts\.

The export step makes the matrix\-preparation decisions reproducible: which rows each matrix contains, which columns were included or excluded, how scaling was applied, and where downstream notebooks should read the canonical raw and scaled inputs\.

### Write Supporting Export Tables

Define the final export paths for the supporting tables\. Matrix files are already written; these outputs document the scaling policy, scaling parameters, matrix inventories, column manifests, and final QA status\.

In [71]:
# ---------------------------------------------------------------------
# Define final supporting export paths
# ---------------------------------------------------------------------

SUPPORTING_EXPORT_PATHS = {
    "scaling_action_table": OUTPUT_DIR / "scaling_action_table.csv",
    "scaling_parameters": OUTPUT_DIR / "scaling_parameters.csv",
    "scaled_matrix_column_manifest": OUTPUT_DIR / "scaled_matrix_column_manifest.csv",
    "raw_matrix_file_inventory": OUTPUT_DIR / "raw_matrix_file_inventory.csv",
    "scaled_matrix_file_inventory": OUTPUT_DIR / "scaled_matrix_file_inventory.csv",
    "matrix_quality_summary": OUTPUT_DIR / "matrix_quality_summary.csv",
    "scaled_matrix_inventory_qa": OUTPUT_DIR / "scaled_matrix_inventory_qa.csv",
    "scaled_numeric_safety_qa": OUTPUT_DIR / "scaled_numeric_safety_qa.csv",
    "scaled_behavior_validation_qa": OUTPUT_DIR / "scaled_behavior_validation_qa.csv",
    "final_modeling_asset_manifest": OUTPUT_DIR / "final_modeling_asset_manifest.csv",
    "final_modeling_asset_manifest_json": OUTPUT_DIR / "final_modeling_asset_manifest.json",
}

supporting_export_path_df = pd.DataFrame(
    [
        {
            "asset_name": asset_name,
            "path": str(asset_path),
        }
        for asset_name, asset_path in SUPPORTING_EXPORT_PATHS.items()
    ]
)

display(supporting_export_path_df)

print("Supporting export paths defined.")

,asset_name,path
0,scaling_action_table,pipeline_data/3.1.1.final_tables/scaling_actio...
1,scaling_parameters,pipeline_data/3.1.1.final_tables/scaling_param...
2,scaled_matrix_column_manifest,pipeline_data/3.1.1.final_tables/scaled_matrix...
3,raw_matrix_file_inventory,pipeline_data/3.1.1.final_tables/raw_matrix_fi...
4,scaled_matrix_file_inventory,pipeline_data/3.1.1.final_tables/scaled_matrix...
5,matrix_quality_summary,pipeline_data/3.1.1.final_tables/matrix_qualit...
6,scaled_matrix_inventory_qa,pipeline_data/3.1.1.final_tables/scaled_matrix...
7,scaled_numeric_safety_qa,pipeline_data/3.1.1.final_tables/scaled_numeri...
8,scaled_behavior_validation_qa,pipeline_data/3.1.1.final_tables/scaled_behavi...
9,final_modeling_asset_manifest,pipeline_data/3.1.1.final_tables/final_modelin...


Supporting export paths defined.


Write the supporting policy and QA tables\. These exports are small CSV files, while the matrix inputs themselves remain parquet files\.

In [72]:
# ---------------------------------------------------------------------
# Write supporting policy and QA tables
# ---------------------------------------------------------------------

SUPPORTING_EXPORT_TABLES = {
    "scaling_action_table": scaling_action_df,
    "scaling_parameters": scaling_parameter_df,
    "scaled_matrix_column_manifest": scaled_matrix_column_manifest_df,
    "raw_matrix_file_inventory": raw_matrix_file_inventory_df,
    "scaled_matrix_file_inventory": scaled_matrix_file_inventory_df,
    "matrix_quality_summary": matrix_quality_summary_df,
    "scaled_matrix_inventory_qa": scaled_matrix_inventory_qa_df,
    "scaled_numeric_safety_qa": scaled_numeric_safety_df,
    "scaled_behavior_validation_qa": scaled_behavior_validation_df,
}

supporting_export_records = []

for asset_name, export_df in SUPPORTING_EXPORT_TABLES.items():
    export_path = SUPPORTING_EXPORT_PATHS[asset_name]
    export_df.to_csv(export_path, index=False)
    supporting_export_records.append(
        {
            "asset_name": asset_name,
            "path": str(export_path),
            "row_count": len(export_df),
            "column_count": len(export_df.columns),
            "exists": export_path.exists(),
            "status": "pass" if export_path.exists() else "review",
        }
    )

supporting_export_status_df = pd.DataFrame(supporting_export_records)

display(supporting_export_status_df)

assert supporting_export_status_df["status"].eq("pass").all(), (
    "One or more supporting export tables was not written."
)

print("Supporting policy and QA tables written.")

,asset_name,path,row_count,column_count,exists,status
0,scaling_action_table,pipeline_data/3.1.1.final_tables/scaling_actio...,400,13,True,pass
1,scaling_parameters,pipeline_data/3.1.1.final_tables/scaling_param...,278,7,True,pass
2,scaled_matrix_column_manifest,pipeline_data/3.1.1.final_tables/scaled_matrix...,393,4,True,pass
3,raw_matrix_file_inventory,pipeline_data/3.1.1.final_tables/raw_matrix_fi...,5,9,True,pass
4,scaled_matrix_file_inventory,pipeline_data/3.1.1.final_tables/scaled_matrix...,5,7,True,pass
5,matrix_quality_summary,pipeline_data/3.1.1.final_tables/matrix_qualit...,5,17,True,pass
6,scaled_matrix_inventory_qa,pipeline_data/3.1.1.final_tables/scaled_matrix...,5,11,True,pass
7,scaled_numeric_safety_qa,pipeline_data/3.1.1.final_tables/scaled_numeri...,5,6,True,pass
8,scaled_behavior_validation_qa,pipeline_data/3.1.1.final_tables/scaled_behavi...,5,10,True,pass


Supporting policy and QA tables written.


Findings\. The supporting export tables were written successfully\. The handoff now includes scaling actions, scaling parameters, the scaled column manifest, raw and scaled file inventories, matrix quality summaries, and scaled\-matrix QA outputs\. These compact CSV artifacts sit beside the parquet matrices so downstream notebooks can load modeling inputs and audit scaling decisions without rerunning matrix construction\.

### Create Final Asset Manifest

Build a final manifest that names every matrix\-facing artifact produced by this notebook\. This is the handoff table for downstream notebooks: it shows where to find raw matrices, scaled matrices, metadata, and compact QA/supporting files\.

In [73]:
# ---------------------------------------------------------------------
# Build final modeling asset manifest
# ---------------------------------------------------------------------


def infer_metadata_path(feature_set_name):
    candidate_patterns = [
        f"{feature_set_name}_row_metadata.parquet",
        f"{feature_set_name}_modeling_row_metadata.parquet",
        f"{feature_set_name}_metadata.parquet",
        f"*{feature_set_name}*row*metadata*.parquet",
        f"*{feature_set_name}*metadata*.parquet",
    ]

    for pattern in candidate_patterns:
        for candidate_path in OUTPUT_DIR.glob(pattern):
            candidate_name = candidate_path.name.lower()
            if "matrix" not in candidate_name and "scaled" not in candidate_name:
                return candidate_path

    return None


final_asset_manifest_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    raw_matrix_path = RAW_MATRIX_PATHS_BY_SET[feature_set_name]
    scaled_matrix_path = SCALED_MATRIX_OUTPUT_PATHS[feature_set_name]
    metadata_path = infer_metadata_path(feature_set_name)

    raw_inventory_row = raw_matrix_file_inventory_df.loc[
        raw_matrix_file_inventory_df["feature_set"].eq(feature_set_name)
    ].iloc[0]
    scaled_inventory_row = scaled_matrix_file_inventory_df.loc[
        scaled_matrix_file_inventory_df["feature_set"].eq(feature_set_name)
    ].iloc[0]

    final_asset_manifest_records.extend(
        [
            {
                "asset_group": "raw_matrix",
                "feature_set": feature_set_name,
                "asset_name": f"{feature_set_name}_raw_modeling_matrix",
                "path": str(raw_matrix_path),
                "file_format": raw_matrix_path.suffix.lstrip("."),
                "row_count": int(raw_inventory_row["matrix_rows"]),
                "column_count": int(raw_inventory_row["model_column_count"]),
                "status": "pass" if raw_matrix_path.exists() else "review",
            },
            {
                "asset_group": "scaled_matrix",
                "feature_set": feature_set_name,
                "asset_name": f"{feature_set_name}_scaled_modeling_matrix",
                "path": str(scaled_matrix_path),
                "file_format": scaled_matrix_path.suffix.lstrip("."),
                "row_count": int(scaled_inventory_row["scaled_matrix_rows"]),
                "column_count": int(scaled_inventory_row["scaled_model_column_count"]),
                "status": "pass" if scaled_matrix_path.exists() else "review",
            },
            {
                "asset_group": "row_metadata",
                "feature_set": feature_set_name,
                "asset_name": f"{feature_set_name}_row_metadata",
                "path": str(metadata_path) if metadata_path is not None else None,
                "file_format": metadata_path.suffix.lstrip(".") if metadata_path is not None else None,
                "row_count": int(raw_inventory_row["metadata_rows"]),
                "column_count": None,
                "status": "pass" if metadata_path is not None and metadata_path.exists() else "review",
            },
        ]
    )

for _, export_row in supporting_export_status_df.iterrows():
    export_path = Path(export_row["path"])
    final_asset_manifest_records.append(
        {
            "asset_group": "supporting_table",
            "feature_set": "all",
            "asset_name": export_row["asset_name"],
            "path": str(export_path),
            "file_format": export_path.suffix.lstrip("."),
            "row_count": int(export_row["row_count"]),
            "column_count": int(export_row["column_count"]),
            "status": export_row["status"],
        }
    )

final_modeling_asset_manifest_df = pd.DataFrame(final_asset_manifest_records)

final_modeling_asset_manifest_df.to_csv(
    SUPPORTING_EXPORT_PATHS["final_modeling_asset_manifest"],
    index=False,
)

with open(SUPPORTING_EXPORT_PATHS["final_modeling_asset_manifest_json"], "w") as manifest_file:
    json.dump(
        final_modeling_asset_manifest_df.to_dict(orient="records"),
        manifest_file,
        indent=2,
    )

display(final_modeling_asset_manifest_df)

print("Final modeling asset manifest created.")

,asset_group,feature_set,asset_name,path,file_format,row_count,column_count,status
0,raw_matrix,bus,bus_raw_modeling_matrix,pipeline_data/3.1.1.final_tables/bus_raw_model...,parquet,1472947,40.0,pass
1,scaled_matrix,bus,bus_scaled_modeling_matrix,pipeline_data/3.1.1.final_tables/bus_scaled_mo...,parquet,1472947,40.0,pass
2,row_metadata,bus,bus_row_metadata,pipeline_data/3.1.1.final_tables/bus_row_metad...,parquet,1472947,NaN,pass
3,raw_matrix,subway,subway_raw_modeling_matrix,pipeline_data/3.1.1.final_tables/subway_raw_mo...,parquet,911455,21.0,pass
4,scaled_matrix,subway,subway_scaled_modeling_matrix,pipeline_data/3.1.1.final_tables/subway_scaled...,parquet,911455,21.0,pass
5,row_metadata,subway,subway_row_metadata,pipeline_data/3.1.1.final_tables/subway_row_me...,parquet,911455,NaN,pass
6,raw_matrix,taxi,taxi_raw_modeling_matrix,pipeline_data/3.1.1.final_tables/taxi_raw_mode...,parquet,1541800,53.0,pass
7,scaled_matrix,taxi,taxi_scaled_modeling_matrix,pipeline_data/3.1.1.final_tables/taxi_scaled_m...,parquet,1541800,53.0,pass
8,row_metadata,taxi,taxi_row_metadata,pipeline_data/3.1.1.final_tables/taxi_row_meta...,parquet,1541800,NaN,pass
9,raw_matrix,fhvhv,fhvhv_raw_modeling_matrix,pipeline_data/3.1.1.final_tables/fhvhv_raw_mod...,parquet,1541800,53.0,pass


Final modeling asset manifest created.


Findings\. The final asset manifest captures the full modeling handoff: 5 raw matrices, 5 scaled matrices, 5 row metadata tables, and 9 suprting tables\. All listed assets are present\. The row and column counts align with the validated matrices; FHVHV is the only scaled matrix with fewer columns than its raw matrix because 7 all\-zero missingness indicators were intentionally excluded\.

### Validate the Final Handoff

Every manifest row should point to an existing artifact, and the manifest should include raw matrices, scaled matrices, row metadata, and supporting tables\.

In [74]:
# ---------------------------------------------------------------------
# Validate final modeling asset manifest
# ---------------------------------------------------------------------

required_asset_groups = {
    "raw_matrix",
    "scaled_matrix",
    "row_metadata",
    "supporting_table",
}

manifest_asset_group_summary_df = (
    final_modeling_asset_manifest_df
    .groupby("asset_group", as_index=False)
    .agg(
        asset_count=("asset_name", "count"),
        review_count=("status", lambda values: (values != "pass").sum()),
    )
)
manifest_asset_group_summary_df["status"] = np.where(
    manifest_asset_group_summary_df["review_count"].eq(0),
    "pass",
    "review",
)

missing_asset_groups = sorted(
    required_asset_groups - set(manifest_asset_group_summary_df["asset_group"])
)

final_manifest_validation_df = pd.DataFrame(
    [
        {
            "qa_check": "manifest_csv_exists",
            "value": SUPPORTING_EXPORT_PATHS["final_modeling_asset_manifest"].exists(),
            "status": "pass" if SUPPORTING_EXPORT_PATHS["final_modeling_asset_manifest"].exists() else "review",
        },
        {
            "qa_check": "manifest_json_exists",
            "value": SUPPORTING_EXPORT_PATHS["final_modeling_asset_manifest_json"].exists(),
            "status": "pass" if SUPPORTING_EXPORT_PATHS["final_modeling_asset_manifest_json"].exists() else "review",
        },
        {
            "qa_check": "missing_required_asset_groups",
            "value": len(missing_asset_groups),
            "status": "pass" if not missing_asset_groups else "review",
        },
        {
            "qa_check": "manifest_rows_with_review_status",
            "value": int((final_modeling_asset_manifest_df["status"] != "pass").sum()),
            "status": "pass" if final_modeling_asset_manifest_df["status"].eq("pass").all() else "review",
        },
    ]
)

display(manifest_asset_group_summary_df)
display(final_manifest_validation_df)

manifest_review_rows_df = final_modeling_asset_manifest_df.loc[
    final_modeling_asset_manifest_df["status"] != "pass"
].reset_index(drop=True)

if not manifest_review_rows_df.empty:
    display(manifest_review_rows_df)

assert final_manifest_validation_df["status"].eq("pass").all(), (
    "Final modeling asset manifest has validation issues."
)

print("Final modeling asset manifest validated.")

,asset_group,asset_count,review_count,status
0,raw_matrix,5,0,pass
1,row_metadata,5,0,pass
2,scaled_matrix,5,0,pass
3,supporting_table,9,0,pass


,qa_check,value,status
0,manifest_csv_exists,True,pass
1,manifest_json_exists,True,pass
2,missing_required_asset_groups,0,pass
3,manifest_rows_with_review_status,0,pass


Final modeling asset manifest validated.


Findings\. The final manifest validation passes\. Required asset groups are all present, the CSV and JSON manifests exist, and 0 manifest rows require review\. This confirms that the 3\.1\.1 handoff is complete and ready for downstream representation\-learning notebooks\.

### Notebook Handoff Summary

Let's end with one compact handoff table\. This summarizes the final raw and scaled matrix shapes, metadata alignment, scaling treatment, and export status for each modeling workflow\.

In [75]:
# ---------------------------------------------------------------------
# Create final notebook handoff summary
# ---------------------------------------------------------------------

raw_handoff_df = (
    final_modeling_asset_manifest_df
    .loc[final_modeling_asset_manifest_df["asset_group"].eq("raw_matrix")]
    [["feature_set", "row_count", "column_count", "status"]]
    .rename(
        columns={
            "row_count": "raw_rows",
            "column_count": "raw_model_columns",
            "status": "raw_status",
        }
    )
)

scaled_handoff_df = (
    final_modeling_asset_manifest_df
    .loc[final_modeling_asset_manifest_df["asset_group"].eq("scaled_matrix")]
    [["feature_set", "row_count", "column_count", "status"]]
    .rename(
        columns={
            "row_count": "scaled_rows",
            "column_count": "scaled_model_columns",
            "status": "scaled_status",
        }
    )
)

metadata_handoff_df = (
    final_modeling_asset_manifest_df
    .loc[final_modeling_asset_manifest_df["asset_group"].eq("row_metadata")]
    [["feature_set", "row_count", "status"]]
    .rename(
        columns={
            "row_count": "metadata_rows",
            "status": "metadata_status",
        }
    )
)

scaling_handoff_df = scaled_behavior_validation_df.rename(
    columns={
        "standard_scaled_column_count": "standard_scaled_columns",
        "binary_indicator_column_count": "binary_indicator_columns",
        "excluded_column_count": "excluded_columns",
        "status": "scaled_behavior_status",
    }
)[[
    "feature_set",
    "standard_scaled_columns",
    "binary_indicator_columns",
    "excluded_columns",
    "scaled_behavior_status",
]]

notebook_handoff_summary_df = (
    raw_handoff_df
    .merge(scaled_handoff_df, on="feature_set", how="outer")
    .merge(metadata_handoff_df, on="feature_set", how="outer")
    .merge(scaling_handoff_df, on="feature_set", how="outer")
)

notebook_handoff_summary_df["row_alignment_status"] = np.where(
    notebook_handoff_summary_df["raw_rows"].eq(notebook_handoff_summary_df["scaled_rows"])
    & notebook_handoff_summary_df["raw_rows"].eq(notebook_handoff_summary_df["metadata_rows"]),
    "pass",
    "review",
)

status_columns = [
    "raw_status",
    "scaled_status",
    "metadata_status",
    "scaled_behavior_status",
    "row_alignment_status",
]
notebook_handoff_summary_df["handoff_status"] = np.where(
    notebook_handoff_summary_df[status_columns].eq("pass").all(axis=1),
    "pass",
    "review",
)

notebook_handoff_summary_df = (
    notebook_handoff_summary_df
    [[
        "feature_set",
        "raw_rows",
        "scaled_rows",
        "metadata_rows",
        "raw_model_columns",
        "scaled_model_columns",
        "standard_scaled_columns",
        "binary_indicator_columns",
        "excluded_columns",
        "handoff_status",
    ]]
    .sort_values("feature_set")
    .reset_index(drop=True)
)

display(notebook_handoff_summary_df)

assert notebook_handoff_summary_df["handoff_status"].eq("pass").all(), (
    "One or more modeling workflows has a handoff-summary issue."
)

print("Notebook handoff summary created.")

,feature_set,raw_rows,scaled_rows,metadata_rows,raw_model_columns,scaled_model_columns,standard_scaled_columns,binary_indicator_columns,excluded_columns,handoff_status
0,bus,1472947,1472947,1472947,40.0,40.0,40,0,0,pass
1,fhvhv,1541800,1541800,1541800,53.0,46.0,37,9,7,pass
2,multimodal,1541800,1541800,1541800,233.0,233.0,142,91,0,pass
3,subway,911455,911455,911455,21.0,21.0,21,0,0,pass
4,taxi,1541800,1541800,1541800,53.0,53.0,38,15,0,pass


Notebook handoff summary created.


> Findings\. The handoff summary gives the final matrix shapes and scaling treatment in one place\. Raw rows, scaled rows, and metadata rows are aligned for every feature set\. Bus, Subway, Taxi, and Multimodal retain the same raw and scaled model\-column counts, while FHVHV intentionally drops 7 all\-zero indicator columns during scaling\. All five workflows finish with pass status\.

### 3\.1\.1 Final Output Inventory

The final outputs from 3\.1\.1 are the canonical modeling\-matrix handoff for downstream unsupervised learning\. These files include raw matrices for interpretable EDA, scaled matrices for modeling, row metadata for interpretation, and QA/scaling tables that document how the matrices were prepared\.

- Raw modeling matrices
Use these for interpretable EDA on the final selected features\. They preserve feature values in their natural or engineered scale\.
\* bus\_raw\_modeling\_matrix\.parquet
\* subway\_raw\_modeling\_matrix\.parquet
\* taxi\_raw\_modeling\_matrix\.parquet
\* fhvhv\_raw\_modeling\_matrix\.parquet
\* multimodal\_raw\_modeling\_matrix\.parquet

- Scaled modeling matrices
Use these for PCA, UMAP, clustering, anomaly detection, and other distance\-sensitive modeling workflows\.
\* bus\_scaled\_modeling\_matrix\.parquet
\* subway\_scaled\_modeling\_matrix\.parquet
\* taxi\_scaled\_modeling\_matrix\.parquet
\* fhvhv\_scaled\_modeling\_matrix\.parquet
\* multimodal\_scaled\_modeling\_matrix\.parquet

- Row metadata tables
Use these to join matrix rows back to interpretable fields such as Taxi Zone, borough, date, temporal bucket, pre/post congestion\-pricing period, and policy geography class\. Join using modeling\_row\_id\.
\* bus\_row\_metadata\.parquet
\* subway\_row\_metadata\.parquet
\* taxi\_row\_metadata\.parquet
\* fhvhv\_row\_metadata\.parquet
\* multimodal\_row\_metadata\.parquet

- Supporting QA and policy tables
These document matrix construction, scaling decisions, numeric QA, row/column counts, and final handoff status\.
\* final\_modeling\_asset\_manifest\.csv
Primary handoff manifest listing raw matrices, scaled matrices, metadata tables, supporting QA files, paths, row counts, and column counts\.
\* scaling\_action\_table\.csv
Feature\-level scaling policy table\. Identifies which columns were standard\-scaled, retained as binary indicators, or excluded during scaling\.
\* scaling\_parameters\.csv
Scaling parameters used to transform raw modeling features into scaled modeling inputs\.
\* scaled\_matrix\_column\_manifest\.csv
Column\-level manifest for the scaled matrices\.
\* matrix\_quality\_summary\.csv
Final quality summary for raw modeling matrices, including row counts, feature counts, null checks, duplicate checks, constant\-feature checks, and numeric safety checks\.
\* raw\_matrix\_file\_inventory\.csv
Inventory of raw modeling matrices by feature set\.
\* scaled\_matrix\_file\_inventory\.csv
Inventory of scaled modeling matrices by feature set\.
\* scaled\_matrix\_inventory\_qa\.csv
QA table confirming scaled matrix row and column alignment\.
\* scaled\_numeric\_safety\_qa\.csv
QA table checking scaled matrices for nulls, NaNs, and infinite values\.
\* scaled\_behavior\_validation\_qa\.csv
QA table validating scaling behavior, retained binary indicators, and excluded low\-information columns\.

## Close

> Notebook 3\.1\.1  converts the selected 1\.5\.6 features into raw and scaled modeling matrices for Bus, Subway, Taxi, FHVHV, and Multimodal workflows; preserves row metadata; documents feature handling and scaling decisions; and exports validated QA/manifests\. No modeling results were generated here\. Downstream notebooks can now start PCA, UMAP, clustering, density estimation, and anomaly\-detection workflows from the exported scaled matrices, with raw matrices and metadata available for audit and interpretation\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>